# Penalty direction prediction: Colab (development, smoke tests, audits, inference, reports)

Two direction heads (kick 70%, goalkeeper 30%), source-cross-fitted player roles, camera compensation, VideoMAE / V-JEPA 2.1 / V-JEPA 2 video backbones, numeric specialist and ensembling. Goal outcome is not a target. Upload this notebook and, for training or competition inference, the dataset ZIP; all helper code is embedded and shared byte-for-byte with `Penalty_Kick_HPC.ipynb` (checked by `tools/check_consistency.py`).

**Use Colab for:** real-backbone smoke tests, the bounded development preset, crop/tracking audits, inference from saved models or HPC bundles, and report review. Long, validated baselines and full nested evaluations run on HPC. Colab Pro+ is not guaranteed uninterrupted compute: `/content` is ephemeral; checkpoints go to Drive, and training resumes at the last completed epoch.

Modes (`RUN_MODE`): **smoke** (default, cheap), **train**, **resume**, **inference** (no fitting; no ZIP for custom videos), **reports** (pandas only; stops early). Recipes are at the end; the full workflow is in `docs/WORKFLOW.md`.


## 1. Install dependencies (once per runtime; skip for reports-only)
Keeps Colab's CUDA/PyTorch build and installs the pinned scientific/Hugging Face stack (the HPC environment pins the same versions). Then **Runtime > Restart session**. Reports mode needs none of this.


In [ ]:
# CELL: install
# Run this cell once, then use Runtime > Restart session before proceeding.
import sys, os, json, hashlib, subprocess
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
from packaging.requirements import Requirement

INSTALL_REQUIREMENTS = [
    'numpy==2.2.6', 'scipy>=1.15,<1.17', 'pandas>=2.2.3,<3',
    'scikit-learn==1.7.2', 'av==16.1.0', 'catboost==1.2.8',
    'transformers==5.3.0', 'huggingface-hub>=1.32,<2',
    'ultralytics==8.4.0', 'opencv-python==4.12.0.88', 'tqdm',
    # Official V-JEPA 2.1 repository code needs timm and einops (V-JEPA 2 uses transformers).
    'timm>=1.0.15,<2', 'einops>=0.8,<1',
]
SETUP_MARKER = Path('/content/penalty_dependencies.json')
SETUP_SIGNATURE = hashlib.sha256(json.dumps(INSTALL_REQUIREMENTS).encode()).hexdigest()

def kernel_identity():
    # Distinguish kernels even if an operating-system PID is reused.
    stat=Path('/proc/self/stat')
    start=stat.read_text().rsplit(')',1)[1].split()[19] if stat.exists() else ''
    return f'{os.getpid()}:{start}'

def requirements_match():
    for text in INSTALL_REQUIREMENTS:
        req=Requirement(text)
        try: installed=version(req.name)
        except PackageNotFoundError: return False
        if not req.specifier.contains(installed,prereleases=True): return False
    return True

if sys.version_info < (3,11):
    raise RuntimeError('This dependency set requires Python 3.11 or newer; use a current Colab runtime.')
state=json.loads(SETUP_MARKER.read_text()) if SETUP_MARKER.exists() else {}
if state.get('signature')==SETUP_SIGNATURE and requirements_match():
    if state.get('installed_kernel')==kernel_identity():
        print('Dependencies installed. REQUIRED: Runtime > Restart session, then run from the next code cell.')
    else:
        print('Dependencies already installed and session restarted. Continue with the next code cell.')
else:
    print('Python:',sys.version.split()[0])
    # Preserve Colab CUDA builds; let pip resolve the remaining packages together.
    constraints=Path('/content/penalty_torch_constraints.txt')
    constraints.write_text('\n'.join(f'{name}=={version(name)}' for name in ['torch','torchvision'])+'\n')
    subprocess.check_call([sys.executable,'-m','pip','install','--only-binary=:all:',
                           '--constraint',str(constraints),*INSTALL_REQUIREMENTS])
    SETUP_MARKER.write_text(json.dumps(dict(signature=SETUP_SIGNATURE,
        installed_kernel=kernel_identity(),requirements=INSTALL_REQUIREMENTS),indent=2))
    print('\nINSTALLATION COMPLETE. REQUIRED: Runtime > Restart session.')
    print('Then run from the next code cell. Do not import NumPy/sklearn in this session.')


## 2. Settings
After the restart, edit only the next cell and use **Runtime > Run after**. Settings are plain Python; the environment cell stops if the restart was skipped and mounts Drive only when a path this mode uses lives on Drive.


In [ ]:
# CELL: settings
# ============================== SETTINGS: the only cell you normally edit ==============================
# Plain Python (no NumPy/torch), safe right after the restart. Same experiment keys as the HPC notebook.
from pathlib import Path
PLATFORM = 'colab'
SHARED_CODE_FINGERPRINT = 'b9bff0474bb7fd9c193a'   # filled by tools/build_notebooks.py

# ---- Execution (does not change experiment identity) ---------------------------------------------
RUN_MODE = 'smoke'            # 'smoke' | 'train' | 'resume' | 'inference' | 'reports'
TRAINED_RUN_DIR = None        # run folder or verified bundle (inference/reports). Relative = under OUTPUT_ROOT
INFERENCE_INPUT = 'private'   # 'private' | 'public_oof' | CSV with clip,path,preferred_foot,gender (no ZIP needed)
FOLD_SELECTION = None         # None = all folds then aggregate; [k] = only fold k; [] = aggregate only
MIGRATE_LEGACY_RUN = False    # inference on a run trained before inference manifests existed
VERIFY_CHECKSUMS = True
RUN_TIMING_BENCHMARK = True   # smoke mode only
GENERATE_HEATMAPS = False
GENERATE_TRACKING_AUDITS = True
ALLOW_EPHEMERAL_OUTPUT = False  # True only for throwaway tests: /content disappears with the runtime

# ---- Utilities (do not change experiment identity) ---------------------------------------------
EXPORT_BUNDLE_TO = None   # e.g. OUTPUT_ROOT/'bundles': checksummed, relocatable copy of a finished run

# ---- Environment: Colab paths. /content is EPHEMERAL staging; Drive is durable ----------------------
DATASET_PATH = Path('/content/drive/MyDrive/Fall2026/CV/penalty-kick-prediction-cs-gy-6643.zip')
OUTPUT_ROOT = Path('/content/drive/MyDrive/Fall2026/CV/penalty_kick_outputs')   # checkpoints, reports, bundles
SCRATCH_ROOT = Path('/content/penalty_scratch')   # extracted videos; lost when the runtime is replaced

# ---- Experiment (identity-defining: changing any of these creates a new experiment) ---------------
EXPERIMENT_NAME = 'dev_videomae_v1'   # human-readable label; the hash of the configuration is the identity
EVALUATION_MODE = 'development'   # 'development' (bounded preset) | 'nested_cv' (full) | 'holdout'
BACKBONE = 'videomae'             # 'videomae' | 'vjepa2_1_vitb_384' | 'vjepa2_vitl_256'
BACKBONE_REVISION = None
TRAINING_POLICY = 'finetune'      # 'finetune' | 'frozen_probe' (freezes encoder WEIGHTS; unrelated to releases)
PREDICT_PRIVATE = False
RUN_ABLATIONS = False
# Bounded development preset: one seed, few epochs, fixed source-disjoint subset chosen without labels.
DEV_SEEDS = [42]
DEV_EPOCHS = 3
DEV_TRAIN_SOURCE_FRACTION = 0.5    # of public sources, for training
DEV_VALIDATION_SOURCE_FRACTION = 0.2  # disjoint sources for inner validation (development-only metric)


In [ ]:
# CELL: platform_helpers
# Colab-specific behavior used by the shared cells. Pure Python.
import os
from pathlib import Path
IS_BATCH = False


class StopNotebook(Exception):
    """Ends Run-all quietly after a mode finished (e.g. reports) without a traceback."""
    def _render_traceback_(self): return []


def stop_notebook(message):
    print(message)
    raise StopNotebook(message)


def show_markdown(text):
    from IPython.display import Markdown, display
    display(Markdown(text))


def offer_download(path):
    from google.colab import files
    files.download(str(path))


def is_ephemeral_path(path):
    """/content is local runtime disk (lost when the runtime is replaced); /content/drive is Google Drive."""
    text = str(Path(path).expanduser())
    return text.startswith('/content') and not text.startswith('/content/drive')


def needs_drive(path):
    return str(path).startswith('/content/drive')


def report_root_for(run_dir):
    # Reports on local files stay local so they never force Drive authentication.
    return Path(OUTPUT_ROOT) if needs_drive(run_dir) else Path(SCRATCH_ROOT)


In [ ]:
# CELL: settings_validation
# Fails fast on incompatible settings, before any decoding, download or model loading. Pure Python.
import os
from pathlib import Path

RUN_MODES = {'smoke', 'train', 'resume', 'inference', 'reports'}
EVALUATION_MODES = {'nested_cv', 'holdout', 'development'}
KNOWN_BACKBONES = {'videomae', 'vjepa2_1_vitb_384', 'vjepa2_vitl_256'}
KNOWN_POLICIES = {'finetune', 'frozen_probe'}


def _problems():
    p = []
    if RUN_MODE not in RUN_MODES: p.append(f'RUN_MODE must be one of {sorted(RUN_MODES)}')
    if EVALUATION_MODE not in EVALUATION_MODES: p.append(f'EVALUATION_MODE must be one of {sorted(EVALUATION_MODES)}')
    if BACKBONE not in KNOWN_BACKBONES: p.append(f'BACKBONE must be one of {sorted(KNOWN_BACKBONES)}')
    if TRAINING_POLICY not in KNOWN_POLICIES: p.append(f'TRAINING_POLICY must be one of {sorted(KNOWN_POLICIES)}')
    if RUN_MODE in {'inference', 'reports'} and TRAINED_RUN_DIR is None:
        p.append(f"RUN_MODE='{RUN_MODE}' needs TRAINED_RUN_DIR (a run folder or a verified bundle).")
    if RUN_ABLATIONS and (EVALUATION_MODE != 'nested_cv' or RUN_MODE not in {'train', 'resume'} or FOLD_SELECTION is not None):
        p.append("RUN_ABLATIONS needs EVALUATION_MODE='nested_cv', RUN_MODE train/resume and FOLD_SELECTION=None.")
    if PREDICT_PRIVATE and EVALUATION_MODE != 'nested_cv' and RUN_MODE in {'train', 'resume'}:
        p.append("PREDICT_PRIVATE uses the outer-fold ensemble: only with EVALUATION_MODE='nested_cv'. "
                 'Development/holdout results are never exported as a submission.')
    if FOLD_SELECTION is not None and (EVALUATION_MODE != 'nested_cv' or RUN_MODE not in {'train', 'resume'}):
        p.append("FOLD_SELECTION applies to nested_cv training (one job per fold).")
    if EVALUATION_MODE == 'development':
        if not (0 < DEV_TRAIN_SOURCE_FRACTION < 1 and 0 < DEV_VALIDATION_SOURCE_FRACTION < 1
                and DEV_TRAIN_SOURCE_FRACTION + DEV_VALIDATION_SOURCE_FRACTION <= 1):
            p.append('DEV_*_SOURCE_FRACTION values must be in (0,1) and sum to at most 1.')
        if not DEV_SEEDS or DEV_EPOCHS < 1: p.append('DEV_SEEDS must be nonempty and DEV_EPOCHS >= 1.')
    if RUN_MODE == 'inference' and INFERENCE_INPUT not in {'private', 'public_oof'} and not str(INFERENCE_INPUT).endswith('.csv'):
        p.append("INFERENCE_INPUT must be 'private', 'public_oof' or a .csv path.")
    return p


_issues = _problems()
if RUN_MODE in {'smoke', 'train', 'resume'} and is_ephemeral_path(OUTPUT_ROOT) and not ALLOW_EPHEMERAL_OUTPUT:
    _issues.append(f'OUTPUT_ROOT={OUTPUT_ROOT} is ephemeral on {PLATFORM}; checkpoints would be lost. Use durable '
                   'storage (Colab: Drive; HPC: /scratch/$NETID) or set ALLOW_EPHEMERAL_OUTPUT=True for a throwaway test.')
if _issues:
    raise ValueError('Settings problems (nothing expensive has run):\n- ' + '\n- '.join(_issues))

QUICK_SMOKE = RUN_MODE == 'smoke'
if TRAINED_RUN_DIR is not None:
    TRAINED_RUN_DIR = Path(TRAINED_RUN_DIR)
    if not TRAINED_RUN_DIR.is_absolute(): TRAINED_RUN_DIR = Path(OUTPUT_ROOT) / TRAINED_RUN_DIR
CUSTOM_INFERENCE = RUN_MODE == 'inference' and INFERENCE_INPUT not in {'private', 'public_oof'}
# The competition ZIP is needed for training and for private/public inference only; custom-video
# inference and reports never open it.
NEEDS_DATASET = RUN_MODE in {'smoke', 'train', 'resume'} or (RUN_MODE == 'inference' and not CUSTOM_INFERENCE)
EXECUTION_SETTINGS = dict(RUN_MODE=RUN_MODE, TRAINED_RUN_DIR=str(TRAINED_RUN_DIR) if TRAINED_RUN_DIR else None,
    INFERENCE_INPUT=str(INFERENCE_INPUT), FOLD_SELECTION=FOLD_SELECTION, DATASET_PATH=str(DATASET_PATH),
    OUTPUT_ROOT=str(OUTPUT_ROOT), SCRATCH_ROOT=str(SCRATCH_ROOT), PLATFORM=PLATFORM)
EXPERIMENT_SETTINGS = dict(EXPERIMENT_NAME=EXPERIMENT_NAME, EVALUATION_MODE=EVALUATION_MODE, BACKBONE=BACKBONE,
    BACKBONE_REVISION=BACKBONE_REVISION, TRAINING_POLICY=TRAINING_POLICY, PREDICT_PRIVATE=PREDICT_PRIVATE,
    RUN_ABLATIONS=RUN_ABLATIONS, DEV_SEEDS=DEV_SEEDS, DEV_EPOCHS=DEV_EPOCHS,
    DEV_TRAIN_SOURCE_FRACTION=DEV_TRAIN_SOURCE_FRACTION, DEV_VALIDATION_SOURCE_FRACTION=DEV_VALIDATION_SOURCE_FRACTION)
print(f'[{PLATFORM}] RUN_MODE={RUN_MODE} | experiment={EXPERIMENT_NAME} ({EVALUATION_MODE}, {BACKBONE}/{TRAINING_POLICY}) | '
      f'PREDICT_PRIVATE={PREDICT_PRIVATE} | dataset needed={NEEDS_DATASET}' + (f' | run={TRAINED_RUN_DIR}' if TRAINED_RUN_DIR else ''))


In [ ]:
# CELL: environment
# Colab environment: restart guard, Drive only when a path this mode uses needs it. No NumPy/torch imports.
import json, os, sys
from pathlib import Path

_setup_path = Path('/content/penalty_dependencies.json')
if _setup_path.exists():
    _setup = json.loads(_setup_path.read_text())
    _stat = Path('/proc/self/stat')
    _start = _stat.read_text().rsplit(')', 1)[1].split()[19] if _stat.exists() else ''
    if _setup['installed_kernel'] == f'{os.getpid()}:{_start}':
        raise RuntimeError('Restart required: Runtime > Restart session, then run from Settings again.')
    from importlib.metadata import version
    from packaging.requirements import Requirement
    for _text in _setup['requirements']:
        if not Requirement(_text).specifier.contains(version(Requirement(_text).name), prereleases=True):
            raise RuntimeError(f'Dependency changed: {_text}. Rerun the installation cell and restart.')
elif RUN_MODE != 'reports':
    raise RuntimeError('Run the dependency installation cell first, then restart the session. '
                       '(Reports mode needs only pandas/numpy and skips this requirement.)')

# Mount Drive only if a path this mode actually uses lives on Drive.
_used = [OUTPUT_ROOT] if RUN_MODE != 'reports' else []
if NEEDS_DATASET: _used.append(DATASET_PATH)
if TRAINED_RUN_DIR is not None: _used.append(TRAINED_RUN_DIR)
if CUSTOM_INFERENCE: _used.append(INFERENCE_INPUT)
if EXPORT_BUNDLE_TO: _used.append(EXPORT_BUNDLE_TO)
if any(needs_drive(p) for p in _used) and not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Drive not mounted (no path used by this mode needs it, or it is already mounted).')
ENVIRONMENT_INFO = dict(platform='colab', python=sys.version.split()[0])
print('WARNING: SCRATCH_ROOT (/content) is ephemeral and Colab Pro+ runtimes can still be disconnected or replaced. '
      'Completed checkpoints go to OUTPUT_ROOT on Drive; training resumes from the last completed epoch.')


In [ ]:
# CELL: run_artifacts
"""Run directories, writer isolation, attempt records, artifact bundles and frozen releases.

Pure Python (no NumPy/torch), so reports mode can use it. Layout under OUTPUT_ROOT:
    runs/<platform>/<experiment name>__<experiment id>/   one writer platform per run directory
        attempts/<attempt id>/attempt.json               one record per execution (retries, Slurm requeues)
        locks/<scope>.lock                               run, fold<k>, aggregate (heartbeat-refreshed)
    splits/  (shared, immutable saved split assignments)  bundles/<bundle id>/  releases/<release id>/
Experiment identity hashes only computation (data, split, model, features, preprocessing, optimizer);
mount points, platform, job IDs and attempt IDs never change it.
"""
import datetime, hashlib, json, os, shutil, socket, threading, time
from pathlib import Path

ARTIFACT_SCHEMA_VERSION = 2
LOCK_STALE_SECONDS = 600       # a lock whose heartbeat is older than this belongs to a dead process
LOCK_HEARTBEAT_SECONDS = 60


def stable_hash(value, length=16):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()[:length]


def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''): digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path, text):
    """Unique temp name per process, fsync, then rename: concurrent writers never interleave bytes and an
    interrupted write leaves the previous complete file in place."""
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f'.{path.name}.{os.getpid()}.{threading.get_ident()}.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        f.write(text); f.flush(); os.fsync(f.fileno())
    os.replace(tmp, path)


def make_attempt_id(platform):
    job = os.environ.get('SLURM_JOB_ID')
    task = os.environ.get('SLURM_ARRAY_TASK_ID')
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    return f'{stamp}_{platform}_' + (f'job{job}' + (f'_{task}' if task else '') if job else f'pid{os.getpid()}')


class RunLockedError(RuntimeError):
    pass


class RunWriterLock:
    """Exclusive writer for one scope of a run directory (e.g. 'run', 'fold3', 'aggregate').

    A second live writer fails immediately. A Slurm requeue of the same job (same SLURM_JOB_ID) or a lock
    whose heartbeat is older than LOCK_STALE_SECONDS is taken over, so preempted spot jobs can resume.
    """

    def __init__(self, run_dir, scope):
        self.path = Path(run_dir) / 'locks' / f'{scope}.lock'
        self.owner = dict(scope=scope, platform=globals().get('PLATFORM', 'unknown'),
                          attempt=globals().get('ATTEMPT_ID', 'unknown'), host=socket.gethostname(),
                          pid=os.getpid(), slurm_job_id=os.environ.get('SLURM_JOB_ID'))
        self._stop = threading.Event(); self._thread = None

    def _write(self):
        atomic_write_text(self.path, json.dumps(dict(self.owner, heartbeat=time.time()), indent=2))

    def _beat(self):
        while not self._stop.wait(LOCK_HEARTBEAT_SECONDS):
            try: self._write()
            except OSError: pass

    def __enter__(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        try:
            fd = os.open(self.path, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            os.close(fd)
        except FileExistsError:
            try: holder = json.loads(self.path.read_text())
            except (OSError, ValueError): holder = {}
            same_job = holder.get('slurm_job_id') and holder.get('slurm_job_id') == self.owner['slurm_job_id']
            same_process = holder.get('pid') == os.getpid() and holder.get('host') == self.owner['host']
            stale = time.time() - float(holder.get('heartbeat', 0)) > LOCK_STALE_SECONDS
            if not (same_job or same_process or stale):
                raise RunLockedError(f'{self.path} is held by {holder.get("platform")} attempt {holder.get("attempt")} '
                                     f'on {holder.get("host")} (heartbeat {time.time() - float(holder.get("heartbeat", 0)):.0f}s ago). '
                                     'Two writers must never share a run directory; use another EXPERIMENT_NAME/OUTPUT_ROOT.')
            print(f'Taking over lock {self.path.name} ({"requeued job" if same_job else "stale heartbeat"}).')
        self._write()
        self._thread = threading.Thread(target=self._beat, daemon=True); self._thread.start()
        return self

    def __exit__(self, *exc):
        self._stop.set()
        try:
            holder = json.loads(self.path.read_text())
            if holder.get('attempt') == self.owner['attempt'] and holder.get('pid') == os.getpid():
                self.path.unlink()
        except (OSError, ValueError):
            pass
        return False


def write_attempt_record(run_dir, settings, environment):
    """One immutable record per execution attempt; never overwritten by retries."""
    path = Path(run_dir) / 'attempts' / globals()['ATTEMPT_ID'] / 'attempt.json'
    record = dict(attempt=globals()['ATTEMPT_ID'], platform=globals().get('PLATFORM'), started=time.strftime('%Y-%m-%dT%H:%M:%S%z'),
                  host=socket.gethostname(), slurm=dict((k, v) for k, v in os.environ.items() if k.startswith('SLURM_')),
                  settings=settings, environment=environment, code_fingerprint=globals().get('SHARED_CODE_FINGERPRINT'))
    atomic_write_text(path, json.dumps(record, indent=2, default=str))
    return path


# ------------------------------------------------------------------ bundles (relocatable, checksummed)
BUNDLE_MARKER = 'BUNDLE_COMPLETE.json'
BUNDLE_REPORT_FILES = ['experiment_manifest.json', 'folds.csv', 'fold_results.json', 'predictions.npz',
                       'tracking_quality.csv', 'tracking_frames.csv', 'role_click_audit.csv', 'role_click_summary.csv',
                       'ablation_comparisons.csv', 'holdout_manifest.json', 'holdout_metrics.csv', 'loss_history.csv',
                       'dataset_splits.csv', 'development_manifest.json', 'development_metrics.csv',
                       'pipeline_summary.md', 'pipeline_summary.json', 'improvement_priorities.csv']


def bundle_members(run_dir, include_reports=True):
    """Relative paths of everything inference and report review need (never resumable optimizer state)."""
    run = Path(run_dir); members = set()
    manifest = run / 'inference_manifest.json'
    if manifest.exists():
        members.add('inference_manifest.json')
        info = json.loads(manifest.read_text())
        members.update(info['files'])
        for m in info['members']:
            for name in ('class_prior.json', 'backbone_serialization.json', 'provenance.json', 'component_provenance.json'):
                if (run / m['stage_dir'] / name).exists(): members.add(f"{m['stage_dir']}/{name}")
    if include_reports:
        members.update(f for f in BUNDLE_REPORT_FILES if (run / f).exists())
        for folder in ('component_reports', 'ablations'):
            members.update(str(p.relative_to(run)).replace('\\', '/') for p in (run / folder).rglob('*')
                           if p.is_file() and not p.name.endswith('.pt') and (folder != 'ablations' or p.suffix in ('.json', '.csv')))
        members.update(str(p.relative_to(run)).replace('\\', '/') for p in run.glob('stages/*/loss_history_seed*.csv'))
    if not members: raise FileNotFoundError(f'{run} has neither an inference manifest nor report files')
    return sorted(m for m in members if '_latest.pt' not in m)


def export_bundle(run_dir, destination, include_reports=True):
    """Copy a completed run into destination/<bundle id>/ with relative paths and checksums.
    Files are staged in a hidden partial directory; the completion marker is written last, then renamed."""
    run = Path(run_dir); members = bundle_members(run, include_reports)
    checksums = {m: sha256_file(run / m) for m in members}
    bundle_id = f'{run.name}__{stable_hash(checksums, 12)}'
    final = Path(destination) / bundle_id
    if (final / BUNDLE_MARKER).exists():
        print('Bundle already exported:', final); return final
    partial = Path(destination) / f'.{bundle_id}.partial'
    for m in members:
        target = partial / m; target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(run / m, target)
        if sha256_file(target) != checksums[m]: raise IOError(f'Copy verification failed for {m}')
    atomic_write_text(partial / BUNDLE_MARKER, json.dumps(dict(schema=ARTIFACT_SCHEMA_VERSION, bundle_id=bundle_id,
        source_run=run.name, source_platform=globals().get('PLATFORM'), created=time.strftime('%Y-%m-%dT%H:%M:%S%z'),
        files={m: dict(sha256=c, bytes=(run / m).stat().st_size) for m, c in checksums.items()}), indent=2))
    if final.exists(): shutil.rmtree(final)
    os.replace(partial, final)
    print(f'Exported {len(members)} files to {final}')
    return final


def verify_bundle(bundle_dir, verify_checksums=True):
    """A copied bundle is usable only with its completion marker and every listed file intact."""
    bundle = Path(bundle_dir); marker = bundle / BUNDLE_MARKER
    if not marker.exists():
        raise FileNotFoundError(f'{bundle} has no {BUNDLE_MARKER}: the transfer is incomplete or this is not a bundle.')
    info = json.loads(marker.read_text()); problems = []
    for rel, meta in info['files'].items():
        p = bundle / rel
        if not p.exists(): problems.append(f'missing: {p}')
        elif p.stat().st_size != meta['bytes']: problems.append(f'size changed: {p}')
        elif verify_checksums and sha256_file(p) != meta['sha256']: problems.append(f'checksum changed: {p}')
    if problems: raise IOError('Bundle verification failed:\n' + '\n'.join(problems))
    return info


def is_bundle(path):
    return (Path(path) / BUNDLE_MARKER).exists()


# ------------------------------------------------------------------ frozen releases
def create_release(output_root, files, settings, extra=None):
    """Freeze code + configuration: copy the listed files (notebook, runner, submission script) and the
    settings into releases/<release id>/, record checksums, make files read-only, then write RELEASE.json.
    Any later edit produces a different release id; the running release is never overwritten."""
    files = {Path(f).name: Path(f) for f in files}
    digests = {name: sha256_file(path) for name, path in files.items()}
    identity = dict(files=digests, settings=settings, code_fingerprint=globals().get('SHARED_CODE_FINGERPRINT'))
    release_id = f"{settings.get('EXPERIMENT_NAME', 'release')}__{stable_hash(identity, 12)}"
    final = Path(output_root) / 'releases' / release_id
    if (final / 'RELEASE.json').exists():
        print('Release already exists (unchanged):', final); return final
    partial = final.with_name(f'.{release_id}.partial')
    if partial.exists(): shutil.rmtree(partial)
    partial.mkdir(parents=True)
    for name, path in files.items(): shutil.copy2(path, partial / name)
    atomic_write_text(partial / 'settings.json', json.dumps(settings, indent=2, default=str))
    digests['settings.json'] = sha256_file(partial / 'settings.json')
    atomic_write_text(partial / 'RELEASE.json', json.dumps(dict(release_id=release_id, schema=ARTIFACT_SCHEMA_VERSION,
        created=time.strftime('%Y-%m-%dT%H:%M:%S%z'), files=digests, **(extra or {}),
        code_fingerprint=globals().get('SHARED_CODE_FINGERPRINT'),
        rule='Immutable: bug fixes or experiment changes create a new release.'), indent=2, default=str))
    for p in partial.iterdir(): os.chmod(p, 0o444)
    os.replace(partial, final)
    print('Release frozen:', final)
    return final


def verify_release(release_dir):
    info = json.loads((Path(release_dir) / 'RELEASE.json').read_text())
    bad = [n for n, d in info['files'].items() if sha256_file(Path(release_dir) / n) != d]
    if bad: raise IOError(f'Release {release_dir} was modified: {bad}')
    return info


In [ ]:
# CELL: constants_and_summary_helpers
# Competition constants (both notebooks, every mode) and the report-only pipeline summary.
import numpy as np
TARGETS = ['kick_direction', 'goalkeeper_direction']
CLASSES = [['left', 'middle', 'right'], ['left', 'stands in the middle', 'right']]
WEIGHTS = np.array([0.70, 0.30], dtype=np.float32)
"""Pipeline diagnosis and next improvements: report-only summary of a saved run directory.

Pure numpy/pandas/json. It never imports torch, decodes videos, loads models or downloads weights.
Structure: load_run_reports (loading + validation) -> analyze_reports (numbers) -> render_summary_markdown
(rendering) -> write_pipeline_summary (files). Each stage is independently testable.
"""
import json, math
from pathlib import Path
import numpy as np
import pandas as pd

SUMMARY_SCHEMA_VERSION = 1
EXPECTED_TARGETS = ['kick_direction', 'goalkeeper_direction']
EXPECTED_CLASSES = [['left', 'middle', 'right'], ['left', 'stands in the middle', 'right']]
EXPECTED_WEIGHTS = [0.7, 0.3]
PRIORITY_COLUMNS = ['priority', 'suspected_component', 'finding', 'evidence_files', 'component_or_comparison',
                    'sample_count', 'fold_support', 'effect_value', 'effect_units', 'evidence_type',
                    'suggested_change', 'next_controlled_test', 'limitations']
EVIDENCE_TYPES = ('retrained_comparison', 'predictive_comparison', 'quality_association',
                  'visual_audit_needed', 'insufficient_evidence')
SUMMARY_RULES = dict(
    small_slice_clips=30,
    max_priorities=5,
    detector_unmatched_rate_flag=0.20,
    selector_failure_rate_flag=0.20,
    low_tracking_definition="{role}_tracking_quality != 'mostly_tracked' (tracked fraction < 0.8 in the exporter)",
    low_camera_definition='camera_valid_fraction < 0.5',
    overfit_min_validation_rise=0.02,
    target_dominance_ratio=2.0,
    association_requires_fold_majority=True,
    evidence_strength=dict(retrained_comparison=1.0, predictive_comparison=0.8, quality_association=0.5,
                           visual_audit_needed=0.4, insufficient_evidence=0.0),
    ranking_rule=('Investigation priority, not a causal ranking. Findings with a quantified weighted-score effect '
                  '(percentage points of the 70/30 score) rank first by |effect| x evidence strength x fold-support '
                  'fraction, halved when the supporting sample is below small_slice_clips. Unquantified findings '
                  'follow, ordered by evidence strength then sample count. At most max_priorities rows.'),
    interpretation=('Outer-fold diagnosis is exploratory development feedback, not an untouched estimate. A '
                    'recommendation is a proposed future experiment; do not tune against the same outer labels '
                    'and report the result as untouched.'),
)


class MixedRunError(ValueError):
    """Report files disagree about the run they belong to."""


# ----------------------------------------------------------------------------- loading and validation
def _read_csv(path):
    if not Path(path).exists(): return None
    try: return pd.read_csv(path)
    except pd.errors.EmptyDataError: return pd.DataFrame()  # exporter writes header-less files for empty audits


def _read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8')) if Path(path).exists() else None


def _fold_key(value):
    text = str(value)
    return 'pooled' if text == 'pooled' else str(int(float(text)))


def load_run_reports(run_dir):
    """Load every known report file from an explicit run directory and validate cross-file consistency.

    Returns a dict bundle: data frames/json, coverage table, warnings, status. Raises MixedRunError on
    conflicting run identity, class order, fold assignment or duplicated clip/component rows.
    """
    run = Path(run_dir)
    if not run.is_dir():
        raise FileNotFoundError(f'Run directory not found: {run}')
    files = {
        'experiment_manifest': run / 'experiment_manifest.json', 'folds': run / 'folds.csv',
        'fold_results': run / 'fold_results.json',
        'component_metrics': run / 'component_reports' / 'component_metrics.csv',
        'component_manifest': run / 'component_reports' / 'component_manifest.json',
        'ensemble_comparison': run / 'component_reports' / 'ensemble_comparison.csv',
        'component_clip_errors': run / 'component_reports' / 'component_clip_errors.csv',
        'component_slices': run / 'component_reports' / 'component_slices.csv',
        'component_confusions': run / 'component_reports' / 'component_confusions.csv',
        'tracking_quality': run / 'tracking_quality.csv', 'role_click_audit': run / 'role_click_audit.csv',
        'role_click_summary': run / 'role_click_summary.csv',
        'ablation_comparisons': run / 'ablation_comparisons.csv',
        'inference_manifest': run / 'inference_manifest.json',
        'inference_run': run / 'inference_run.json',
        'holdout_metrics': run / 'holdout_metrics.csv', 'holdout_loss_history': run / 'loss_history.csv',
        'development_manifest': run / 'development_manifest.json',
    }
    bundle = dict(run_dir=str(run), files={k: str(v.relative_to(run)) for k, v in files.items()}, warnings=[])
    loaders = {k: (_read_json if str(v).endswith('.json') else _read_csv) for k, v in files.items()}
    for key, path in files.items(): bundle[key] = loaders[key](path)
    bundle['coverage'] = [dict(report=key, file=bundle['files'][key],
                               status='present' if Path(path).exists() else 'missing') for key, path in files.items()]
    bundle['loss_histories'] = _load_loss_histories(run)
    bundle['ablation_manifests'] = {p.parent.name: _read_json(p)
                                    for p in sorted((run / 'ablations').glob('*/experiment_manifest.json'))}
    _validate(bundle)
    return bundle


def _load_loss_histories(run):
    frames = []
    for path in sorted((run / 'stages').glob('*/loss_history_seed*.csv')):
        frame = pd.read_csv(path)
        seed = path.stem.replace('loss_history_seed', '')
        frames.append(frame.assign(stage=path.parent.name, seed=seed, file=str(path.relative_to(run))))
    return pd.concat(frames, ignore_index=True) if frames else None


def _validate(bundle):
    warn = bundle['warnings'].append
    manifest = bundle['experiment_manifest']
    if bundle['inference_run'] is not None and manifest is None:
        bundle['status'] = 'inference_only'
        warn('Inference-only output: there are no labels, so no accuracy or loss can be reported.')
        return
    if manifest is None and bundle.get('development_manifest') is not None:
        bundle['status'] = 'development'
        warn('Development preset: bounded subset, one seed, few epochs. Inner-validation results only; this is '
             'not complete OOF performance and must not be compared with full nested tables.')
        return
    if manifest is None:
        bundle['status'] = 'holdout' if bundle['holdout_metrics'] is not None else 'empty'
        if bundle['status'] == 'empty':
            warn('No experiment_manifest.json: this is not a nested-diagnostics run directory.')
        return
    classes = manifest.get('classes', {})
    if list(classes) != EXPECTED_TARGETS or [list(v) for v in classes.values()] != EXPECTED_CLASSES:
        raise MixedRunError(f'Unexpected target/class order in experiment_manifest.json: {classes}')
    config = manifest.get('config', {})
    weights = config.get('metric_weights')
    if weights is not None and not np.allclose(weights, EXPECTED_WEIGHTS):
        raise MixedRunError(f'Metric weights {weights} are not the 70/30 competition weights')
    plan = manifest.get('split_plan', {})
    expected_folds = int(config.get('outer_folds', len(plan.get('folds', [])) or 0))
    bundle['expected_folds'] = expected_folds
    bundle['split_id'] = plan.get('split_id', config.get('split_id'))
    component_manifest = bundle['component_manifest']
    if component_manifest is not None:
        if component_manifest.get('targets') != EXPECTED_TARGETS or component_manifest.get('classes') != EXPECTED_CLASSES:
            raise MixedRunError('component_manifest.json target/class order differs from the experiment manifest')
    # Fold identities ---------------------------------------------------------------------------------
    folds = bundle['folds']; fold_map = None
    if folds is not None:
        if not {'clip', 'source_id', 'fold'} <= set(folds) or not folds['clip'].is_unique:
            raise MixedRunError('folds.csv must have unique clip rows with source_id and fold')
        if (folds.groupby('source_id').fold.nunique() > 1).any():
            raise MixedRunError('folds.csv places one source in multiple outer folds')
        identity_clips = plan.get('identity', {}).get('clips')
        if identity_clips is not None and list(folds['clip']) != list(identity_clips):
            raise MixedRunError('folds.csv clip order differs from the saved split identity')
        if plan.get('folds'):
            for item in plan['folds']:
                saved = set(folds['clip'].iloc[item['validation']])
                if saved != set(folds.loc[folds.fold == item['fold'], 'clip']):
                    raise MixedRunError(f'folds.csv fold {item["fold"]} differs from the saved split plan')
        fold_map = folds.set_index('clip').fold
    bundle['fold_map'] = fold_map
    results = bundle['fold_results']
    if results is None:  # fold jobs still running: read each completed fold's own result
        run = Path(bundle['run_dir'])
        results = [json.loads((d / 'result.json').read_text()) for d in sorted(run.glob('folds/fold*'))
                   if (d / 'COMPLETE').exists()]
    done = sorted({int(r['fold']) for r in results})
    if len(done) != len(results): raise MixedRunError('fold_results.json repeats a fold')
    for r in results:
        if bundle['split_id'] is not None and r.get('seeds') is not None and list(r['seeds']) != list(manifest.get('seeds', r['seeds'])):
            raise MixedRunError(f'Fold {r["fold"]} used seeds {r["seeds"]}, manifest lists {manifest.get("seeds")}')
    bundle['completed_folds'] = done
    missing = sorted(set(range(expected_folds)) - set(done))
    bundle['missing_folds'] = missing
    complete = not missing and bundle['component_clip_errors'] is not None and bundle['component_metrics'] is not None
    bundle['status'] = 'complete' if complete else 'incomplete'
    if missing: warn(f'Missing outer folds {missing}: complete-OOF conclusions are suppressed.')
    elif not complete: warn('All folds are recorded but pooled component reports are missing; rerun the export.')
    # Clip/component reports --------------------------------------------------------------------------
    errors = bundle['component_clip_errors']
    if errors is not None:
        if errors.duplicated(['component', 'clip']).any():
            raise MixedRunError('component_clip_errors.csv has duplicated clip/component rows')
        if fold_map is not None:
            if set(errors['clip']) - set(fold_map.index):
                raise MixedRunError('component_clip_errors.csv contains clips absent from folds.csv')
            if not (errors['fold'].to_numpy() == fold_map.loc[errors['clip']].to_numpy()).all():
                raise MixedRunError('component_clip_errors.csv fold assignment differs from folds.csv')
            for name, part in errors.groupby('component'):
                if len(part) != len(fold_map):
                    bundle['status'] = 'incomplete'
                    warn(f'Component {name} covers {len(part)}/{len(fold_map)} clips; pooled conclusions suppressed.')
        if component_manifest is not None and set(component_manifest.get('components', [])) != set(errors['component']):
            raise MixedRunError('component_clip_errors.csv components differ from component_manifest.json')
        if component_manifest is not None and fold_map is not None and component_manifest.get('clips') != list(fold_map.index):
            raise MixedRunError('component_manifest.json clip order differs from folds.csv')
    metrics = bundle['component_metrics']
    if metrics is not None:
        metrics['fold'] = metrics['fold'].map(_fold_key)
        if errors is not None and set(metrics['component']) != set(errors['component']):
            raise MixedRunError('component_metrics.csv components differ from component_clip_errors.csv')
        if metrics.duplicated(['component', 'fold']).any():
            raise MixedRunError('component_metrics.csv repeats a component/fold row')
        pooled = metrics[metrics.fold == 'pooled']
        if fold_map is not None and not (pooled['clips'] == len(fold_map)).all():
            raise MixedRunError('Pooled component metrics do not use the full evaluated population')
    for key in ('ensemble_comparison',):
        frame = bundle[key]
        if frame is not None: frame['fold'] = frame['fold'].map(_fold_key)
    quality = bundle['tracking_quality']
    if quality is not None and fold_map is not None:
        if not quality['clip'].is_unique or set(quality['clip']) - set(fold_map.index):
            raise MixedRunError('tracking_quality.csv clips are duplicated or absent from folds.csv')
        if 'fold' in quality and not (quality['fold'].to_numpy() == fold_map.loc[quality['clip']].to_numpy()).all():
            raise MixedRunError('tracking_quality.csv fold assignment differs from folds.csv')
    # Ablations: same folds/seeds/backbone; feature groups differ by design ----------------------------
    reference_backbone = config.get('backbone', 'videomae (legacy default)')
    for variant, vm in bundle['ablation_manifests'].items():
        vc = vm.get('config', {})
        if vm.get('split_plan', {}).get('split_id') != bundle['split_id']:
            raise MixedRunError(f'Ablation {variant} uses a different saved split')
        if vm.get('seeds') != manifest.get('seeds'):
            raise MixedRunError(f'Ablation {variant} uses different seeds')
        if vc.get('backbone', 'videomae (legacy default)') != reference_backbone or vc.get('epochs') != config.get('epochs'):
            raise MixedRunError(f'Ablation {variant} uses a different backbone/epoch policy')
    comparisons = bundle['ablation_comparisons']
    if comparisons is not None:
        comparisons['fold'] = comparisons['fold'].map(_fold_key)
        for variant in set(comparisons['variant']) | set(comparisons['reference']):
            if variant != 'production' and variant not in bundle['ablation_manifests']:
                warn(f'Ablation comparison references {variant} but ablations/{variant}/experiment_manifest.json is missing.')


# ----------------------------------------------------------------------------- numerical analysis
def _finding(**values):
    row = {k: values.get(k) for k in PRIORITY_COLUMNS if k != 'priority'}
    row['weighted_impact_pp'] = values.get('weighted_impact_pp')
    row['small_sample'] = bool(values.get('small_sample', False))
    row['fold_support_fraction'] = values.get('fold_support_fraction')
    assert row['evidence_type'] in EVIDENCE_TYPES
    return row


def target_contributions(errors, component='ensemble'):
    part = errors[errors.component == component]
    n = len(part)
    out = {}
    for target, weight in zip(EXPECTED_TARGETS, EXPECTED_WEIGHTS):
        wrong = int((~part[target + '_correct'].astype(bool)).sum())
        out[target] = dict(wrong=wrong, clips=n, accuracy=1 - wrong / n,
                           weighted_error_contribution=weight * wrong / n)
    return out


def _per_fold_support(frame, value_column, predicate):
    rows = frame[frame.fold != 'pooled']
    hits = int(predicate(rows[value_column]).sum())
    return hits, len(rows), rows[['fold', value_column]].to_dict('records')


def analyze_reports(bundle, rules=None):
    rules = dict(SUMMARY_RULES, **(rules or {}))
    findings = []; overview = dict(status=bundle['status'])
    status = bundle['status']
    if status in ('inference_only', 'empty'):
        return dict(overview=overview, findings=[], not_evaluated=[], rules=rules)
    complete = status == 'complete'
    errors = bundle.get('component_clip_errors'); metrics = bundle.get('component_metrics')
    n = len(bundle['fold_map']) if bundle.get('fold_map') is not None else None
    if complete:
        pooled = metrics[metrics.fold == 'pooled'].set_index('component')
        overview['components'] = {c: dict(weighted_accuracy_pct=100 * r.weighted_accuracy,
            kick_accuracy_pct=100 * r.kick_direction_accuracy, keeper_accuracy_pct=100 * r.goalkeeper_direction_accuracy,
            weighted_cross_entropy=r.weighted_cross_entropy) for c, r in pooled.iterrows()}
        if 'ensemble' in set(errors.component):
            contrib = target_contributions(errors)
            overview['target_contributions'] = contrib
            findings += _target_dominance(bundle, contrib, rules)
        findings += _ensemble_harm(bundle, rules, n)
        findings += _numeric_specialist(bundle, rules, n)
        findings += _tracking_association(bundle, rules, n)
    findings += _click_audit(bundle, rules)   # audit proxies; valid per evaluated fold
    findings += _ablations(bundle, rules, complete)
    findings += _overfitting(bundle, rules)
    not_evaluated = _not_evaluated(bundle)
    return dict(overview=overview, findings=findings, not_evaluated=not_evaluated, rules=rules)


def _target_dominance(bundle, contrib, rules):
    k, g = (contrib[t]['weighted_error_contribution'] for t in EXPECTED_TARGETS)
    if min(k, g) <= 0 or max(k, g) / min(k, g) < rules['target_dominance_ratio']:
        return []
    target = EXPECTED_TARGETS[int(g > k)]
    conf = bundle.get('component_confusions')
    worst = ''
    if conf is not None:
        off = conf[(conf.component == 'ensemble') & (conf.target == target) & (conf.true_label != conf.predicted_label)]
        if len(off):
            top = off.sort_values('clips', ascending=False).iloc[0]
            worst = f' Largest confusion: true {top.true_label!r} predicted {top.predicted_label!r} ({int(top.clips)} clips).'
    c = contrib[target]
    return [_finding(suspected_component=f'{target} head / inputs', evidence_type='predictive_comparison',
        finding=(f'{target} accounts for {100 * c["weighted_error_contribution"]:.1f} pp of weighted score loss vs '
                 f'{100 * min(k, g):.1f} pp for the other target ({c["wrong"]} wrong of {c["clips"]}).' + worst),
        evidence_files='component_reports/component_clip_errors.csv; component_reports/component_confusions.csv',
        component_or_comparison='ensemble: kick vs keeper weighted error contribution (denominator = all evaluated clips)',
        sample_count=c['clips'], fold_support='pooled only', effect_value=round(100 * c['weighted_error_contribution'], 3),
        effect_units='percentage points of weighted score lost (0.70/0.30 x wrong / N)',
        suggested_change=f'Inspect {target} class confusions and the crops/features most relevant to that target.',
        next_controlled_test='Change one input/head element for this target and compare on the same saved folds with inner-only selection.',
        limitations='Descriptive measurement; does not identify which stage causes the errors.')]


def _ensemble_harm(bundle, rules, n):
    comp = bundle.get('ensemble_comparison')
    if comp is None: return []
    out = []
    for target, weight in zip(EXPECTED_TARGETS, EXPECTED_WEIGHTS):
        rows = comp[comp.target == target]; pooled = rows[rows.fold == 'pooled']
        if pooled.empty: continue
        p = pooled.iloc[0]
        if p.correct_changed_to_mistake <= p.mistakes_corrected: continue
        folds = rows[rows.fold != 'pooled']
        hits = int((folds.correct_changed_to_mistake > folds.mistakes_corrected).sum())
        detail = ', '.join(f'fold {r.fold}: {100 * r.accuracy_delta:+.1f} pp' for r in folds.itertuples())
        out.append(_finding(suspected_component='ensemble weight selection', evidence_type='predictive_comparison',
            finding=(f'For {target}, the frozen ensemble changed {int(p.correct_changed_to_mistake)} correct reference '
                     f'predictions into mistakes and corrected {int(p.mistakes_corrected)} '
                     f'({100 * p.accuracy_delta:+.2f} pp target accuracy; per fold {detail}).'),
            evidence_files='component_reports/ensemble_comparison.csv; fold_results.json',
            component_or_comparison=f'ensemble vs fold-specific inner-selected reference ({target})',
            sample_count=int(p.clips), fold_support=f'{hits}/{len(folds)} folds harmed > corrected',
            fold_support_fraction=hits / max(len(folds), 1), effect_value=round(100 * p.accuracy_delta, 3),
            effect_units=f'percentage points of {target} accuracy', weighted_impact_pp=100 * p.weighted_score_delta,
            suggested_change='Investigate inner-validation weight selection (grid granularity, tie-breaks) or a simpler mixture.',
            next_controlled_test='Compare current mixture with reference-only and equal-weight mixtures selected on inner validation, same saved folds.',
            limitations='Outer labels must not be used to choose the replacement; inner validation sets are small.'))
    return out


def _numeric_specialist(bundle, rules, n):
    metrics = bundle['component_metrics']
    names = set(metrics.component)
    if not {'pose_motion_tree', 'class_prior'} <= names: return []
    out = []
    weights = {}
    for r in bundle.get('fold_results') or []:
        if 'pose_motion_tree' in r.get('ensemble_components', []):
            k = r['ensemble_components'].index('pose_motion_tree')
            for j, t in enumerate(EXPECTED_TARGETS): weights.setdefault(t, []).append(r['weights'][j][k])
    for target in EXPECTED_TARGETS:
        col = target + '_accuracy'
        tree = metrics[metrics.component == 'pose_motion_tree'].set_index('fold')[col]
        prior = metrics[metrics.component == 'class_prior'].set_index('fold')[col]
        delta = (tree - prior).dropna()
        if delta.get('pooled', 0) > 0: continue
        folds = delta.drop('pooled', errors='ignore')
        hits = int((folds <= 0).sum())
        w = weights.get(target)
        out.append(_finding(suspected_component='numeric pose/motion CatBoost specialist', evidence_type='predictive_comparison',
            finding=(f'Standalone {target} accuracy of the numeric specialist is {100 * delta["pooled"]:+.2f} pp vs the '
                     f'training-only class prior' + (f'; mean inner-selected ensemble weight {np.mean(w):.2f}.' if w else '.')),
            evidence_files='component_reports/component_metrics.csv; fold_results.json',
            component_or_comparison=f'pose_motion_tree vs class_prior ({target})', sample_count=n,
            fold_support=f'{hits}/{len(folds)} folds tree <= prior', fold_support_fraction=hits / max(len(folds), 1),
            effect_value=round(100 * delta['pooled'], 3), effect_units=f'percentage points of {target} accuracy',
            suggested_change='Inspect feature scale, missingness, dimensionality and overfitting of the numeric specialist.',
            next_controlled_test='Retrain without the specialist (or with a reduced feature group) on the same folds; check whether it still adds ensemble diversity.',
            limitations='Weak standalone accuracy does not show it is harmful inside the ensemble.'))
    return out


def _group_gap(errors, mask, rules, n):
    low = errors[mask]; high = errors[~mask]
    if not len(low) or not len(high): return None
    gap = low.weighted_error.mean() - high.weighted_error.mean()
    folds = []
    for fold, part in errors.groupby('fold'):
        m = mask.loc[part.index]
        if m.any() and (~m).any():
            folds.append(dict(fold=int(fold), low_minus_high_pp=100 * (part[m].weighted_error.mean() - part[~m].weighted_error.mean()),
                              low_clips=int(m.sum())))
    hits = sum(f['low_minus_high_pp'] > 0 for f in folds)
    excess = (gap * len(low)) / n  # error mass attributable if the low group matched the other group (associational)
    return dict(low=len(low), gap=gap, folds=folds, hits=hits, excess=excess)


def _tracking_association(bundle, rules, n):
    errors = bundle['component_clip_errors']
    part = errors[errors.component == 'ensemble'].reset_index(drop=True)
    out = []
    checks = []
    for role in ('kicker', 'keeper'):
        col = role + '_tracking_quality'
        if col in part: checks.append((f'{role} tracking', part[col].astype(str) != 'mostly_tracked',
                                       f'{role} crop coverage / role tracking', col))
    if 'camera_valid_fraction' in part:
        checks.append(('camera estimate', part['camera_valid_fraction'] < .5, 'camera-motion compensation', 'camera_valid_fraction'))
    for label, mask, component, col in checks:
        g = _group_gap(part, mask, rules, n)
        if g is None or g['gap'] <= 0: continue
        if rules['association_requires_fold_majority'] and not g['hits'] > len(g['folds']) / 2:
            continue  # sign not consistent across a majority of folds: not reported as a priority
        small = g['low'] < rules['small_slice_clips']
        per = ', '.join(f"fold {f['fold']}: {f['low_minus_high_pp']:+.1f} pp (n={f['low_clips']})" for f in g['folds'])
        out.append(_finding(suspected_component=component, evidence_type='quality_association',
            finding=(f'Clips with low {label} have {100 * g["gap"]:.1f} pp higher weighted error rate '
                     f'({g["low"]} low-quality clips{"; SMALL slice" if small else ""}). Per fold: {per}.'),
            evidence_files='component_reports/component_clip_errors.csv; tracking_quality.csv',
            component_or_comparison=f'ensemble errors split by {col}', sample_count=g['low'],
            fold_support=f'{g["hits"]}/{len(g["folds"])} folds with both groups show higher error',
            fold_support_fraction=g['hits'] / max(len(g['folds']), 1), small_sample=small,
            effect_value=round(100 * g['gap'], 3), effect_units='percentage points of weighted error rate (low minus other clips)',
            weighted_impact_pp=100 * g['excess'],
            suggested_change='Review fallback crops and sampled visual audits for these clips; then retrain a targeted change.',
            next_controlled_test='Retrain a targeted tracking/camera change and compare on the same saved folds with inner-only selection.',
            limitations='Association only: hard clips may also be hard to track. Not a causal component effect.'))
    return out


def _click_audit(bundle, rules):
    summary = bundle.get('role_click_summary'); audits = bundle.get('role_click_audit')
    out = []
    if summary is not None and len(summary) and 'role' in summary:
        for r in summary.itertuples():
            rate = getattr(r, 'unmatched_detection_proxy_rate', np.nan)
            if pd.notna(rate) and rate >= rules['detector_unmatched_rate_flag']:
                out.append(_finding(suspected_component=f'person detector ({r.role})', evidence_type='visual_audit_needed',
                    finding=(f'{100 * rate:.1f}% of timestamp-eligible {r.role} clicks have no detection containing the '
                             f'click at the nearest sampled frame ({int(r.timestamp_eligible_clips)} eligible clips).'),
                    evidence_files='role_click_summary.csv; role_click_audit.csv',
                    component_or_comparison=f'{r.role} click-containment detector proxy',
                    sample_count=int(r.timestamp_eligible_clips), fold_support='pooled audit rows',
                    small_sample=r.timestamp_eligible_clips < rules['small_slice_clips'],
                    effect_value=round(100 * rate, 3), effect_units='percent of timestamp-eligible annotated clips',
                    suggested_change='Inspect detector coverage at eligible timestamps (confidence threshold, image size).',
                    next_controlled_test='Re-run detection with a changed detector setting on a sampled audit set, then retrain on the same folds.',
                    limitations='Click containment is a localization proxy, not identity ground truth.'))
    if audits is not None and len(audits) and 'failure_category' in audits:
        eligible = audits[audits.timestamp_eligible.astype(bool)]
        for role, part in eligible.groupby('role'):
            bad = part.failure_category.isin(['role_missing_with_candidate', 'selected_other_box_proxy'])
            rate = float(bad.mean()) if len(part) else np.nan
            if pd.notna(rate) and rate >= rules['selector_failure_rate_flag']:
                folds = part.assign(bad=bad).groupby('fold').bad.mean()
                out.append(_finding(suspected_component=f'role selector / tracker ({role})', evidence_type='visual_audit_needed',
                    finding=(f'{100 * rate:.1f}% of eligible {role} clicks fall inside a candidate box while the selected '
                             f'role is missing or a different box (proxy).'),
                    evidence_files='role_click_audit.csv', component_or_comparison=f'{role} selector failure proxy',
                    sample_count=len(part), fold_support=f'{int((folds >= rules["selector_failure_rate_flag"]).sum())}/{len(folds)} folds at or above flag',
                    small_sample=len(part) < rules['small_slice_clips'],
                    effect_value=round(100 * rate, 3), effect_units='percent of timestamp-eligible annotated clips',
                    suggested_change='Inspect role training matches, selector confidence and track continuity.',
                    next_controlled_test='Change the selector/tracker, regenerate crops, retrain on the same folds.',
                    limitations='Proxy for localization; possible identity switches need visual review, not an automatic rate.'))
    return out


def _ablations(bundle, rules, complete):
    comp = bundle.get('ablation_comparisons')
    if comp is None or comp.empty: return []
    out = []
    for (variant, reference), part in comp.groupby(['variant', 'reference'], sort=False):
        pooled = part[part.fold == 'pooled']
        if pooled.empty: continue
        p = pooled.iloc[0]; folds = part[part.fold != 'pooled']
        delta = p.weighted_accuracy_delta
        if delta >= 0: continue
        hits = int((folds.weighted_accuracy_delta < 0).sum())
        per = ', '.join(f'fold {r.fold}: {100 * r.weighted_accuracy_delta:+.1f} pp' for r in folds.itertuples())
        compensation = variant == 'G_compensated_motion' and reference == 'F_raw_motion'
        out.append(_finding(suspected_component=f'feature/stream change {reference} -> {variant}', evidence_type='retrained_comparison',
            finding=(f'Adding {variant} over {reference} changed pooled weighted accuracy by {100 * delta:+.2f} pp '
                     f'(cross-entropy {p.weighted_cross_entropy_delta:+.4f}; per fold {per}).'),
            evidence_files=f'ablation_comparisons.csv; ablations/{variant}/experiment_manifest.json',
            component_or_comparison=f'{variant} vs {reference} (ensemble, matched folds)', sample_count=int(p.clips),
            fold_support=f'{hits}/{len(folds)} folds negative', fold_support_fraction=hits / max(len(folds), 1),
            effect_value=round(100 * delta, 3), effect_units='percentage points of weighted accuracy (variant minus reference)',
            weighted_impact_pp=100 * delta,
            suggested_change=('Inspect invalid camera estimates and camera quality; keep raw motion as a control.' if compensation
                              else 'Remove or revise that feature group and repeat the controlled comparison.'),
            next_controlled_test='Repeat the matched-fold comparison with a revised feature group and inner-only selection.',
            limitations='Measured under this training policy and seed list; outer-fold reuse makes this exploratory.'))
    return out


def _overfitting(bundle, rules):
    hist = bundle.get('loss_histories')
    if hist is None or not {'train_loss', 'validation_loss'} <= set(hist): return []
    flagged = []; total = 0
    for (stage, seed), h in hist.groupby(['stage', 'seed']):
        h = h.dropna(subset=['train_loss', 'validation_loss']).sort_values('epoch')
        if len(h) < 2: continue
        total += 1
        best = h.loc[h.validation_loss.idxmin()]; last = h.iloc[-1]
        if last.validation_loss - best.validation_loss >= rules['overfit_min_validation_rise'] and last.train_loss < best.train_loss:
            flagged.append(dict(stage=stage, seed=seed, rise=float(last.validation_loss - best.validation_loss),
                                best_epoch=int(best.epoch), last_epoch=int(last.epoch)))
    if not flagged: return []
    rises = [f['rise'] for f in flagged]
    return [_finding(suspected_component='video fine-tuning (capacity/regularization)', evidence_type='predictive_comparison',
        finding=(f'{len(flagged)}/{total} inner-validated fits keep lowering training loss while inner validation loss rises '
                 f'(mean rise {np.mean(rises):.3f}; e.g. {flagged[0]["stage"]} seed {flagged[0]["seed"]}: best epoch '
                 f'{flagged[0]["best_epoch"]} of {flagged[0]["last_epoch"]}).'),
        evidence_files='stages/*/loss_history_seed*.csv', component_or_comparison='eval-mode train vs inner-validation weighted cross-entropy',
        sample_count=len(flagged), fold_support=f'{len(flagged)}/{total} fits', fold_support_fraction=len(flagged) / total,
        effect_value=round(float(np.mean(rises)), 4), effect_units='weighted cross-entropy rise after best epoch (loss, not accuracy)',
        suggested_change='Consider capacity/regularization/augmentation changes selected on inner validation; best-epoch selection already limits impact.',
        next_controlled_test='Compare a regularization change with the same inner splits and seeds; select epochs on inner validation only.',
        limitations='Loss curves; possible overfitting, not a measured accuracy effect.')]


def _not_evaluated(bundle):
    rows = []
    comparisons = bundle.get('ablation_comparisons')
    evaluated = set() if comparisons is None else set(map(tuple, comparisons[['variant', 'reference']].drop_duplicates().to_numpy()))
    pairs = [('B_context_kicker', 'A_context'), ('C_context_keeper', 'A_context'), ('D_all_visual', 'A_context'),
             ('E_plus_pose', 'D_all_visual'), ('F_raw_motion', 'E_plus_pose'), ('G_compensated_motion', 'F_raw_motion'),
             ('H_plus_metadata', 'G_compensated_motion'), ('production', 'H_plus_metadata')]
    for pair in pairs:
        if pair not in evaluated:
            rows.append(dict(comparison=f'{pair[0]} vs {pair[1]}', status='not evaluated',
                             note='No retrained matched-fold comparison exists; this is not a zero effect.'))
    for key, note in [('tracking_quality', 'tracking-quality associations'), ('role_click_audit', 'click-audit proxies')]:
        if bundle.get(key) is None:
            rows.append(dict(comparison=note, status='not evaluated', note=f'{key} report missing.'))
    return rows


# ----------------------------------------------------------------------------- ranking and rendering
def rank_priorities(findings, rules):
    strength = rules['evidence_strength']

    def key(f):
        impact = f.get('weighted_impact_pp')
        support = f.get('fold_support_fraction'); support = 1.0 if support is None else support
        if impact is not None and np.isfinite(impact):
            score = abs(impact) * strength[f['evidence_type']] * support * (0.5 if f['small_sample'] else 1.0)
            return (0, -score, -strength[f['evidence_type']])
        return (1, -strength[f['evidence_type']], -(f.get('sample_count') or 0))
    ranked = sorted(findings, key=key)[:rules['max_priorities']]
    for i, f in enumerate(ranked, 1): f['priority'] = i
    return ranked


def insufficient_evidence_row(bundle):
    if bundle['status'] == 'incomplete':
        missing = bundle.get('missing_folds') or []
        next_step = (f'Finish outer fold(s) {missing} (rerun with RUN_MODE="resume"), then regenerate this summary.'
                     if missing else 'Rerun the pooled report export, then regenerate this summary.')
    elif bundle['status'] == 'empty':
        next_step = ('This folder has no nested-diagnostics reports (for example a QUICK_SMOKE run). Train with '
                     'QUICK_SMOKE=False and EVALUATION_MODE="nested_cv", then regenerate this summary.')
    elif bundle['status'] == 'development':
        next_step = ('Development results screen ideas only. Promote a promising change to a new release and run the '
                     'full nested_cv evaluation (HPC) before drawing performance conclusions.')
    elif bundle['status'] == 'holdout':
        next_step = 'Run EVALUATION_MODE="nested_cv" to obtain component, tracking and ensemble reports.'
    else:
        next_step = ('Run the optional retrained ablations (RUN_ABLATIONS=True) and inspect visual audits; '
                     'no loaded report currently supports a ranked weakness.')
    return dict(priority=1, suspected_component='undetermined',
                finding='The weakest component cannot yet be determined from the loaded reports.',
                evidence_files='; '.join(c['file'] for c in bundle['coverage'] if c['status'] == 'present') or 'none',
                component_or_comparison='n/a', sample_count=None, fold_support='n/a', effect_value=None, effect_units='n/a',
                evidence_type='insufficient_evidence', suggested_change='Collect the missing evidence before changing the pipeline.',
                next_controlled_test=next_step, limitations='No evidence-backed ranking is possible yet.')


def build_pipeline_summary(run_dir, rules=None):
    bundle = load_run_reports(run_dir)
    analysis = analyze_reports(bundle, rules)
    ranked = rank_priorities(analysis['findings'], analysis['rules'])
    if not ranked and bundle['status'] != 'inference_only':
        ranked = [insufficient_evidence_row(bundle)]
    manifest = bundle.get('experiment_manifest') or {}
    config = manifest.get('config', {})
    identity = dict(run_id=Path(run_dir).name, run_dir=str(run_dir), dataset_id=config.get('dataset'),
                    split_id=bundle.get('split_id'), evaluation_mode=config.get('evaluation_mode'),
                    backbone=config.get('backbone', {'name': 'videomae', 'checkpoint': config.get('video'), 'note': 'legacy default'}),
                    training_policy=config.get('training_policy', 'finetune'),
                    feature_variant=config.get('ablation', 'production'),
                    seeds=manifest.get('seeds'), expected_folds=bundle.get('expected_folds'),
                    completed_folds=bundle.get('completed_folds'), missing_folds=bundle.get('missing_folds'))
    if bundle['status'] == 'inference_only':
        info = bundle['inference_run'] or {}
        identity.update(run_id=Path(run_dir).name, backbone=info.get('backbone'), inference_input=info.get('input'),
                        fitted_run=info.get('trained_run_dir'))
    summary = dict(schema_version=SUMMARY_SCHEMA_VERSION, identity=identity, status=bundle['status'],
                   overview=analysis['overview'], priorities=[{k: f.get(k) for k in PRIORITY_COLUMNS} for f in ranked],
                   priority_details=ranked, not_evaluated=analysis['not_evaluated'], coverage=bundle['coverage'],
                   warnings=bundle['warnings'], rules=analysis['rules'])
    return summary


def _fmt(value, digits=2):
    if value is None or (isinstance(value, float) and not math.isfinite(value)): return 'n/a'
    return f'{value:.{digits}f}' if isinstance(value, float) else str(value)


def render_summary_markdown(summary):
    ident = summary['identity']; lines = ['# Pipeline diagnosis and next improvements', '']
    backbone = ident.get('backbone')
    backbone_text = backbone.get('name', backbone) if isinstance(backbone, dict) else backbone
    lines += [f'**Run** `{ident["run_id"]}` | status **{summary["status"]}** | dataset `{ident.get("dataset_id")}` | '
              f'split `{ident.get("split_id")}` | backbone `{backbone_text}` ({ident.get("training_policy")}) | '
              f'feature variant `{ident.get("feature_variant")}` | folds {ident.get("completed_folds")} of {ident.get("expected_folds")}', '']
    if summary['status'] == 'inference_only':
        lines += ['This directory holds label-free predictions. No accuracy, loss or weakness ranking can be computed '
                  'without labels; summarize the fitted training run instead.', '']
    ov = summary['overview']
    if ov.get('components'):
        lines += ['## Overview (pooled outer-fold, exploratory)', '',
                  '| component | weighted acc. % | kick acc. % | keeper acc. % | weighted CE |', '|---|---|---|---|---|']
        for name, r in ov['components'].items():
            lines.append(f'| {name} | {_fmt(r["weighted_accuracy_pct"])} | {_fmt(r["kick_accuracy_pct"])} | '
                         f'{_fmt(r["keeper_accuracy_pct"])} | {_fmt(r["weighted_cross_entropy"], 4)} |')
        lines.append('')
    if ov.get('target_contributions'):
        parts = [f'{t}: {100 * c["weighted_error_contribution"]:.2f} pp ({c["wrong"]}/{c["clips"]} wrong)'
                 for t, c in ov['target_contributions'].items()]
        lines += ['Weighted score lost by target (0.70/0.30 x wrong / all evaluated clips): ' + '; '.join(parts), '']
    lines += ['## Investigation priorities', '', '_' + summary['rules']['ranking_rule'] + '_', '']
    if summary['priorities']:
        cols = PRIORITY_COLUMNS
        lines += ['| ' + ' | '.join(cols) + ' |', '|' + '---|' * len(cols)]
        for row in summary['priorities']:
            lines.append('| ' + ' | '.join(_fmt(row.get(c), 3).replace('|', '/') for c in cols) + ' |')
        lines.append('')
        if summary['priorities'][0]['evidence_type'] == 'insufficient_evidence':
            lines += ['**The weakest component cannot yet be determined.** Next diagnostic: '
                      + summary['priorities'][0]['next_controlled_test'], '']
    if summary['not_evaluated']:
        lines += ['## Not evaluated (not zero effects)', ''] + [f'- {r["comparison"]}: {r["note"]}' for r in summary['not_evaluated']] + ['']
    if summary['warnings']:
        lines += ['## Warnings', ''] + [f'- {w}' for w in summary['warnings']] + ['']
    lines += ['## Report coverage', ''] + [f'- `{c["file"]}`: {c["status"]}' for c in summary['coverage']] + ['']
    lines += ['## Interpretation', '', summary['rules']['interpretation'],
              'Image streams share one head and are not independent classifiers; overlapping slices are not additive; '
              'click containment is not identity correctness; possible tracking switches are visual observations. '
              'Fold counts are transparent heuristics, not significance tests. No competition-score improvement is promised.', '']
    return '\n'.join(lines)


def write_pipeline_summary(run_dir, rules=None, output_dir=None):
    """output_dir defaults to the run folder; reports on someone else's run/bundle should write elsewhere."""
    run = Path(run_dir); out = Path(output_dir) if output_dir is not None else run
    out.mkdir(parents=True, exist_ok=True)
    summary = build_pipeline_summary(run, rules)
    text = render_summary_markdown(summary)
    (out / 'pipeline_summary.md').write_text(text, encoding='utf-8')
    (out / 'pipeline_summary.json').write_text(json.dumps(summary, indent=2, default=str), encoding='utf-8')
    pd.DataFrame(summary['priorities'], columns=PRIORITY_COLUMNS).to_csv(out / 'improvement_priorities.csv', index=False)
    return summary, text


In [ ]:
# CELL: reports_mode
# RUN_MODE='reports' ends here: only json/pandas/numpy are used. No torch, video decoder, detector,
# dataset ZIP or model download. A bundle is verified first; output never goes into a foreign run/bundle.
def report_output_dir(run_dir):
    return report_root_for(run_dir) / 'reports' / f'{Path(run_dir).name}__{ATTEMPT_ID}'


ATTEMPT_ID = make_attempt_id(PLATFORM)
if RUN_MODE == 'reports':
    if not TRAINED_RUN_DIR.is_dir(): raise FileNotFoundError(f'TRAINED_RUN_DIR not found: {TRAINED_RUN_DIR}')
    if is_bundle(TRAINED_RUN_DIR):
        verify_bundle(TRAINED_RUN_DIR, VERIFY_CHECKSUMS); print('Bundle verified:', TRAINED_RUN_DIR)
    REPORT_DIR = report_output_dir(TRAINED_RUN_DIR)
    pipeline_summary, pipeline_summary_markdown = write_pipeline_summary(TRAINED_RUN_DIR, output_dir=REPORT_DIR)
    show_markdown(pipeline_summary_markdown)
    if EXPORT_BUNDLE_TO:   # a reports job can also package the finished run for transfer
        exported_bundle = export_bundle(TRAINED_RUN_DIR, Path(EXPORT_BUNDLE_TO)); verify_bundle(exported_bundle)
    stop_notebook(f'Reports written to {REPORT_DIR}. Reports mode stops here (nothing heavy was imported).')


In [ ]:
# CELL: heavy_imports
# Training/inference dependencies (never reached in reports mode). Catches API conflicts before any
# decoding or large download, and checks GPU visibility before a model is loaded.
import os, json, hashlib, zipfile, random
from pathlib import Path
import numpy as np
import scipy, pandas as pd, sklearn, torch, torchvision, av, cv2
from PIL import Image
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from catboost import CatBoostClassifier
from ultralytics import YOLO
from transformers import VideoMAEConfig, VideoMAEImageProcessor, VideoMAEForVideoClassification
from packaging.version import Version
import joblib
if Version(torch.__version__.split('+')[0]) < Version('2.6'):
    raise RuntimeError('VideoMAE checkpoint loading requires PyTorch >= 2.6.')
# Exercise the actual fine-tuning API with a tiny randomly initialized CPU model.
_tiny=VideoMAEForVideoClassification(VideoMAEConfig(image_size=32,patch_size=16,num_frames=2,
    tubelet_size=2,hidden_size=32,intermediate_size=64,num_hidden_layers=1,num_attention_heads=4,num_labels=3))
_out=_tiny(pixel_values=torch.zeros(1,2,3,32,32)).logits
assert _out.shape==(1,3) and hasattr(_tiny,'videomae') and hasattr(_tiny,'fc_norm')
_out.sum().backward()
assert any(p.grad is not None for p in _tiny.videomae.parameters())
del _tiny,_out
print('Dependency imports and VideoMAE forward/backward preflight: PASS')
if BACKBONE == 'vjepa2_vitl_256':
    from transformers import VJEPA2Config, VJEPA2Model
    _tiny=VJEPA2Model(VJEPA2Config(crop_size=32,frames_per_clip=4,hidden_size=32,num_attention_heads=4,
        num_hidden_layers=1,pred_hidden_size=32,pred_num_hidden_layers=1,pred_num_attention_heads=4))
    _tiny.gradient_checkpointing_enable(); _tiny.train()
    _out=_tiny.encoder(pixel_values_videos=torch.zeros(1,4,3,32,32)).last_hidden_state
    assert _out.shape==(1,2*2*2,32); _out.mean().backward()
    assert any(p.grad is not None for p in _tiny.encoder.parameters()); del _tiny,_out
    print('V-JEPA 2 (Transformers) encoder forward/backward preflight: PASS')
elif BACKBONE == 'vjepa2_1_vitb_384':
    import timm, einops  # official V-JEPA 2.1 repository dependencies
    print('V-JEPA 2.1 dependencies importable; exact encoder keys/shapes are validated (strict) when weights load.')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
ENVIRONMENT_INFO.update(torch=torch.__version__, cuda=torch.version.cuda, device=DEVICE,
                        gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
print('Device:', DEVICE, ENVIRONMENT_INFO['gpus'])
if RUN_MODE in {'smoke', 'train', 'resume'} and DEVICE != 'cuda':
    raise RuntimeError('No GPU is visible to PyTorch. Colab: select a GPU runtime. HPC: request a GPU partition '
                       '(and the Singularity --nv flag).')
Path(OUTPUT_ROOT).mkdir(parents=True, exist_ok=True)
if NEEDS_DATASET and not Path(DATASET_PATH).is_file():
    raise FileNotFoundError(f'Dataset ZIP not found: {DATASET_PATH}')


## Extract locally and audit the dataset


In [ ]:
# CELL: dataset_loading
# Opens the competition ZIP only when the mode needs it (training, private or public inference).
# Custom-video inference never touches it: no ZIP, labels, clip-count assertions or submission template.
EXPECTED_COUNTS = dict(public=1300, private=900)   # competition dataset audit (not applied to custom inference)

def extract_dataset(zip_path, scratch_root):
    signature = hashlib.sha256()
    with Path(zip_path).open('rb') as f:
        for chunk in iter(lambda: f.read(8*1024*1024), b''): signature.update(chunk)
    data_id = signature.hexdigest()[:16]
    root = Path(scratch_root) / 'penalty_data' / data_id
    if not (root / '.complete').exists():
        # Extract into a private partial folder, then rename: concurrent jobs never see half a dataset.
        partial = root.with_name(f'.{data_id}.{os.getpid()}.partial')
        with zipfile.ZipFile(zip_path) as z:
            for info in tqdm(z.infolist(), desc='Extract'):
                destination = (partial / info.filename).resolve()
                assert destination.is_relative_to(partial.resolve()), 'Unsafe archive path'
                z.extract(info, partial)
        (partial / '.complete').touch()
        try: os.replace(partial, root)
        except OSError:
            import shutil; shutil.rmtree(partial, ignore_errors=True)   # another job finished first
    return data_id, root

def find_one(names):
    matches = [p for p in ROOT.rglob('*.csv') if p.name in names]
    assert len(matches) == 1, matches
    return matches[0]

if NEEDS_DATASET:
    DATA_ID, ROOT = extract_dataset(DATASET_PATH, SCRATCH_ROOT)
    train = pd.read_csv(find_one({'train_ground_truth.csv','public_ground_truth.csv'}))
    test = pd.read_csv(find_one({'private_metadata.csv'}))
    template = pd.read_csv(find_one({'submission_template.csv'}))
    assert len(train)==EXPECTED_COUNTS['public'] and len(test)==EXPECTED_COUNTS['private']
    assert train['clip'].is_unique and test['clip'].is_unique and template['clip'].is_unique
    assert set(template['clip'])==set(test['clip'])
    assert not set(train['clip']) & set(test['clip'])
    video_paths = {}
    for p in ROOT.rglob('*.mp4'):
        assert p.name not in video_paths, f'Duplicate filename: {p.name}'
        video_paths[p.name] = p
    assert set(pd.concat([train['clip'],test['clip']])) <= video_paths.keys()
    Y = np.column_stack([train[t].map({c:i for i,c in enumerate(cs)}).to_numpy() for t,cs in zip(TARGETS,CLASSES)])
    assert not pd.isna(Y).any()
    Y=Y.astype(int)
    display(train[TARGETS].apply(pd.Series.value_counts))
    print('Public sources:', train.source_id.nunique(), 'Private sources:', test.source_id.nunique())
else:
    # Custom inference: no competition data. Empty tables keep later definitions valid.
    DATA_ID, ROOT, video_paths, template = None, None, {}, None
    _meta = ['clip','source_id','preferred_foot','gender']
    train = pd.DataFrame(columns=_meta+TARGETS); test = pd.DataFrame(columns=_meta)
    Y = np.zeros((0, len(TARGETS)), dtype=int)
    print('Custom-video inference: the competition ZIP is not opened.')


## Backbone adapters and inference helpers
**Backbones.** `BACKBONE` selects one adapter; each has explicit construction, preprocessing contract, feature dimension, `encode_video`, gradient checkpointing and serialization:

| `BACKBONE` | Loader | Frames x resolution | Feature | Pooling (design choice to test on inner validation) |
|---|---|---|---|---|
| `videomae` (baseline) | `MCG-NJU/videomae-base-finetuned-kinetics` via Transformers | 16 x 224 | 768 | token mean + checkpoint `fc_norm` (original) |
| `vjepa2_1_vitb_384` | official `facebookresearch/vjepa2` code pinned at commit `204698b4…` (`vjepa2_1_vit_base_384`, `pretrained=False`), encoder weights from the README checkpoint `vjepa2_1_vitb_dist_vitG_384.pt` (`ema_encoder`, strict key/shape load) | 16 x 384 | 768 | token mean + new LayerNorm |
| `vjepa2_vitl_256` | `facebook/vjepa2-vitl-fpc64-256` via Transformers `VJEPA2Model` (encoder only) | 16 x 256 | 1024 | token mean + new LayerNorm |

The V-JEPA world-model predictor and the action-conditioned model are not used; the two direction heads are trained on the competition targets. Frames keep the notebook's chronological pre-kick window (16 frames over the final 1.6 s, quadratic spacing); crops are letterboxed rather than center-cropped, which differs from the official evaluation transform. V-JEPA uses the official ImageNet mean/std. The official hub file at that commit points weights at `localhost` ("for testing"), so the adapter builds the hub architecture with `pretrained=False` and downloads the encoder from `dl.fbaipublicfiles.com` itself; V-JEPA 2 is never silently substituted if 2.1 fails. Crop caches are keyed by frames/resolution/backbone, so a new backbone regenerates crops from the shared detector cache (decoded detections and role geometry are reused). Comparing backbones on the same saved folds and seeds is meaningful only as a non-compute-matched comparison: model sizes and resolutions differ.

**Inference helpers** below build and validate `inference_manifest.json` (relative artifact paths, sha256 checksums, backbone/architecture, preprocessing, feature schema, class order, per-fold scalers/selectors/trees/priors, frozen ensemble weights, seed weights) and run label-free prediction with separate, content-keyed prediction caches.


In [ ]:
# CELL: backbone_adapters
"""Backbone adapters: VideoMAE (baseline), V-JEPA 2.1 (official repo) and V-JEPA 2 (Transformers).

Every adapter maps a normalized [batch, frames, channels, height, width] tensor to [batch, feature_dim].
The multi-view model keeps its attribute names `backbone` (encoder) and `norm` (pooled-feature norm), so
existing VideoMAE checkpoints keep their exact state-dict keys. Nothing here downloads weights at import.
"""
import copy, hashlib, json, sys, zipfile
from pathlib import Path

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
VJEPA2_REPO = 'facebookresearch/vjepa2'
# main @ 2026-03-23 ("Fix figure (#143)"), checked 2026-11-05. Its src/hub/backbones.py points
# VJEPA_BASE_URL at http://localhost:8300 "for testing", so the hub entry is built with
# pretrained=False and the encoder is loaded here from the README's official checkpoint URL.
VJEPA2_REVISION = '204698b45b3712590f06245fbfba32d3be539812'

BACKBONES = {
    'videomae': dict(
        family='videomae', loader='transformers.VideoMAEForVideoClassification',
        checkpoint='MCG-NJU/videomae-base-finetuned-kinetics', revision=None,
        frames=16, size=224, tubelet=2, feature_dim=768, input_layout='B,T,C,H,W',
        normalization='checkpoint image processor mean/std',
        pooling='mean over encoder tokens, then the checkpoint fc_norm (original notebook behavior)',
        gradient_checkpointing='VideoMAEModel.gradient_checkpointing_enable()',
        trainable_policies=('finetune', 'frozen_probe')),
    'vjepa2_1_vitb_384': dict(
        family='vjepa2_1', loader='official repo src.hub.backbones.vjepa2_1_vit_base_384(pretrained=False)',
        repo=VJEPA2_REPO, revision=VJEPA2_REVISION, entry_point='vjepa2_1_vit_base_384',
        checkpoint='https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitb_dist_vitG_384.pt',
        checkpoint_key='ema_encoder', frames=16, size=384, tubelet=2, feature_dim=768,
        input_layout='encoder expects B,C,T,H,W; adapter permutes from B,T,C,H,W',
        normalization='ImageNet mean/std (official evals make_transforms)',
        pooling='encoder forward(training=False) returns final-norm tokens; mean over tokens, then a new LayerNorm',
        gradient_checkpointing='VisionTransformer.use_activation_checkpointing attribute (official forward)',
        trainable_policies=('finetune', 'frozen_probe')),
    'vjepa2_vitl_256': dict(
        family='vjepa2', loader='transformers.VJEPA2Model (encoder only; predictor discarded)',
        checkpoint='facebook/vjepa2-vitl-fpc64-256', revision=None,
        frames=16, size=256, tubelet=2, feature_dim=1024, input_layout='B,T,C,H,W',
        normalization='ImageNet mean/std (VJEPA2VideoProcessor defaults)',
        pooling='encoder last_hidden_state (final LayerNorm applied); mean over tokens, then a new LayerNorm',
        gradient_checkpointing='VJEPA2Model.gradient_checkpointing_enable() before taking .encoder',
        trainable_policies=('finetune', 'frozen_probe')),
}
TRAINING_POLICIES = {
    'finetune': 'All encoder, norm, numeric-MLP and head parameters are trainable (original notebook policy).',
    'frozen_probe': 'Encoder frozen (eval mode, no gradients); pooled-feature norm, numeric MLP and heads trainable. '
                    'Plumbing/compute probe, not a substitute for fine-tuning.',
}


def backbone_spec(name, revision=None):
    if name not in BACKBONES:
        raise ValueError(f'Unknown BACKBONE {name!r}; choose one of {sorted(BACKBONES)}')
    spec = copy.deepcopy(BACKBONES[name]); spec['name'] = name
    if revision is not None:
        if spec['family'] == 'vjepa2_1' and revision != spec['revision']:
            raise ValueError('V-JEPA 2.1 code revision is pinned; edit VJEPA2_REVISION deliberately after re-validating.')
        spec['revision'] = revision
    spec['frames'] = int(spec['frames']); spec['size'] = int(spec['size'])
    if spec['frames'] % spec['tubelet']:
        raise ValueError('Frame count must be divisible by the tubelet size')
    return spec


def backbone_identity(spec, policy):
    """Short serializable identity used in configs, checkpoints and manifests."""
    if policy not in TRAINING_POLICIES: raise ValueError(f'Unknown TRAINING_POLICY {policy!r}')
    if policy not in spec['trainable_policies']: raise ValueError(f'{policy} is not supported for {spec["name"]}')
    keys = ['name', 'family', 'checkpoint', 'revision', 'frames', 'size', 'feature_dim', 'pooling']
    return dict({k: spec.get(k) for k in keys}, training_policy=policy)


def _clean_vjepa_keys(state):
    # Same cleaning as the official src/hub/backbones.py::_clean_backbone_key.
    return {k.replace('module.', '').replace('backbone.', ''): v for k, v in state.items()}


def _sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''): digest.update(block)
    return digest.hexdigest()


def vjepa2_repo_dir(revision, repo_dir=None):
    """Download (once) and return the official repo at a pinned commit. repo_dir overrides for tests."""
    import torch, urllib.request
    if repo_dir is not None:
        return Path(repo_dir)
    target = Path(torch.hub.get_dir()) / f'facebookresearch_vjepa2_{revision}'
    if not (target / 'src' / 'hub' / 'backbones.py').exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        archive = target.with_suffix('.zip')
        urllib.request.urlretrieve(f'https://github.com/{VJEPA2_REPO}/archive/{revision}.zip', archive)
        with zipfile.ZipFile(archive) as z:
            root = z.namelist()[0].split('/')[0]
            z.extractall(target.parent)
        (target.parent / root).rename(target); archive.unlink()
    return target


def _import_vjepa2_hub(repo):
    repo = str(Path(repo).resolve())
    for package in ('src', 'app'):
        loaded = sys.modules.get(package)
        origin = getattr(loaded, '__path__', None)
        if loaded is not None and not any(str(p).startswith(repo) for p in (origin or [])):
            raise RuntimeError(f'A different top-level package {package!r} is already imported; restart the '
                               'session before loading the official V-JEPA repository code.')
    if repo not in sys.path: sys.path.insert(0, repo)
    import importlib
    return importlib.import_module('src.hub.backbones')


class BackboneAdapter:
    """Construction, preprocessing contract, encode_video, checkpointing and serialization for one backbone."""

    def __init__(self, spec, policy='finetune', repo_dir=None):
        self.spec = spec; self.policy = policy; self.repo_dir = repo_dir
        backbone_identity(spec, policy)
        self.feature_dim = int(spec['feature_dim'])
        self.frames = spec['frames']; self.size = spec['size']
        self.provenance = {}

    # ---------- construction ----------
    def build(self, architecture=None):
        """Return (encoder, norm, architecture). architecture=None loads pretrained weights; a saved
        architecture dict constructs the same modules without downloading pretrained weights."""
        import torch
        family = self.spec['family']
        if family == 'videomae':
            vm = globals()['VideoMAEForVideoClassification']
            if architecture is None:
                pretrained = vm.from_pretrained(self.spec['checkpoint'], **({'revision': self.spec['revision']}
                                                                            if self.spec.get('revision') else {}))
                config = pretrained.config
                self.provenance['resolved_revision'] = getattr(config, '_commit_hash', None)
            else:
                from transformers import VideoMAEConfig
                pretrained = vm(VideoMAEConfig.from_dict(architecture['config']))
                config = pretrained.config
            if config.num_frames != self.frames or config.image_size != self.size:
                raise ValueError(f'VideoMAE checkpoint expects {config.num_frames} frames at {config.image_size}px; '
                                 f'notebook uses {self.frames} at {self.size}px')
            encoder, norm = pretrained.videomae, copy.deepcopy(pretrained.fc_norm)
            hidden = config.hidden_size
            arch = dict(kind='videomae', config=_jsonable(config.to_dict()))
        elif family == 'vjepa2':
            from transformers import VJEPA2Config, VJEPA2Model
            if architecture is None:
                kwargs = {'revision': self.spec['revision']} if self.spec.get('revision') else {}
                full = VJEPA2Model.from_pretrained(self.spec['checkpoint'], **kwargs)
                self.provenance['resolved_revision'] = getattr(full.config, '_commit_hash', None)
            else:
                full = VJEPA2Model(VJEPA2Config.from_dict(architecture['config']))
            config = full.config
            if config.crop_size != self.size or config.tubelet_size != self.spec['tubelet']:
                raise ValueError(f'V-JEPA 2 checkpoint uses crop {config.crop_size}, tubelet {config.tubelet_size}')
            if self.policy == 'finetune': full.gradient_checkpointing_enable()
            encoder = full.encoder; del full.predictor
            hidden = config.hidden_size; norm = torch.nn.LayerNorm(hidden)
            arch = dict(kind='vjepa2', config=_jsonable(config.to_dict()))
        elif family == 'vjepa2_1':
            hub = _import_vjepa2_hub(vjepa2_repo_dir(self.spec['revision'], self.repo_dir))
            overrides = dict(architecture.get('entry_kwargs', {})) if architecture else {}
            encoder, predictor = getattr(hub, self.spec['entry_point'])(pretrained=False, **overrides)
            del predictor  # world-model predictor is not used for direction classification
            if architecture is None:
                path = self._download_vjepa2_1_checkpoint()
                try:
                    state = torch.load(path, map_location='cpu', weights_only=True)
                except Exception as error:  # official file may hold non-tensor training metadata
                    print(f'NOTE: weights_only load failed ({type(error).__name__}); loading the official Meta '
                          'checkpoint the way the official hub code does (weights_only=False).')
                    state = torch.load(path, map_location='cpu', weights_only=False)
                if self.spec['checkpoint_key'] not in state:
                    raise KeyError(f'{self.spec["checkpoint_key"]} missing from {path}; keys={sorted(state)}')
                weights = _clean_vjepa_keys(state[self.spec['checkpoint_key']])
                result = encoder.load_state_dict(weights, strict=True)  # exact key/shape validation
                self.provenance.update(checkpoint_sha256=_sha256_file(path), checkpoint_file=str(path),
                                       load_result=str(result), encoder_tensors=len(weights))
                del state, weights
            hidden = int(encoder.embed_dim); norm = torch.nn.LayerNorm(hidden)
            arch = dict(kind='vjepa2_1', entry_point=self.spec['entry_point'], revision=self.spec['revision'],
                        entry_kwargs=overrides, embed_dim=hidden, depth=len(encoder.blocks))
        else:
            raise ValueError(f'Unsupported backbone family {family}')
        if hidden != self.feature_dim:
            raise ValueError(f'{self.spec["name"]} hidden size {hidden} != declared feature_dim {self.feature_dim}')
        if self.policy == 'finetune': self.enable_gradient_checkpointing(encoder)
        return encoder, norm, arch

    def _download_vjepa2_1_checkpoint(self):
        import torch
        folder = Path(torch.hub.get_dir()) / 'checkpoints'; folder.mkdir(parents=True, exist_ok=True)
        path = folder / Path(self.spec['checkpoint']).name
        if not path.exists():
            torch.hub.download_url_to_file(self.spec['checkpoint'], str(path), progress=True)
        return path

    def enable_gradient_checkpointing(self, encoder):
        family = self.spec['family']
        if family == 'videomae':
            encoder.gradient_checkpointing_enable()
        elif family == 'vjepa2':
            # transformers 5.x GradientCheckpointingLayer flag, set via the parent model before extraction.
            layers = [m for m in encoder.modules() if hasattr(m, 'gradient_checkpointing')]
            if not any(getattr(m, 'gradient_checkpointing') for m in layers):
                raise RuntimeError('V-JEPA 2 gradient checkpointing was not enabled by the Transformers API')
        elif family == 'vjepa2_1':
            if not hasattr(encoder, 'use_activation_checkpointing'):
                raise RuntimeError('Official V-JEPA 2.1 encoder lacks use_activation_checkpointing')
            encoder.use_activation_checkpointing = True

    # ---------- forward ----------
    def encode_video(self, encoder, norm, video):
        """video: normalized float tensor [B, T, C, H, W] in chronological order -> [B, feature_dim]."""
        return norm(self.pool_tokens(encoder, video))

    def pool_tokens(self, encoder, video):
        """Encoder tokens mean-pooled to [B, feature_dim], before the pooled-feature norm."""
        if video.ndim != 5 or video.shape[1] != self.frames or video.shape[2] != 3:
            raise ValueError(f'Expected [B,{self.frames},3,H,W], got {tuple(video.shape)}')
        family = self.spec['family']
        if family == 'videomae':
            tokens = encoder(pixel_values=video).last_hidden_state
        elif family == 'vjepa2':
            tokens = encoder(pixel_values_videos=video).last_hidden_state
        else:
            tokens = encoder(video.permute(0, 2, 1, 3, 4).contiguous(), training=False)
        if tokens.ndim != 3 or tokens.shape[-1] != self.feature_dim:
            raise ValueError(f'Unexpected encoder token shape {tuple(tokens.shape)}')
        return tokens.mean(1)

    # ---------- serialization ----------
    def serialization(self, architecture):
        return dict(identity=backbone_identity(self.spec, self.policy), spec=_jsonable(self.spec),
                    architecture=architecture, provenance=_jsonable(self.provenance),
                    training_policy=self.policy, policy_description=TRAINING_POLICIES[self.policy])


def _jsonable(value):
    return json.loads(json.dumps(value, default=lambda x: x.tolist() if hasattr(x, 'tolist') else str(x)))


def apply_training_policy(model, policy):
    """Freeze/unfreeze according to policy; returns optimizer parameter groups with explicit learning rates."""
    import torch
    head = list(model.norm.parameters()) + list(model.numeric.parameters()) + list(model.heads.parameters())
    if policy == 'finetune':
        for p in model.parameters(): p.requires_grad_(True)
        return [dict(params=list(model.backbone.parameters()), lr=1e-5, name='backbone'),
                dict(params=head, lr=1e-4, name='norm_numeric_heads')]
    if policy == 'frozen_probe':
        for p in model.backbone.parameters(): p.requires_grad_(False)
        for p in head: p.requires_grad_(True)
        return [dict(params=head, lr=1e-3, name='norm_numeric_heads')]
    raise ValueError(f'Unknown training policy {policy}')


def trainable_parameter_report(model):
    groups = {'backbone': model.backbone, 'norm': model.norm, 'numeric': model.numeric, 'heads': model.heads}
    return {k: dict(trainable=int(sum(p.numel() for p in m.parameters() if p.requires_grad)),
                    total=int(sum(p.numel() for p in m.parameters()))) for k, m in groups.items()}


In [ ]:
# CELL: inference_helpers
"""Inference-only workflow: fitted-artifact manifest, validation, prediction caches and label-free prediction.

Nothing in this cell fits anything: no fit_video/fit_roles/diagnostic_stage/run_nested_diagnostics,
no scaler fitting, no CatBoost .fit, no checkpoint or ensemble selection, no optimizer. It only loads
saved artifacts and performs label-free decoding/detection/cropping through the injected `build_view` hook.
"""
import copy, hashlib, json, os, pickle, time
from pathlib import Path
import numpy as np
import pandas as pd

INFERENCE_SCHEMA_VERSION = 1
INFERENCE_CODE_VERSION = 'inference-v1'
LEGACY_BACKBONE_IDENTITY = dict(name='videomae', training_policy='finetune')


class InferenceArtifactError(RuntimeError):
    """A required fitted artifact is missing, modified or incompatible."""


def _sha256_bytes(data):
    return hashlib.sha256(data).hexdigest()


def sha256_file(path, chunk=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''): digest.update(block)
    return digest.hexdigest()


def stable_hash(value, length=16):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()[:length]


def preprocessing_record(config, image_mean, image_std, camera_cache_version='similarity-jpeg-v1'):
    """Everything that determines evaluation crops/tensors for a fitted model (not the clip list)."""
    record = dict(frames=int(config['frames']), window=float(config['window']), size=int(config['size']),
                  detector_conf=config['detector_conf'], detector_imgsz=config['detector_imgsz'], pose=config['pose'],
                  tracking=config['tracking'], role_min_prob=config['role_min_prob'], crop_version=config['version'],
                  camera_cache_version=camera_cache_version, frame_sampling='last WINDOW seconds, quadratic spacing toward kick',
                  crop_policy='letterbox context + role crops (1.7 x 1.35 box) with full-frame fallback',
                  view_cache='JPEG quality 95', image_mean=[float(x) for x in image_mean],
                  image_std=[float(x) for x in image_std], input_layout='views x frames x channels x H x W, uint8 -> /255 -> (x-mean)/std')
    record['preprocessing_id'] = stable_hash({k: v for k, v in record.items() if k != 'preprocessing_id'})
    return record


def fitted_artifact_id(identity):
    """Stable ID of a fitted member. Must not include the requested prediction clip list."""
    forbidden = {'evaluation_indices', 'evaluation_clips', 'prediction_clips'}
    if forbidden & set(identity): raise ValueError('Fitted identity must exclude the prediction clip list')
    return stable_hash(identity, 24)


def _config_without_evaluation(config):
    return {k: v for k, v in config.items() if k not in ('smoke', 'smoke_max_batches')}


# ------------------------------------------------------------------------------------------- export
def export_inference_manifest(run_dir, *, preprocessing, backbone=None, training_labels=None, dataset_id=None,
                              extra_clip_names=None, holdout_config=None, overwrite=True, verbose=True):
    """Write inference_manifest.json for a nested_cv or holdout run whose trained artifacts exist.

    Used at the end of training and as the one-time exporter for existing runs. training_labels (a frame
    with clip + target columns) is used ONLY when a stage lacks class_prior.json, restricted to that stage's
    own training clips, and the recovered prior is persisted. Inference never needs labels.
    """
    run = Path(run_dir)
    experiment = run / 'experiment_manifest.json'
    if experiment.exists():
        manifest_in = json.loads(experiment.read_text())
        config = manifest_in['config']; plan = manifest_in['split_plan']; seeds = list(manifest_in['seeds'])
        clips = list(plan['identity']['clips']); mode = 'nested_cv'
        results_path = run / 'fold_results.json'
        if not results_path.exists(): raise InferenceArtifactError(f'Missing {results_path}')
        results = json.loads(results_path.read_text())
        expected = int(config['outer_folds'])
        done = sorted(int(r['fold']) for r in results)
        if done != list(range(expected)):
            raise InferenceArtifactError(f'Run is incomplete: folds {done} of {expected}. Resume training first; '
                                         'a partial fold ensemble is never exported.')
        folds = {int(f['fold']): f for f in plan['folds']}
        stages = [(int(r['fold']), r) for r in results]
    elif (run / 'holdout_manifest.json').exists() or holdout_config is not None:
        info = json.loads((run / 'holdout_manifest.json').read_text()) if (run / 'holdout_manifest.json').exists() else {}
        config = info.get('config', holdout_config); seeds = list(info.get('seeds', config['seeds'])); mode = 'holdout'
        split = pd.read_csv(run / 'dataset_splits.csv'); clips = split['clip'].tolist()
        candidates = sorted((run / 'stages').glob('train_validation_test_*'))
        if len(candidates) != 1: raise InferenceArtifactError(f'Expected one holdout stage folder, found {candidates}')
        stages = [(0, dict(stage_folder=str(candidates[0]), ensemble_components=['video_ensemble'],
                           weights=[[1.0], [1.0]], selected_epochs=None))]
        folds = {0: dict(training=np.flatnonzero(split.split == 'training').tolist(),
                         validation=np.flatnonzero(split.split != 'training').tolist())}
    else:
        raise InferenceArtifactError(f'{run} is neither a nested_cv run (experiment_manifest.json) nor a holdout run.')
    targets = ['kick_direction', 'goalkeeper_direction']
    classes = [['left', 'middle', 'right'], ['left', 'stands in the middle', 'right']]
    members = []; files = {}
    for fold, record in stages:
        stage = run / 'stages' / Path(record['stage_folder']).name
        if not stage.is_dir(): raise InferenceArtifactError(f'Missing stage folder {stage}')
        rel = lambda p: str(Path(p).relative_to(run)).replace('\\', '/')
        provenance_path = stage / 'component_provenance.json'
        component_prov = json.loads(provenance_path.read_text()) if provenance_path.exists() else None
        training_idx = (component_prov['training_indices'] if component_prov else folds[fold]['training'])
        if any(i >= len(clips) for i in training_idx): raise InferenceArtifactError('Training index outside public clips')
        training_clips = [clips[i] for i in training_idx]
        if mode == 'nested_cv' and sorted(training_idx) != sorted(folds[fold]['training']):
            raise InferenceArtifactError(f'Fold {fold} stage training indices differ from the saved split plan')
        heldout = [clips[i] for i in folds[fold]['validation']]
        required = dict(role_selector=stage / 'roles_all_training.joblib', numeric_scaler=stage / 'numeric_scaler.joblib')
        names = list(record['ensemble_components'])
        if 'pose_motion_tree' in names: required['trees'] = stage / 'pose_camera_trees.joblib'
        seed_entries = []
        for seed in seeds:
            path = stage / f'video_seed{seed}.pt'
            required[f'video_seed{seed}'] = path
            selected = (record.get('selected_epochs') or {}).get(str(seed))
            seed_entries.append(dict(seed=int(seed), weights=rel(path),
                weight_kind='best_validation' if mode == 'holdout' else 'selected_epoch_final_outer_refit',
                epochs=selected))
        missing = [str(p) for p in required.values() if not Path(p).exists()]
        if missing: raise InferenceArtifactError('Missing trained artifacts (no partial ensembles are exported):\n' + '\n'.join(missing))
        prior_path = stage / 'class_prior.json'
        if prior_path.exists():
            prior_info = json.loads(prior_path.read_text())
        elif 'class_prior' in names or mode == 'nested_cv':
            if training_labels is None:
                raise InferenceArtifactError(f'{prior_path} is missing. Re-run export with training_labels (public '
                    'training labels; only this stage\'s training clips are read) to recover and persist it.')
            labels = training_labels.set_index('clip').loc[training_clips]
            prior = [np.bincount(labels[t].map({c: i for i, c in enumerate(cs)}).to_numpy().astype(int),
                                 minlength=len(cs)) / len(labels) for t, cs in zip(targets, classes)]
            prior_info = dict(prior=[p.tolist() for p in prior], class_order=classes, training_clip_count=len(labels),
                              source='migrated_from_training_labels', training_clips_sha=stable_hash(training_clips))
            prior_path.write_text(json.dumps(prior_info, indent=2))
        else:
            prior_info = None
        if prior_info is not None and prior_info.get('training_clips_sha', stable_hash(training_clips)) != stable_hash(training_clips):
            raise InferenceArtifactError(f'{prior_path} was computed on different training clips')
        serial_path = stage / 'backbone_serialization.json'
        if serial_path.exists(): backbone_info = json.loads(serial_path.read_text())
        elif backbone is not None: backbone_info = backbone
        else: raise InferenceArtifactError(f'{serial_path} missing; pass the saved backbone architecture for migration.')
        evaluation_crops = {}
        crop_prov = stage / 'provenance.json'
        if crop_prov.exists():
            names_all = clips + list(extra_clip_names or [])
            for entry in json.loads(crop_prov.read_text()):
                if entry['kind'] != 'evaluation': continue
                for i in entry['prediction_indices']:
                    if i < len(names_all) and (stage / f'{i}.pkl').exists():
                        evaluation_crops[names_all[i]] = rel(stage / f'{i}.pkl')
        identity = dict(dataset=config.get('dataset'), split_id=config.get('split_id'), mode=mode, fold=fold,
                        training_clips_sha=stable_hash(training_clips), backbone=backbone_info['identity'],
                        feature_signature=stable_hash(config.get('feature_manifest')), seeds=seeds,
                        selected_epochs=record.get('selected_epochs'), preprocessing_id=preprocessing['preprocessing_id'],
                        config=_config_without_evaluation(config))
        member = dict(member_id=f'fold{fold}' if mode == 'nested_cv' else 'holdout', fold=fold,
                      kind='outer_refit' if mode == 'nested_cv' else 'holdout_best_validation',
                      stage_dir=rel(stage), fitted_artifact_id=fitted_artifact_id(identity),
                      role_selector=rel(required['role_selector']), numeric_scaler=rel(required['numeric_scaler']),
                      trees=rel(required['trees']) if 'trees' in required else None,
                      class_prior=None if prior_info is None else prior_info['prior'],
                      class_prior_source=None if prior_info is None else prior_info.get('source', 'training_labels_at_fit'),
                      seeds=seed_entries, ensemble=dict(components=names, weights=[list(map(float, w)) for w in record['weights']],
                                                        seed_average='mean'),
                      training_clips=training_clips, training_sources=(component_prov or {}).get('training_sources'),
                      heldout_public_clips=heldout, evaluation_crops=evaluation_crops,
                      backbone=backbone_info)
        for key, path in required.items(): files[rel(path)] = None
        members.append(member)
    for path in files: files[path] = dict(sha256=sha256_file(run / path), bytes=(run / path).stat().st_size)
    feature = dict(variant=config.get('ablation', 'production'), numeric_features=config.get('feature_manifest'),
                   numeric_dimension=len(config.get('feature_manifest') or []),
                   streams=members[0]['backbone'].get('streams'))
    backbones = {json.dumps(m['backbone']['identity'], sort_keys=True) for m in members}
    if len(backbones) != 1: raise InferenceArtifactError('Members disagree about the backbone identity')
    manifest = dict(schema_version=INFERENCE_SCHEMA_VERSION, code_version=INFERENCE_CODE_VERSION,
                    created=time.strftime('%Y-%m-%dT%H:%M:%S'), run_id=run.name, evaluation_mode=mode,
                    dataset_id=dataset_id or config.get('dataset'), split_id=config.get('split_id'),
                    targets=targets, classes=classes, task_weights=[0.7, 0.3], config=config,
                    preprocessing=preprocessing, feature=feature, backbone=members[0]['backbone'],
                    combination=dict(across_members='mean', default_routing='ensemble',
                                     note='Private/new clips: mean of member ensembles. Public OOF replay: each clip uses only '
                                          'the member whose outer fold held it out.'),
                    members=members, files=files)
    path = run / 'inference_manifest.json'
    if path.exists() and not overwrite: raise InferenceArtifactError(f'{path} exists')
    tmp = path.with_name(f'.{path.name}.{os.getpid()}.tmp'); tmp.write_text(json.dumps(manifest, indent=2)); tmp.replace(path)
    if verbose: print(f'Inference manifest: {path} ({len(members)} member(s), {len(files)} checksummed files)')
    return manifest


# ------------------------------------------------------------------------------------------- load/validate
def load_inference_manifest(run_dir, verify_checksums=True):
    run = Path(run_dir)
    path = run / 'inference_manifest.json'
    if not path.exists():
        raise InferenceArtifactError(f'{path} not found. For a run trained before this update, set '
                                     'MIGRATE_LEGACY_RUN=True once to export it from the saved artifacts.')
    manifest = json.loads(path.read_text())
    if manifest.get('schema_version') != INFERENCE_SCHEMA_VERSION:
        raise InferenceArtifactError(f'Unsupported manifest schema {manifest.get("schema_version")}')
    if manifest['targets'] != ['kick_direction', 'goalkeeper_direction'] or manifest['classes'] != [
            ['left', 'middle', 'right'], ['left', 'stands in the middle', 'right']] or not np.allclose(manifest['task_weights'], [.7, .3]):
        raise InferenceArtifactError('Manifest target/class order or task weights are incompatible')
    problems = []
    for rel, info in manifest['files'].items():
        p = run / rel
        if not p.exists(): problems.append(f'missing: {p}'); continue
        if p.stat().st_size != info['bytes']: problems.append(f'size changed: {p}'); continue
        if verify_checksums and sha256_file(p) != info['sha256']: problems.append(f'checksum changed: {p}')
    for m in manifest['members']:
        referenced = [m['role_selector'], m['numeric_scaler']] + [s['weights'] for s in m['seeds']] + ([m['trees']] if m['trees'] else [])
        for rel in referenced:
            if rel not in manifest['files']: problems.append(f'unlisted artifact: {run / rel}')
        if not m['seeds']: problems.append(f'{m["member_id"]}: no seed weights')
        names = m['ensemble']['components']
        if len(m['ensemble']['weights']) != 2 or any(len(w) != len(names) for w in m['ensemble']['weights']):
            problems.append(f'{m["member_id"]}: ensemble weights do not match components')
        if 'class_prior' in names and m['class_prior'] is None: problems.append(f'{m["member_id"]}: class prior missing')
        if 'pose_motion_tree' in names and not m['trees']: problems.append(f'{m["member_id"]}: trees missing')
    if problems: raise InferenceArtifactError('Inference artifacts are incomplete or modified:\n' + '\n'.join(problems))
    return manifest


# ------------------------------------------------------------------------------------------- routing
def route_clips(manifest, clip_names, routing):
    """Return {clip: [member_ids]} and an exposure label per clip."""
    members = manifest['members']
    routes = {}; exposure = {}
    for clip in clip_names:
        trained = [m['member_id'] for m in members if clip in set(m['training_clips'])]
        held = [m['member_id'] for m in members if clip in set(m['heldout_public_clips'])]
        if routing == 'oof_replay':
            if len(held) != 1:
                raise InferenceArtifactError(f'{clip}: OOF replay needs exactly one held-out member, found {held}')
            routes[clip] = held; exposure[clip] = 'out_of_fold'
        elif routing == 'ensemble':
            routes[clip] = [m['member_id'] for m in members]
            exposure[clip] = 'in_sample' if trained else 'not_in_training'
        else:
            raise ValueError(f'Unknown routing {routing}')
    return routes, exposure


def validate_probability_arrays(arrays, count, classes):
    if len(arrays) != 2: raise InferenceArtifactError('Expected two target probability arrays')
    for p, c in zip(arrays, classes):
        p = np.asarray(p)
        if p.shape != (count, len(c)) or not np.isfinite(p).all() or (p < -1e-7).any() or (p > 1 + 1e-6).any():
            raise InferenceArtifactError(f'Invalid probability array shape/value: {p.shape}')
        if not np.allclose(p.sum(1), 1, atol=1e-5): raise InferenceArtifactError('Probability rows must sum to one')
    return arrays


def combine_member(member, components, count):
    """Apply the member's frozen per-task ensemble weights to its saved components."""
    names = member['ensemble']['components']
    missing = [n for n in names if n not in components]
    if missing: raise InferenceArtifactError(f'{member["member_id"]}: missing components {missing}')
    return [sum(w * np.asarray(components[n][j]) for w, n in zip(member['ensemble']['weights'][j], names))
            for j in range(2)]


# ------------------------------------------------------------------------------------------- caches
def clip_cache_key(clip, content_id, preprocessing_id):
    """Geometry/crop cache key: decoded frames, detections and crops do not depend on metadata."""
    return stable_hash(dict(clip=clip, content=content_id, preprocessing=preprocessing_id, code=INFERENCE_CODE_VERSION), 20)


def prediction_cache_key(crop_key, metadata_row, fitted_artifact_id, feature_signature):
    """Prediction key: every effective model input. Metadata (preferred foot, gender) enters the numeric
    branch and the trees, so changing it must recompute; new weights (fitted id) invalidate too."""
    return stable_hash(dict(crop=crop_key, metadata=[float(x) for x in metadata_row], fitted=fitted_artifact_id,
                            features=feature_signature, code=INFERENCE_CODE_VERSION), 24)


def content_id_for(path, cache):
    """sha256 of the video file, memoized by (path, size, mtime)."""
    p = Path(path); stat = p.stat(); key = f'{p}|{stat.st_size}|{int(stat.st_mtime)}'
    if key not in cache: cache[key] = sha256_file(p)
    return cache[key]


def prepare_video_tensor(video_uint8, image_mean, image_std):
    """Identical arithmetic to ClipDataset without augmentation: views x T x C x H x W."""
    x = np.asarray(video_uint8).astype(np.float32) / 255.
    mean = np.asarray(image_mean, dtype=np.float32)[None, None, :, None, None]
    std = np.asarray(image_std, dtype=np.float32)[None, None, :, None, None]
    return (x - mean) / std


def tree_probabilities(trees, features, classes):
    out = []
    for j, c in enumerate(classes):
        prob = np.zeros((len(features), len(c)))
        prob[:, np.asarray(trees[j].classes_, dtype=int)] = trees[j].predict_proba(features)
        out.append(prob)
    return out


# ------------------------------------------------------------------------------------------- runner
def run_saved_inference(run_dir, clips, *, hooks, routing='ensemble', verify_checksums=True, device='cpu',
                        dataset_id=None, output_name=None, compare_oof=True, output_root=None):
    """Predict saved models on `clips` (DataFrame: clip, path, preferred_foot, gender[, source_id]).

    hooks: build_view(record, selector, member) -> view dict (label-free decode/detect/crop),
           save_view(view, path), load_view(path, decode_video) -> cached view (decoded 'video' when requested),
           model_factory(manifest, member) -> torch module with saved architecture (no pretrained download),
           numeric_features(view, metadata_row, feature_config) -> 1-D float array,
           load_joblib(path).
    """
    import torch
    run = Path(run_dir)
    manifest = load_inference_manifest(run, verify_checksums)
    before = {rel: info['sha256'] for rel, info in manifest['files'].items()}
    classes = manifest['classes']; pre = manifest['preprocessing']
    clips = clips.reset_index(drop=True)
    if not clips['clip'].is_unique: raise InferenceArtifactError('Requested clip IDs must be unique')
    names = clips['clip'].tolist()
    routes, exposure = route_clips(manifest, names, routing)
    content_cache = {}
    content = {r.clip: content_id_for(r.path, content_cache) for r in clips.itertuples()}
    metadata = np.column_stack([(clips.preferred_foot == 'left'), (clips.preferred_foot == 'right'),
                                (clips.gender == 'F'), (clips.gender == 'M')]).astype(np.float32)
    # Caches/outputs live under output_root (default: <run>/inference). Notebooks pass a folder outside the
    # trained run, so an imported or still-running run is never written to.
    base = Path(output_root) if output_root is not None else run / 'inference'
    stats = dict(crops_reused_training=0, crops_reused_inference=0, crops_built=0, predictions_reused=0, predictions_computed=0)
    member_outputs = {}
    feature_config = manifest['feature']
    for member in manifest['members']:
        needed = [i for i, c in enumerate(names) if member['member_id'] in routes[c]]
        if not needed: continue
        member_dir = base / 'predictions' / member['fitted_artifact_id']
        crop_dir = base / 'crops' / f'{member["member_id"]}__{member["fitted_artifact_id"][:12]}__{pre["preprocessing_id"][:12]}'
        member_dir.mkdir(parents=True, exist_ok=True); crop_dir.mkdir(parents=True, exist_ok=True)
        crop_keys = {i: clip_cache_key(names[i], content[names[i]], pre['preprocessing_id']) for i in needed}
        feature_signature = stable_hash(manifest['feature'])
        keys = {i: prediction_cache_key(crop_keys[i], metadata[i], member['fitted_artifact_id'], feature_signature)
                for i in needed}
        pending = [i for i in needed if not (member_dir / f'{keys[i]}.npz').exists()]
        components = {}
        if pending:
            selector = None; views = {}
            for i in pending:
                crop_path = crop_dir / f'{crop_keys[i]}.pkl'
                training_crop = member['evaluation_crops'].get(names[i])
                if crop_path.exists():
                    stats['crops_reused_inference'] += 1; views[i] = crop_path
                elif (training_crop and dataset_id is not None and dataset_id == manifest['dataset_id']
                      and (run / training_crop).exists()):   # bundles may omit training-time crops
                    stats['crops_reused_training'] += 1; views[i] = run / training_crop
                else:
                    if selector is None: selector = hooks['load_joblib'](run / member['role_selector'])
                    hooks['save_view'](hooks['build_view'](clips.iloc[i], selector, member), crop_path)
                    stats['crops_built'] += 1; views[i] = crop_path
            scaler = hooks['load_joblib'](run / member['numeric_scaler'])
            raw = np.stack([hooks['numeric_features'](hooks['load_view'](views[i], decode_video=False), metadata[i],
                                                      feature_config) for i in pending]).reshape(len(pending), -1)
            numeric = scaler.transform(raw).astype(np.float32)
            seed_probs = {}
            for entry in member['seeds']:
                model = hooks['model_factory'](manifest, member)
                state = torch.load(run / entry['weights'], map_location='cpu', weights_only=True)
                if 'optimizer' in state: raise InferenceArtifactError(f'{entry["weights"]} is a resumable checkpoint, not inference weights')
                model.load_state_dict(state['model'], strict=True); model.to(device).eval()
                rows = [[] for _ in classes]
                with torch.inference_mode():
                    for k, i in enumerate(pending):  # stream one decoded clip at a time (bounded memory)
                        view = hooks['load_view'](views[i], decode_video=True)
                        video = torch.from_numpy(prepare_video_tensor(view['video'], pre['image_mean'], pre['image_std']))[None]
                        n = torch.from_numpy(numeric[k:k + 1])
                        with torch.autocast(device_type='cuda' if str(device).startswith('cuda') else 'cpu',
                                            dtype=torch.float16, enabled=str(device).startswith('cuda')):
                            logits = model(video.to(device), n.to(device))
                        for j, p in enumerate(logits): rows[j].append(p.float().softmax(1).cpu().numpy())
                seed_probs[entry['seed']] = [np.concatenate(r) for r in rows]
                del model, state
            tree_probs = None
            if member['trees']:
                tree_probs = tree_probabilities(hooks['load_joblib'](run / member['trees']), raw, classes)
            for k, i in enumerate(pending):
                arrays = {f'seed{s}_task{j}': p[j][k] for s, p in seed_probs.items() for j in range(2)}
                if tree_probs is not None: arrays.update({f'tree_task{j}': tree_probs[j][k] for j in range(2)})
                tmp = member_dir / f'.{keys[i]}.{os.getpid()}.tmp.npz'
                np.savez(tmp, **arrays, clip=np.array(names[i]), content=np.array(content[names[i]]),
                         metadata=np.asarray(metadata[i]))
                tmp.replace(member_dir / f'{keys[i]}.npz')
            stats['predictions_computed'] += len(pending)
        stats['predictions_reused'] += len(needed) - len(pending)
        cached = [np.load(member_dir / f'{keys[i]}.npz') for i in needed]
        for c, i in zip(cached, needed):
            if (str(c['clip']) != names[i] or str(c['content']) != content[names[i]]
                    or not np.array_equal(c['metadata'], metadata[i])):
                raise InferenceArtifactError(f'Prediction cache mismatch for {names[i]}')
        seeds = [e['seed'] for e in member['seeds']]
        for s in seeds:
            components[f'video_seed{s}'] = [np.stack([c[f'seed{s}_task{j}'] for c in cached]) for j in range(2)]
        components['video_ensemble'] = [sum(components[f'video_seed{s}'][j] for s in seeds) / len(seeds) for j in range(2)]
        if member['trees']:
            components['pose_motion_tree'] = [np.stack([c[f'tree_task{j}'] for c in cached]) for j in range(2)]
        if member['class_prior'] is not None:
            components['class_prior'] = [np.tile(np.asarray(member['class_prior'][j]), (len(needed), 1)) for j in range(2)]
        components['ensemble'] = combine_member(member, components, len(needed))
        for name, arrays in components.items(): validate_probability_arrays(arrays, len(needed), classes)
        member_outputs[member['member_id']] = dict(indices=needed, components=components)
    final = [np.zeros((len(names), len(c))) for c in classes]
    counts = np.zeros(len(names))
    for member_id, out in member_outputs.items():
        for j in range(2): final[j][out['indices']] += out['components']['ensemble'][j]
        counts[out['indices']] += 1
    if (counts == 0).any(): raise InferenceArtifactError('Some clips received no member prediction')
    final = [p / counts[:, None] for p in final]
    validate_probability_arrays(final, len(names), classes)
    after = {rel: sha256_file(run / rel) for rel in before} if verify_checksums else before
    if after != before: raise InferenceArtifactError('A fitted artifact changed during inference')
    input_id = stable_hash(dict(clips=names, content=[content[c] for c in names], routing=routing,
                                metadata=metadata.tolist(),
                                preprocessing=pre['preprocessing_id'], members=[m['fitted_artifact_id'] for m in manifest['members']]))
    out_dir = base / 'runs' / (output_name or f'{routing}_{input_id}'); out_dir.mkdir(parents=True, exist_ok=True)
    table = pd.DataFrame({'clip': names, 'exposure': [exposure[c] for c in names],
                          'members': [';'.join(routes[c]) for c in names]})
    for j, (target, cs) in enumerate(zip(manifest['targets'], classes)):
        table[target] = np.asarray(cs)[final[j].argmax(1)]
        for k, label in enumerate(cs): table[f'{target}_probability_{k}__{label}'] = final[j][:, k]
    table.to_csv(out_dir / 'final_probabilities.csv', index=False)
    np.savez_compressed(out_dir / 'probabilities.npz', clip=np.asarray(names, dtype=str),
                        **{f'final_task{j}': final[j] for j in range(2)},
                        **{f'{mid}__{name}__task{j}': arr[j] for mid, out in member_outputs.items()
                           for name, arr in out['components'].items() for j in range(2)},
                        **{f'{mid}__indices': np.asarray(out['indices']) for mid, out in member_outputs.items()})
    comparison = None
    if compare_oof and routing == 'oof_replay' and (run / 'predictions.npz').exists() and manifest['evaluation_mode'] == 'nested_cv':
        saved = np.load(run / 'predictions.npz')
        order = {c: i for i, c in enumerate(json.loads((run / 'experiment_manifest.json').read_text())['split_plan']['identity']['clips'])}
        idx = [order[c] for c in names]
        comparison = {manifest['targets'][j]: float(np.abs(saved[f'oof_{j}'][idx] - final[j]).max()) for j in range(2)}
    record = dict(trained_run_dir=str(run), input=dict(count=len(names), routing=routing, input_id=input_id),
                  backbone=manifest['backbone']['identity'], members=[m['member_id'] for m in manifest['members']],
                  fitted_artifact_ids={m['member_id']: m['fitted_artifact_id'] for m in manifest['members']},
                  preprocessing_id=pre['preprocessing_id'], cache_statistics=stats,
                  exposure_counts=table.exposure.value_counts().to_dict(), max_abs_difference_vs_saved_oof=comparison,
                  note=('OOF replay: each public clip used only its held-out fold model.' if routing == 'oof_replay' else
                        'Fold-ensemble predictions; rows marked in_sample were seen by a training member and are not OOF.'),
                  labels_used=False, fitting_performed=False)
    (out_dir / 'inference_run.json').write_text(json.dumps(record, indent=2, default=str))
    return dict(final=final, table=table, members=member_outputs, record=record, output_dir=out_dir, manifest=manifest)


def write_submission(result, template, output_path):
    table = result['table']; manifest = result['manifest']
    submission = template[['clip']].merge(table[['clip'] + manifest['targets']], on='clip', how='left', validate='one_to_one')
    if submission.isna().any().any() or len(submission) != len(template):
        raise InferenceArtifactError('Submission does not cover every template clip')
    submission.to_csv(output_path, index=False)
    return submission


## Diagnostics helpers and CPU-only smoke checks
These checks run automatically in smoke/train/resume modes before data loading. It embeds all reporting, tracking, and ablation code and produces small synthetic reports under `SCRATCH_ROOT/diagnostics_smoke`. It does not train a real video model or estimate competition performance. The GPU pipeline smoke is `RUN_MODE='smoke'`.


In [ ]:
# CELL: diagnostics_helpers
"""Strict two-target OOF exports. Embedded verbatim into the Colab notebook."""
import json
from pathlib import Path
import numpy as np
import pandas as pd


def validate_component_probabilities(probabilities, count, class_order, expected_classes):
    if [list(x) for x in class_order] != [list(x) for x in expected_classes]:
        raise ValueError('Probability class order does not match the saved label mapping')
    if len(probabilities) != 2 or len(expected_classes) != 2:
        raise ValueError('Exactly kick direction and goalkeeper direction are required')
    arrays=[]
    for p,classes in zip(probabilities,expected_classes):
        p=np.asarray(p,dtype=float)
        if p.shape != (count,len(classes)):
            raise ValueError(f'Probability shape {p.shape}; expected {(count,len(classes))}')
        if not np.isfinite(p).all() or (p<0).any() or (p>1).any():
            raise ValueError('Probabilities must be finite and in [0,1]')
        if not np.allclose(p.sum(axis=1),1,rtol=0,atol=1e-5):
            raise ValueError('Probability rows must sum to one')
        arrays.append(p)
    return arrays


def component_metrics(truth, probabilities, targets, classes, weights):
    truth=np.asarray(truth)
    if truth.shape!=(len(truth),2) or not len(truth):
        raise ValueError('Need a nonempty aligned two-target label matrix')
    if list(targets)!=['kick_direction','goalkeeper_direction'] or not np.allclose(weights,[.7,.3]):
        raise ValueError('Competition metric must be 70% kick / 30% goalkeeper')
    arrays=validate_component_probabilities(probabilities,len(truth),classes,classes)
    result={}; weighted_accuracy=0.; weighted_ce=0.
    for j,(target,p) in enumerate(zip(targets,arrays)):
        y=truth[:,j]
        if not np.isfinite(y).all() or not np.equal(y,y.astype(int)).all() or (y<0).any() or (y>=len(classes[j])).any():
            raise ValueError('Invalid class index in truth')
        y=y.astype(int)
        accuracy=float((p.argmax(1)==y).mean())
        ce=float(-np.log(np.clip(p[np.arange(len(y)),y],1e-12,1)).mean())
        result[target+'_accuracy']=accuracy; result[target+'_cross_entropy']=ce
        weighted_accuracy+=float(weights[j])*accuracy; weighted_ce+=float(weights[j])*ce
    return dict(clips=len(truth),weighted_accuracy=weighted_accuracy,
                weighted_cross_entropy=weighted_ce,**result)


def select_reference_components(inner_components,inner_truth,targets,classes,weights):
    """Call only on inner-validation predictions; independent reference per task."""
    if not inner_components: raise ValueError('No inner components to select from')
    metrics={name:component_metrics(inner_truth,p,targets,classes,weights)
             for name,p in inner_components.items()}
    return [min(metrics,key=lambda name:(-metrics[name][target+'_accuracy'],
            metrics[name][target+'_cross_entropy'],name)) for target in targets]


class ComponentOOF:
    def __init__(self,frame,truth,fold_ids,targets,classes,weights):
        self.frame=frame.reset_index(drop=True).copy(); self.truth=np.asarray(truth)
        self.folds=np.asarray(fold_ids); self.targets=list(targets)
        self.classes=[list(c) for c in classes]; self.weights=np.asarray(weights,dtype=float)
        n=len(self.frame)
        if not {'clip','source_id'}<=set(frame) or not frame['clip'].is_unique:
            raise ValueError('Unique clip IDs and source_id are required')
        if self.truth.shape!=(n,2) or self.folds.shape!=(n,) or not n:
            raise ValueError('Frame, labels, and saved fold assignments must align')
        if not np.issubdtype(self.folds.dtype,np.integer) or (self.folds<0).any():
            raise ValueError('Every evaluated clip must have a saved outer fold')
        if self.frame.source_id.isna().any(): raise ValueError('Missing source ID')
        groups=self.frame.assign(_fold=self.folds).groupby('source_id',dropna=False)._fold.nunique()
        if (groups!=1).any(): raise ValueError('One source appears in multiple outer folds')
        if len(targets)!=2 or not np.allclose(self.weights,[.7,.3]):
            raise ValueError('Expected two targets and 70/30 metric weights')
        # Validate labels even before any probability assignment.
        component_metrics(self.truth,[np.full((n,len(c)),1/len(c)) for c in self.classes],
                          self.targets,self.classes,self.weights)
        for j,target in enumerate(self.targets):
            if target in self.frame:
                expected=np.asarray(self.classes[j])[self.truth[:,j].astype(int)]
                if not np.array_equal(self.frame[target].to_numpy(),expected):
                    raise ValueError('Frame labels do not match the supplied label matrix')
        self.probabilities={}; self.coverage={}

    def add(self,component,indices,probabilities,clip_ids,class_order):
        ids=np.asarray(indices)
        if ids.ndim!=1 or not np.issubdtype(ids.dtype,np.integer) or not len(ids):
            raise ValueError('Prediction indices must be a nonempty integer vector')
        if len(np.unique(ids))!=len(ids) or (ids<0).any() or (ids>=len(self.frame)).any():
            raise ValueError('Duplicate or invalid prediction index')
        if not np.array_equal(np.asarray(clip_ids),self.frame['clip'].iloc[ids].to_numpy()):
            raise ValueError('Prediction clip order is misaligned')
        arrays=validate_component_probabilities(probabilities,len(ids),class_order,self.classes)
        if component not in self.probabilities:
            self.probabilities[component]=[np.full((len(self.frame),len(c)),np.nan) for c in self.classes]
            self.coverage[component]=np.zeros(len(self.frame),dtype=bool)
        if self.coverage[component][ids].any(): raise ValueError('Duplicate OOF predictions')
        for saved,p in zip(self.probabilities[component],arrays): saved[ids]=p
        self.coverage[component][ids]=True

    def require_complete(self):
        if not self.probabilities: raise ValueError('No component predictions')
        for name,p in self.probabilities.items():
            if not self.coverage[name].all():
                raise ValueError(f'Incomplete OOF coverage for {name}: {int(self.coverage[name].sum())}/{len(self.frame)}')
            validate_component_probabilities(p,len(self.frame),self.classes,self.classes)

    def error_rows(self,quality=None):
        self.require_complete(); chunks=[]
        for name,probs in self.probabilities.items():
            columns=[x for x in ['clip','source_id','preferred_foot','gender'] if x in self.frame]
            result=self.frame[columns].copy(); result['fold']=self.folds; result['component']=name
            result['weighted_loss']=0.; result['weighted_error']=0.
            for j,target in enumerate(self.targets):
                p=probs[j]; y=self.truth[:,j].astype(int); pred=p.argmax(1)
                result[target+'_true']=np.asarray(self.classes[j])[y]
                result[target+'_predicted']=np.asarray(self.classes[j])[pred]
                result[target+'_true_probability']=p[np.arange(len(y)),y]
                ce=-np.log(np.clip(p[np.arange(len(y)),y],1e-12,1))
                result[target+'_cross_entropy']=ce; result[target+'_correct']=pred==y
                for k,label in enumerate(self.classes[j]):
                    result[f'{target}_probability_{k}__{label}']=p[:,k]
                result['weighted_loss']+=float(self.weights[j])*ce
                result['weighted_error']+=float(self.weights[j])*(pred!=y)
            if quality is not None:
                if 'clip' not in quality or not quality['clip'].is_unique:
                    raise ValueError('Quality indicators require one row per clip')
                if set(quality['clip'])!=set(self.frame['clip']):
                    raise ValueError('Quality indicators must cover precisely the evaluated clips')
                overlap=(set(quality)-{'clip'})&set(result)
                if overlap: raise ValueError(f'Quality columns overlap report columns: {sorted(overlap)}')
                result=result.merge(quality,on='clip',how='left',validate='one_to_one',sort=False)
            chunks.append(result)
        return pd.concat(chunks,ignore_index=True)

    def metric_table(self):
        self.require_complete(); records=[]
        for name,p in self.probabilities.items():
            for fold in [*sorted(np.unique(self.folds)),'pooled']:
                ids=np.arange(len(self.frame)) if fold=='pooled' else np.flatnonzero(self.folds==fold)
                records.append(dict(component=name,fold=fold,**component_metrics(self.truth[ids],
                    [a[ids] for a in p],self.targets,self.classes,self.weights)))
        return pd.DataFrame(records)

    def ensemble_comparison(self,references,ensemble='ensemble'):
        self.require_complete()
        if ensemble not in self.probabilities: raise ValueError('Missing final ensemble')
        records=[]
        for fold in sorted(np.unique(self.folds)):
            reference=references.get(int(fold),references.get(str(int(fold))))
            if reference is None or len(reference)!=2: raise ValueError('Missing inner-selected references')
            ids=np.flatnonzero(self.folds==fold)
            for j,target in enumerate(self.targets):
                if reference[j] not in self.probabilities: raise ValueError('Reference component not exported')
                y=self.truth[ids,j]; a=self.probabilities[reference[j]][j][ids].argmax(1)
                b=self.probabilities[ensemble][j][ids].argmax(1)
                corrected=int(((a!=y)&(b==y)).sum()); harmed=int(((a==y)&(b!=y)).sum())
                records.append(dict(fold=int(fold),target=target,reference_component=reference[j],clips=len(ids),
                    mistakes_corrected=corrected,correct_changed_to_mistake=harmed,
                    disagreements=int((a!=b).sum()),disagreement_rate=float((a!=b).mean()),
                    accuracy_delta=(corrected-harmed)/len(ids),
                    weighted_score_delta=float(self.weights[j])*(corrected-harmed)/len(ids),
                    contribution_to_pooled_weighted_delta=float(self.weights[j])*(corrected-harmed)/len(self.frame)))
        for j,target in enumerate(self.targets):
            subset=[r for r in records if r['target']==target]
            corrected=sum(r['mistakes_corrected'] for r in subset); harmed=sum(r['correct_changed_to_mistake'] for r in subset)
            disagreements=sum(r['disagreements'] for r in subset); n=len(self.frame)
            records.append(dict(fold='pooled',target=target,reference_component='fold-specific inner-selected reference',
                clips=n,mistakes_corrected=corrected,correct_changed_to_mistake=harmed,disagreements=disagreements,
                disagreement_rate=disagreements/n,accuracy_delta=(corrected-harmed)/n,
                weighted_score_delta=float(self.weights[j])*(corrected-harmed)/n,
                contribution_to_pooled_weighted_delta=float(self.weights[j])*(corrected-harmed)/n))
        return pd.DataFrame(records)

    def export(self,output,quality=None,references=None,ensemble='ensemble'):
        self.require_complete(); output=Path(output); output.mkdir(parents=True,exist_ok=True)
        errors=self.error_rows(quality); metrics=self.metric_table()
        slices,confusions=component_slice_reports(errors,self.targets,self.classes,self.weights,len(self.frame))
        errors.to_csv(output/'component_clip_errors.csv',index=False)
        metrics.to_csv(output/'component_metrics.csv',index=False)
        errors.sort_values(['component','weighted_loss'],ascending=[True,False],kind='stable').to_csv(output/'highest_loss.csv',index=False)
        errors.sort_values(['component','weighted_error','weighted_loss'],ascending=[True,False,False],kind='stable').to_csv(output/'greatest_accuracy_error.csv',index=False)
        slices.to_csv(output/'component_slices.csv',index=False); confusions.to_csv(output/'component_confusions.csv',index=False)
        manifest=dict(targets=self.targets,classes=self.classes,weights=self.weights.tolist(),
            components=list(self.probabilities),clips=self.frame['clip'].tolist(),fold_ids=self.folds.tolist(),
            interpretation='Outer-fold diagnostics are exploratory development feedback, not an untouched final estimate. '
                'Loss is cross-entropy; score is accuracy. Slices overlap and are not additive. Quality associations are not causal.',
            probability_columns={target:[f'{target}_probability_{k}__{label}' for k,label in enumerate(classes)]
                                 for target,classes in zip(self.targets,self.classes)})
        if references is not None:
            manifest['inner_selected_reference_components']={str(k):list(v) for k,v in references.items()}
        (output/'component_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
        arrays={'clip':self.frame['clip'].to_numpy(dtype=str),'fold_ids':self.folds,'truth':self.truth}
        for index,(name,probs) in enumerate(self.probabilities.items()):
            for j,p in enumerate(probs): arrays[f'component_{index}_target_{j}']=p
        np.savez_compressed(output/'component_oof_probabilities.npz',**arrays)
        result=dict(errors=errors,metrics=metrics,slices=slices,confusions=confusions)
        if references is not None:
            comparison=self.ensemble_comparison(references,ensemble)
            comparison.to_csv(output/'ensemble_comparison.csv',index=False); result['comparison']=comparison
        return result


def component_slice_reports(errors,targets,classes,weights,population):
    """Each component has the same complete evaluated population denominator."""
    data=errors.copy(); fields=[c for c in ['source_id','preferred_foot','gender'] if c in data]
    fields += [t+'_true' for t in targets]
    # Bin continuous quality indicators; retain categorical fields as reported.
    quality_fields=[c for c in data if any(k in c.lower() for k in
        ['tracking','tracked','coverage','pose','camera','fallback','jump','gap','confidence','clipping'])
        and not c.startswith(tuple(targets)) and c not in fields]
    for c in quality_fields:
        if pd.api.types.is_numeric_dtype(data[c]) and not pd.api.types.is_bool_dtype(data[c]):
            name=c+'_slice'
            if any(k in c for k in ['fraction','coverage','confidence','rate']):
                data[name]=pd.cut(data[c],[-np.inf,0,.5,.9,np.inf],labels=['zero_or_less','low_(0,.5]','medium_(.5,.9]','high_(.9,+inf)']).astype('object').fillna('missing')
            else:
                data[name]=pd.cut(data[c],[-np.inf,0,1,5,np.inf],labels=['zero_or_less','(0,1]','(1,5]','above_5']).astype('object').fillna('missing')
            fields.append(name)
        elif data[c].nunique(dropna=False)<=30: fields.append(c)
    records=[]; matrices=[]
    for component,part in data.groupby('component',sort=False):
        if len(part)!=population: raise ValueError('Slice denominator must be the full evaluated population')
        for field in fields:
            for value,g in part.groupby(field,dropna=False,observed=True):
                row=dict(component=component,slice_field=field,slice_value=str(value),clips=len(g),
                    small_slice=len(g)<20,weighted_accuracy=1-float(g.weighted_error.mean()),
                    mean_weighted_loss=float(g.weighted_loss.mean()),
                    contribution_to_total_weighted_errors=float(g.weighted_error.sum()/population),
                    population_denominator=population)
                for j,t in enumerate(targets):
                    row[t+'_accuracy']=float(g[t+'_correct'].mean())
                    row[t+'_mean_cross_entropy']=float(g[t+'_cross_entropy'].mean())
                    row[t+'_weighted_error_contribution']=float(weights[j])*(~g[t+'_correct']).sum()/population
                records.append(row)
        for j,t in enumerate(targets):
            for true in classes[j]:
                for predicted in classes[j]:
                    matrices.append(dict(component=component,target=t,true_label=true,predicted_label=predicted,
                        clips=int(((part[t+'_true']==true)&(part[t+'_predicted']==predicted)).sum())))
    return pd.DataFrame(records),pd.DataFrame(matrices)


def smoke_component_reports(output):
    """Synthetic smoke scores test arithmetic only; they are not model performance."""
    targets=['kick_direction','goalkeeper_direction']; classes=[['left','middle','right'],['left','stands in the middle','right']]
    weights=[.7,.3]; y=np.array([[0,1],[1,2],[2,0],[0,1]])
    frame=pd.DataFrame({'clip':['a','b','c','d'],'source_id':['s0','s0','s1','s1'],'gender':['F','M','F','M']})
    folds=np.array([0,0,1,1]); base=[np.eye(3)[y[:,j]]*.8+.2/3 for j in range(2)]
    worse=[p.copy() for p in base]; worse[0][0]=[.1,.8,.1]; worse[1][1]=[.8,.1,.1]
    report=ComponentOOF(frame,y,folds,targets,classes,weights)
    report.add('video_seed42',[0,1],[p[:2] for p in worse],frame['clip'][:2],classes)
    try: report.require_complete()
    except ValueError: pass
    else: raise AssertionError('Incomplete OOF predictions accepted')
    report.add('video_seed42',[2,3],[p[2:] for p in worse],frame['clip'][2:],classes)
    report.add('ensemble',np.arange(4),base,frame['clip'],classes)
    for bad_ids,bad_clips,bad_classes in [([0],['wrong'],classes),([0],['a'],[classes[0][::-1],classes[1]])]:
        try: report.add('bad',bad_ids,[p[:1] for p in base],bad_clips,bad_classes)
        except ValueError: pass
        else: raise AssertionError('Misalignment accepted')
    for bad_probabilities in [[np.full((1,3),np.nan),base[1][:1]],
                              [np.zeros((1,3)),base[1][:1]],
                              [np.ones((1,2))*.5,base[1][:1]]]:
        try: report.add('bad',[0],bad_probabilities,['a'],classes)
        except ValueError: pass
        else: raise AssertionError('Invalid probability array accepted')
    try: report.add('video_seed42',[0],[p[:1] for p in base],['a'],classes)
    except ValueError: pass
    else: raise AssertionError('Duplicate OOF prediction accepted')
    try: ComponentOOF(frame,y,np.array([0,1,1,1]),targets,classes,weights)
    except ValueError: pass
    else: raise AssertionError('Source leakage accepted')
    quality=pd.DataFrame({'clip':frame['clip'],'kicker_tracked_fraction':[0,.5,1,1],
                          'pose_valid_fraction':[0,.2,.8,1],'camera_valid_fraction':[0,1,1,1]})
    results=report.export(output,quality=quality,references={0:['video_seed42']*2,1:['video_seed42']*2})
    e=results['errors'].query("component=='video_seed42'")
    np.testing.assert_allclose(e.weighted_error,[.7,.3,0,0])
    np.testing.assert_allclose(e.weighted_loss,.7*e.kick_direction_cross_entropy+.3*e.goalkeeper_direction_cross_entropy)
    pooled=results['comparison'].query("fold=='pooled'")
    np.testing.assert_allclose(pooled.weighted_score_delta.sum(),.25)
    source_slices=results['slices'].query("component=='video_seed42' and slice_field=='source_id'")
    np.testing.assert_allclose(source_slices.contribution_to_total_weighted_errors.sum(),.25)
    assert results['confusions'].groupby(['component','target']).clips.sum().eq(4).all()
    selected=select_reference_components({'good':base,'bad':worse},y,targets,classes,weights)
    assert selected==['good','good']
    return dict(status='PASS',checks=['complete OOF coverage','clip/class alignment','70/30 accuracy and CE',
        'slice population denominator','ensemble corrections','CSV/NPZ export'],
        note='Synthetic execution check only; scores are not performance estimates.')


"""Post-prediction role/tracking diagnostics; embed this file as a notebook cell.

These functions never generate crops, fit selectors, or change model inputs.
Click agreement is a localization proxy, not an identity annotation.
"""
import json
from pathlib import Path
import numpy as np
import pandas as pd


def _inside_box(point, box):
    return bool(np.all(point >= box[:2]) and np.all(point <= box[2:]))


def _longest_missing_run(valid):
    longest = current = 0
    for present in valid:
        current = 0 if present else current + 1
        longest = max(longest, current)
    return longest


def tracking_clip_diagnostics(clip, source_id, fold, raw, view, annotation=None,
                              timestamp_tolerance=.10, jump_threshold_heights=1.25):
    """Return (one clip-quality row, per-frame rows, heldout click rows).

    Track indices refer to the cached detections used to generate the actual crops.
    Missing detections produce full-frame fallback crops in the current pipeline.
    Distances use pixels divided by selected player height, preserving aspect ratio.
    """
    times = np.asarray(raw['times'], dtype=float)
    ids = np.asarray(view.get('track_ids', np.full((len(times), 2), -1)), dtype=int)
    traces = np.asarray(view['traces'])
    if ids.shape != (len(times), 2) or traces.shape != (len(times), 2, 3):
        raise ValueError('Tracking/cache shape does not match sampled timestamps')
    if 'track_ids' not in view:
        raise ValueError('Rebuild old crop caches: selected detection indices are required for auditing')
    base = dict(clip=clip, source_id=source_id, fold=int(fold))
    summary = dict(base)
    frames = []
    role_records = {}
    for r, role in enumerate(['kicker', 'keeper']):
        records = []
        previous_box = None
        previous_t = None
        previous_center = None
        for t, (rgb, detection) in enumerate(zip(raw['frames'], raw['detections'])):
            boxes = np.asarray(detection['boxes']).reshape(-1, 4)
            h, w = rgb.shape[:2]
            j = int(ids[t, r])
            present = 0 <= j < len(boxes)
            if j >= len(boxes):
                raise ValueError('Saved track points outside cached detections')
            record = dict(base, role=role, frame=t, time_seconds=float(times[t]),
                          detection_count=len(boxes), selected_detection=j,
                          selected=present, fallback='none' if present else 'full_frame',
                          role_confidence=float(traces[t, r, 2]) if present else 0.,
                          camera_valid=bool(raw['camera_quality'][t] > 0),
                          camera_quality=float(raw['camera_quality'][t]),
                          pose_visible_fraction=0., crop_clipped=False,
                          raw_jump_player_heights=np.nan, gap_since_previous_selected_seconds=np.nan,
                          suspicious_jump=False)
            for name in ['box_x1','box_y1','box_x2','box_y2',
                         'crop_x1','crop_y1','crop_x2','crop_y2']:
                record[name] = np.nan
            if present:
                box = boxes[j].astype(float)
                center = (box[:2] + box[2:]) / 2
                crop_size = (box[2:] - box[:2]) * [1.7, 1.35]
                intended_lo, intended_hi = center - crop_size/2, center + crop_size/2
                lo, hi = np.clip(intended_lo, 0, 1), np.clip(intended_hi, 0, 1)
                record.update(dict(zip(['box_x1','box_y1','box_x2','box_y2'], box.tolist())))
                record.update(dict(zip(['crop_x1','crop_y1','crop_x2','crop_y2'], np.r_[lo,hi].tolist())))
                record['crop_clipped'] = bool(np.any(intended_lo < 0) or np.any(intended_hi > 1))
                confidence = np.asarray(detection['confidence'])[j]
                record['pose_visible_fraction'] = float((confidence >= .25).mean())
                if previous_box is not None:
                    height = max(float((previous_box[3]-previous_box[1])*h), 2.)
                    jump = float(np.linalg.norm(center*[w,h] - previous_center*[w,h])/height)
                    record['raw_jump_player_heights'] = jump
                    record['gap_since_previous_selected_seconds'] = float(times[t]-previous_t)
                    record['suspicious_jump'] = jump > jump_threshold_heights
                previous_box, previous_center, previous_t = box, center, times[t]
            records.append(record)
        role_records[role] = records
        frames.extend(records)
        selected = np.array([x['selected'] for x in records])
        summary[role+'_tracked_fraction'] = float(selected.mean())
        summary[role+'_fallback_fraction'] = float((~selected).mean())
        summary[role+'_missing_frame_count'] = int((~selected).sum())
        summary[role+'_longest_missing_run_frames'] = _longest_missing_run(selected)
        summary[role+'_tracking_quality'] = ('missing' if not selected.any() else
                                            'partial' if selected.mean() < .8 else 'mostly_tracked')
        summary[role+'_mean_confidence'] = float(np.mean([x['role_confidence'] for x in records]))
        summary[role+'_pose_visible_fraction'] = float(np.mean([x['pose_visible_fraction'] for x in records]))
        summary[role+'_pose_quality'] = ('low' if summary[role+'_pose_visible_fraction'] < .5 else 'high')
        summary[role+'_clipped_crop_fraction'] = float(np.mean([x['crop_clipped'] for x in records]))
        summary[role+'_suspicious_jump_count'] = int(sum(x['suspicious_jump'] for x in records))
        # Observed gaps are sample-grid statistics, not continuous-time identities.
        gaps = [x['gap_since_previous_selected_seconds'] for x in records
                if np.isfinite(x['gap_since_previous_selected_seconds'])]
        summary[role+'_max_selected_gap_seconds'] = float(max(gaps)) if gaps else np.nan
    summary['camera_valid_fraction'] = float((np.asarray(raw['camera_quality']) > 0).mean())
    summary['camera_quality'] = 'low' if summary['camera_valid_fraction'] < .5 else 'high'
    summary['detector_empty_fraction'] = float(np.mean([len(x['boxes']) == 0 for x in raw['detections']]))
    summary['pose_quality'] = 'low' if np.mean([summary[r+'_pose_visible_fraction'] for r in ['kicker','keeper']]) < .5 else 'high'
    summary['tracking_quality'] = 'low' if min(summary[r+'_tracked_fraction'] for r in ['kicker','keeper']) < .8 else 'high'
    click_rows = []
    if annotation is not None:
        for r, role in enumerate(['kicker','keeper']):
            fields = [role+'_click_time_seconds', role+'_x_norm', role+'_y_norm']
            if any(f not in annotation or pd.isna(annotation[f]) for f in fields):
                continue
            click_time = float(annotation[fields[0]])
            point = np.array([annotation[fields[1]], annotation[fields[2]]], dtype=float)
            t = int(np.argmin(np.abs(times-click_time)))
            record = role_records[role][t]
            boxes = np.asarray(raw['detections'][t]['boxes']).reshape(-1,4)
            candidates_inside = [i for i,b in enumerate(boxes) if _inside_box(point,b)]
            selected_inside = bool(record['selected'] and record['selected_detection'] in candidates_inside)
            mismatch = float(abs(times[t]-click_time))
            eligible = mismatch <= timestamp_tolerance
            # Exact click-nearest detector frame diagnoses detection independently,
            # without rerunning role selection or altering sampled crop inputs.
            exact = raw.get('clicked', [])
            exact_d = exact[r] if len(exact)>r else None
            exact_boxes = np.asarray(exact_d['boxes']).reshape(-1,4) if exact_d is not None else np.empty((0,4))
            exact_match = any(_inside_box(point,b) for b in exact_boxes) if exact_d is not None else None
            distance = np.nan
            if record['selected']:
                box = boxes[record['selected_detection']]
                h,w = raw['frames'][t].shape[:2]
                center = (box[:2]+box[2:])/2
                distance = float(np.linalg.norm((point-center)*[w,h])/max((box[3]-box[1])*h,2.))
            if not eligible: failure = 'timestamp_mismatch_unresolved'
            elif not candidates_inside: failure = 'no_click_containing_detection_proxy'
            elif not record['selected']: failure = 'role_missing_with_candidate'
            elif not selected_inside: failure = 'selected_other_box_proxy'
            else: failure = 'click_inside_selected_box_proxy'
            click_rows.append(dict(base,role=role,annotation_time_seconds=click_time,
                sampled_time_seconds=float(times[t]),sample_offset_seconds=mismatch,
                timestamp_eligible=eligible,timestamp_tolerance_seconds=timestamp_tolerance,
                selected=record['selected'],missing_role=not record['selected'],
                click_inside_selected_box=selected_inside,
                click_distance_player_heights=distance,
                detection_count=len(boxes),click_containing_detection_count=len(candidates_inside),
                unmatched_detection_proxy=not bool(candidates_inside),
                failure_category=failure,exact_click_detector_available=exact_d is not None,
                exact_click_frame_offset_seconds=float(abs(exact_d['time']-click_time)) if exact_d is not None else np.nan,
                exact_click_unmatched_detection_proxy=(not exact_match) if exact_d is not None else np.nan))
    return summary, frames, click_rows


def assert_tracking_audit_provenance(indices, mapping, fit_sources=None):
    """Require evidence that audited clips were held out from the role selector."""
    held = set(rows.source_id.iloc[list(indices)].tolist())
    if fit_sources is not None:
        assert not held & set(fit_sources), 'Audit sources overlap learned role fit sources'
    for i in indices:
        provenance = Path(mapping[int(i)]).parent/'provenance.json'
        if provenance.exists():
            entries = json.loads(provenance.read_text())
            matching = [e for e in entries if int(i) in e['prediction_indices']]
            assert len(matching)==1 and matching[0]['kind']=='evaluation', 'Audit requires heldout evaluation crops'
            assert rows.source_id.iloc[int(i)] not in matching[0]['fit_sources']
        elif fit_sources is None:
            raise ValueError('Tracking audit requires saved provenance or explicit role fit sources')


def summarize_role_click_audits(audits):
    """Aggregate rates with explicit timestamp eligibility; empty annotations are valid."""
    summaries=[]
    if not len(audits): return pd.DataFrame(summaries)
    for role,all_rows in audits.groupby('role'):
        aligned=all_rows[all_rows.timestamp_eligible]
        summaries.append(dict(role=role,annotated_clips=len(all_rows),timestamp_eligible_clips=len(aligned),
            timestamp_mismatch_rate=float((~all_rows.timestamp_eligible).mean()),
            mean_timestamp_offset_seconds=float(all_rows.sample_offset_seconds.mean()),
            missing_role_rate=float(aligned.missing_role.mean()) if len(aligned) else np.nan,
            unmatched_detection_proxy_rate=float(aligned.unmatched_detection_proxy.mean()) if len(aligned) else np.nan,
            click_inside_selected_box_rate=float(aligned.click_inside_selected_box.mean()) if len(aligned) else np.nan,
            mean_distance_player_heights=float(aligned.click_distance_player_heights.mean()),
            note='Nearest sampled prediction, not inference rerun at annotation time. '
                 'Aligned sample frames only for role rates; localization proxies are not identity accuracy.'))
    return pd.DataFrame(summaries)


def collect_tracking_diagnostics(indices,mapping,fold,output_dir=None,fit_sources=None):
    """Audit only already-predicted public clips; returns quality/frame/click tables."""
    indices=list(map(int,indices))
    assert_tracking_audit_provenance(indices,mapping,fit_sources)
    quality=[]; frame_rows=[]; click_rows=[]
    for i in indices:
        row=rows.iloc[i]; clip=row['clip']
        annotation=clicks.loc[clip] if clip in clicks.index else None
        summary,frames,audits=tracking_clip_diagnostics(clip,row.source_id,fold,
            extract_raw(clip),load_view(mapping,i),annotation)
        quality.append(summary); frame_rows.extend(frames); click_rows.extend(audits)
    quality=pd.DataFrame(quality); frames=pd.DataFrame(frame_rows); audits=pd.DataFrame(click_rows)
    if output_dir is not None:
        output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
        quality.to_csv(output_dir/'tracking_quality.csv',index=False)
        frames.to_csv(output_dir/'tracking_frames.csv',index=False)
        audits.to_csv(output_dir/'role_click_audit.csv',index=False)
        summarize_role_click_audits(audits).to_csv(output_dir/'role_click_summary.csv',index=False)
        (output_dir/'tracking_limitations.txt').write_text(
            'All audits are post-prediction. Click-containing boxes are localization proxies, not identity truth. '
            'No identity-switch rate is measurable from the available isolated clicks. Raw jumps may reflect camera motion. '
            'Role predictions are taken from the nearest sampled classification frame; inference is not rerun at the annotated time. '
            'Exact click-nearest detector frames are used only for the separate detector-containment proxy. '
            'Timestamp-ineligible clicks are excluded from role rates. Missing roles use full-frame crops. '
            'Tracking-quality associations with prediction errors are not causal component effects.\n')
    return quality,frames,audits


def render_tracking_audits(error_df,mapping,output_dir,high_loss_count=3,correct_count=2,seed=42):
    """Save context/pose/crop panels for high-loss and reproducibly sampled correct clips."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
    errors=error_df.copy()
    loss_column='weighted_loss' if 'weighted_loss' in errors else 'weighted_log_loss'
    errors=errors.drop_duplicates('clip').sort_values('clip')
    high=errors.sort_values([loss_column,'clip'],ascending=[False,True]).head(high_loss_count)
    correct=errors[(errors.weighted_error==0)&~errors['clip'].isin(high['clip'])]
    if len(correct): correct=correct.sample(n=min(correct_count,len(correct)),random_state=seed)
    selected=pd.concat([high.assign(audit_selection='highest_loss'),correct.assign(audit_selection='sampled_correct')])
    selected[['clip','audit_selection']].assign(selection_seed=seed).to_csv(output_dir/'visual_audit_selection.csv',index=False)
    clip_indices={rows['clip'].iloc[int(i)]:int(i) for i in mapping}
    for _,entry in selected.iterrows():
        i=clip_indices[entry['clip']]; assert_tracking_audit_provenance([i],mapping)
        raw=extract_raw(entry['clip']); view=load_view(mapping,i,decode_video=True)
        times=np.asarray(raw['times']); anchor=int(view.get('anchor_frame',len(times)//2))
        time_ids=sorted(set([0,anchor,len(times)//2,len(times)-1]))
        fig,axes=plt.subplots(len(time_ids),3,figsize=(12,3.2*len(time_ids)),squeeze=False)
        for row_number,t in enumerate(time_ids):
            rgb=raw['frames'][t]; h,w=rgb.shape[:2]; ax=axes[row_number,0]; ax.imshow(rgb)
            detection=raw['detections'][t]
            for r,(role,color) in enumerate([('kicker','orange'),('keeper','cyan')]):
                j=int(view['track_ids'][t,r]); present=0<=j<len(detection['boxes'])
                if present:
                    box=np.asarray(detection['boxes'][j])*[w,h,w,h]
                    ax.add_patch(Rectangle(box[:2],box[2]-box[0],box[3]-box[1],fill=False,edgecolor=color,linewidth=2))
                    joints=np.asarray(detection['joints'][j])*[w,h]
                    visible=np.asarray(detection['confidence'][j])>=.25
                    ax.scatter(joints[visible,0],joints[visible,1],s=8,c=color)
                if entry['clip'] in clicks.index:
                    annotation=clicks.loc[entry['clip']]
                    offset=abs(float(times[t])-float(annotation[f'{role}_click_time_seconds']))
                    if offset<=.10:
                        ax.scatter(annotation[f'{role}_x_norm']*w,annotation[f'{role}_y_norm']*h,
                                   s=70,c=color,marker='x',linewidths=2)
                axes[row_number,r+1].imshow(view['video'][r+1,t].transpose(1,2,0))
                axes[row_number,r+1].set_title(f'{role}: '+('selected crop' if present else 'FULL-FRAME FALLBACK'))
            ax.set_title(f'Context + selected boxes/pose | t={times[t]:.3f}s')
            for a in axes[row_number]: a.axis('off')
        fig.suptitle(f"{entry['clip']} | {entry['audit_selection']} | loss={entry[loss_column]:.3f}\n"
                     'Orange kicker / cyan keeper; crosses: post-prediction public clicks within 0.10 s. Not identity ground truth.')
        fig.tight_layout(); fig.savefig(output_dir/(Path(entry['clip']).stem+'_audit.png'),dpi=130,bbox_inches='tight')
        plt.close(fig)
    return selected[['clip','audit_selection']]


def tracking_diagnostics_smoke(output_dir):
    """CPU-only synthetic audit checks; no videos, labels, detector, or model needed."""
    output_dir=Path(output_dir); output_dir.mkdir(parents=True,exist_ok=True)
    box=np.array([[.4,.3,.6,.7]],dtype=float)
    detection=dict(boxes=box,joints=np.full((1,17,2),.5),confidence=np.ones((1,17)))
    empty=dict(boxes=np.empty((0,4)),joints=np.empty((0,17,2)),confidence=np.empty((0,17)))
    raw=dict(times=np.array([0.,.5,1.]),frames=[np.zeros((100,200,3),dtype=np.uint8) for _ in range(3)],
             detections=[detection,empty,detection],camera_quality=np.array([0.,0.,1.]),
             clicked=[dict(time=0.,**detection),dict(time=.5,**empty)])
    view=dict(track_ids=np.array([[0,-1],[-1,-1],[0,-1]]),
              traces=np.array([[[.5,.5,.8],[0,0,0]],[[0,0,0],[0,0,0]],[[.5,.5,.7],[0,0,0]]]))
    annotation={'kicker_click_time_seconds':0.,'kicker_x_norm':.6,'kicker_y_norm':.5,
                'keeper_click_time_seconds':.5,'keeper_x_norm':.2,'keeper_y_norm':.4}
    quality,frames,audits=tracking_clip_diagnostics('synthetic.mp4','heldout_source',0,raw,view,annotation)
    assert quality['kicker_tracked_fraction']==2/3 and quality['keeper_fallback_fraction']==1.
    assert quality['kicker_longest_missing_run_frames']==1
    assert quality['keeper_longest_missing_run_frames']==3
    np.testing.assert_allclose(audits[0]['click_distance_player_heights'],.5)
    assert audits[0]['click_inside_selected_box']
    assert audits[1]['missing_role'] and audits[1]['unmatched_detection_proxy']
    assert frames[1]['fallback']=='full_frame'
    # A temporally remote click must not be described as an identity/role failure.
    off=dict(annotation,kicker_click_time_seconds=3.)
    _,_,mismatch=tracking_clip_diagnostics('synthetic.mp4','heldout_source',0,raw,view,off)
    assert mismatch[0]['failure_category']=='timestamp_mismatch_unresolved'
    # A selected wrong box can be distinguished from no click-containing detection.
    two=dict(detection,boxes=np.r_[box,[[.1,.2,.3,.6]]],
             joints=np.full((2,17,2),.5),confidence=np.ones((2,17)))
    raw_other=dict(raw,detections=[two,empty,detection])
    view_other=dict(view,track_ids=np.array([[1,-1],[-1,-1],[0,-1]]))
    _,_,wrong=tracking_clip_diagnostics('synthetic.mp4','heldout_source',0,raw_other,view_other,annotation)
    assert wrong[0]['failure_category']=='selected_other_box_proxy'
    pd.DataFrame([quality]).to_csv(output_dir/'tracking_quality.csv',index=False)
    pd.DataFrame(frames).to_csv(output_dir/'tracking_frames.csv',index=False)
    pd.DataFrame(audits).to_csv(output_dir/'role_click_audit.csv',index=False)
    return dict(status='PASS',checks=['height-normalized aspect-correct distances','missing role/full-frame fallback',
        'tracking coverage and gaps','timestamp mismatch exclusion','detector/selector failure proxies'],
        limitation='Synthetic execution check only; no performance or identity-switch estimate.')


"""Self-contained, retraining-based stream and numeric-feature ablations.

The notebook embeds this source. No companion file is needed in Colab.
"""
import copy
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd


ABLATION_CONFIGS = {
    'A_context': dict(streams=[0], pose=False, motion='none', metadata=False),
    'B_context_kicker': dict(streams=[0, 1], pose=False, motion='none', metadata=False),
    'C_context_keeper': dict(streams=[0, 2], pose=False, motion='none', metadata=False),
    'D_all_visual': dict(streams=[0, 1, 2], pose=False, motion='none', metadata=False),
    'E_plus_pose': dict(streams=[0, 1, 2], pose=True, motion='none', metadata=False),
    'F_raw_motion': dict(streams=[0, 1, 2], pose=True, motion='raw', metadata=False),
    'G_compensated_motion': dict(streams=[0, 1, 2], pose=True, motion='compensated', metadata=False),
    'H_plus_metadata': dict(streams=[0, 1, 2], pose=True, motion='compensated', metadata=True),
    # This exact feature set preserves the pre-diagnostics production representation.
    'production': dict(streams=[0, 1, 2], pose=True, motion='both', metadata=True,
                       legacy_full_summary=True),
}
ACTIVE_ABLATION_NAME = 'production'
ACTIVE_ABLATION = copy.deepcopy(ABLATION_CONFIGS[ACTIVE_ABLATION_NAME])

# Per-player make_views layout: 34 normalized joint coordinates, 17 confidences,
# center x/y, width/height, role confidence, 36 raw velocities, 36 compensated
# velocities, 18 validity flags. Two players (146 each), then seven camera/time fields.
ROLE_FEATURE_WIDTH = 146
FRAME_FEATURE_WIDTH = 299
STATIC_POSE = tuple(range(0, 56))
RAW_MOTION = tuple(range(56, 92))
COMPENSATED_MOTION = tuple(range(92, 128))
COMPENSATED_VALIDITY = tuple(range(128, 146))
CAMERA_FEATURES = tuple(range(292, 297))
TIME_FEATURES = (297, 298)
SUMMARY_OPERATIONS = ('mean', 'std', 'last', 'last_minus_first')
METADATA_NAMES = ('preferred_foot_left', 'preferred_foot_right', 'gender_F', 'gender_M')


def set_active_ablation(name):
    global ACTIVE_ABLATION_NAME, ACTIVE_ABLATION
    if name not in ABLATION_CONFIGS:
        raise ValueError(f'Unknown ablation: {name}')
    ACTIVE_ABLATION_NAME = name
    ACTIVE_ABLATION = copy.deepcopy(ABLATION_CONFIGS[name])
    return copy.deepcopy(ACTIVE_ABLATION)


def ablation_stream_ids(config=None):
    config = ACTIVE_ABLATION if config is None else config
    streams = tuple(config['streams'])
    if not streams or len(set(streams)) != len(streams) or any(s not in (0, 1, 2) for s in streams):
        raise ValueError('Streams must be a nonempty unique subset of context/kicker/keeper [0,1,2].')
    return streams


def _ablation_numeric_blocks(config):
    """Return (columns, reductions, group); exclusions happen before every reduction."""
    if config.get('legacy_full_summary', False):
        return [(tuple(range(FRAME_FEATURE_WIDTH)), SUMMARY_OPERATIONS, 'production_all')]
    blocks = []
    if config['pose']:
        # No pose delta/std/temporal pooling in the pose-only condition: these can
        # themselves encode movement. Pose and geometry are measured at the final frame.
        blocks.append((tuple(r * ROLE_FEATURE_WIDTH + j for r in range(2) for j in STATIC_POSE),
                       ('last',), 'static_pose_geometry_confidence'))
    if config['motion'] in ('raw', 'both'):
        blocks.append((tuple(r * ROLE_FEATURE_WIDTH + j for r in range(2) for j in RAW_MOTION),
                       SUMMARY_OPERATIONS, 'raw_motion'))
    if config['motion'] in ('compensated', 'both'):
        blocks.append((tuple(r * ROLE_FEATURE_WIDTH + j for r in range(2) for j in COMPENSATED_MOTION),
                       SUMMARY_OPERATIONS, 'compensated_motion'))
        blocks.append((tuple(r * ROLE_FEATURE_WIDTH + j for r in range(2) for j in COMPENSATED_VALIDITY),
                       ('mean', 'last'), 'compensated_validity'))
    if config['motion'] != 'none':
        # Keep exactly the same time features in raw/compensated comparisons.
        blocks.append((TIME_FEATURES, ('mean', 'last'), 'sampling_time'))
    return blocks


def numeric_feature_manifest(config=None):
    config = ACTIVE_ABLATION if config is None else config
    result = []
    for columns, reductions, group in _ablation_numeric_blocks(config):
        for operation in reductions:
            result.extend(dict(group=group, raw_column=int(i), reduction=operation,
                               name=f'{group}__column_{i}__{operation}') for i in columns)
    if config['metadata']:
        result.extend(dict(group='metadata', raw_column=None, reduction='none', name=name)
                      for name in METADATA_NAMES)
    return result


def ablation_numeric_features(value, metadata_row, config=None):
    config = ACTIVE_ABLATION if config is None else config
    ablation_stream_ids(config)
    x = np.asarray(value['numeric'], dtype=np.float32)
    if x.ndim != 2 or x.shape[1] != FRAME_FEATURE_WIDTH or len(x) == 0:
        raise ValueError(f'Expected nonempty T x {FRAME_FEATURE_WIDTH} numeric cache, got {x.shape}')
    features = []
    for columns, reductions, _ in _ablation_numeric_blocks(config):
        selected = x[:, columns]
        operations = dict(mean=lambda: selected.mean(0), std=lambda: selected.std(0),
                          last=lambda: selected[-1], last_minus_first=lambda: selected[-1]-selected[0])
        features.extend(operations[name]() for name in reductions)
    if config['metadata']:
        meta = np.asarray(metadata_row, dtype=np.float32)
        if meta.shape != (4,):
            raise ValueError('Expected the four existing preferred-foot/gender metadata indicators')
        features.append(meta)
    result = np.concatenate(features).astype(np.float32) if features else np.empty(0, dtype=np.float32)
    if len(result) != len(numeric_feature_manifest(config)) or not np.isfinite(result).all():
        raise ValueError('Non-finite or mismatched ablation features')
    return result


class EmptyNumericScaler:
    """A visual-only model has no numeric branch and no learned preprocessing."""
    n_features_in_ = 0

    def transform(self, values):
        values = np.asarray(values)
        if values.ndim != 2 or values.shape[1] != 0:
            raise ValueError('Visual-only scaler requires an empty numeric feature matrix')
        return values.astype(np.float32)


def fit_feature_scaler(training_features):
    values = np.asarray(training_features)
    if values.ndim != 2 or not len(values):
        raise ValueError('Scaler input must be a nonempty training matrix')
    if values.shape[1] == 0:
        return EmptyNumericScaler()
    from sklearn.preprocessing import StandardScaler
    return StandardScaler().fit(values)


def make_ablation_model_class():
    """Call after the notebook imports torch/transformers; use as MultiViewVideo.

    Keeping this a factory lets feature smoke checks run without importing torch
    or downloading any weights. Notebook globals are read at runtime: BACKBONE_SPEC,
    TRAINING_POLICY, N_FRAMES, SIZE, CLASSES. The encoder comes from BackboneAdapter.
    """
    import torch

    class ConfigurableMultiViewVideo(torch.nn.Module):
        def __init__(self, numeric_dim, architecture=None):
            """architecture=None loads pretrained encoder weights; a saved architecture dict rebuilds the
            same modules without a pretrained download (inference then loads the run's own weights)."""
            super().__init__()
            self.stream_ids = ablation_stream_ids()
            self.has_numeric = numeric_dim > 0
            self.feature_signature = ablation_configuration_manifest()['feature_signature']
            if numeric_dim != len(numeric_feature_manifest()):
                raise ValueError('Numeric feature dimension differs from active ablation manifest')
            spec = globals().get('BACKBONE_SPEC') or backbone_spec('videomae')
            self.training_policy = globals().get('TRAINING_POLICY', 'finetune')
            if (spec['frames'], spec['size']) != (N_FRAMES, SIZE):
                raise ValueError(f'Backbone expects {spec["frames"]} frames at {spec["size"]}px; crops use {N_FRAMES} at {SIZE}px')
            self.adapter = BackboneAdapter(spec, self.training_policy, repo_dir=globals().get('VJEPA2_REPO_DIR'))
            self.backbone, self.norm, self.architecture = self.adapter.build(architecture)
            hidden = self.adapter.feature_dim
            self.numeric = (torch.nn.Sequential(torch.nn.Linear(numeric_dim, 128),
                torch.nn.GELU(), torch.nn.Dropout(.3)) if self.has_numeric else torch.nn.Identity())
            fused_dim = len(self.stream_ids) * hidden + (128 if self.has_numeric else 0)
            self.heads = torch.nn.ModuleList([torch.nn.Linear(fused_dim, n) for n in map(len, CLASSES)])
            self.dropout = torch.nn.Dropout(.3)
            apply_training_policy(self, self.training_policy)  # sets requires_grad explicitly

        def train(self, mode=True):
            super().train(mode)
            if self.training_policy == 'frozen_probe':
                self.backbone.eval()  # frozen encoder: no dropout/drop-path, no gradient
            return self

        def encode_stream(self, video):
            """[B, T, C, H, W] normalized -> [B, hidden] via the backbone adapter."""
            if self.training_policy == 'frozen_probe':
                with torch.no_grad():
                    pooled = self.adapter.pool_tokens(self.backbone, video)
                return self.norm(pooled)
            return self.adapter.encode_video(self.backbone, self.norm, video)

        def forward(self, video, numeric):
            # Unselected streams never enter the encoder and have no head columns.
            streams = [self.encode_stream(video[:, view]) for view in self.stream_ids]
            if self.has_numeric:
                streams.append(self.numeric(numeric))
            elif numeric.shape[-1] != 0:
                raise ValueError('Visual-only ablation unexpectedly received numeric features')
            feature = torch.cat(streams, dim=1)
            return [head(self.dropout(feature)) for head in self.heads]

    return ConfigurableMultiViewVideo


def ablation_configuration_manifest(name=None):
    name = ACTIVE_ABLATION_NAME if name is None else name
    config = copy.deepcopy(ABLATION_CONFIGS[name])
    features = numeric_feature_manifest(config)
    manifest = dict(name=name, configuration=config, streams=list(ablation_stream_ids(config)),
                    numeric_dimension=len(features), numeric_features=features,
                    architecture_note='Shared pretrained video backbone; concatenate only selected streams. '
                    'Classification heads are resized for stream count. Numeric MLP is omitted for visual-only variants. '
                    'Fewer streams reduce encoder calls and head parameters; these compute differences are intentional.',
                    scope='Retrained ablation, not inference masking. Role detection/tracking and crop policy stay fixed. '
                    'Pose-only means final-frame geometry/confidence; motion groups have explicit temporal summaries. '
                    'Image streams inherently retain visible pose, motion, and camera information.')
    manifest['feature_signature'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()[:16]
    return manifest


def estimate_ablation_fits(names, outer_folds, seeds, inner_splits=1):
    """One inner selection fit + one outer refit per seed and fold by default."""
    if outer_folds < 2 or inner_splits < 1 or not seeds:
        raise ValueError('Provide outer fold count, at least one inner split, and seeds')
    rows = []
    for name in names:
        manifest = ablation_configuration_manifest(name)
        fits = int(outer_folds * len(seeds) * (inner_splits + 1))
        rows.append(dict(variant=name, video_model_fits=fits, outer_folds=outer_folds,
                         seeds=len(seeds), visual_streams=len(manifest['streams']),
                         numeric_features=manifest['numeric_dimension'],
                         encoder_stream_fits=fits * len(manifest['streams'])))
    return pd.DataFrame(rows)


def run_controlled_ablations(run_experiment, names, outer_folds, seeds, output_dir, enabled=False):
    """Callback must use the same persisted outer/inner assignments for every variant.

    run_experiment(name) is the notebook's full nested grouped experiment, including
    inner checkpoint/ensemble selection and outer assessment. No prediction masking.
    """
    names = list(names)
    if len(set(names)) != len(names):
        raise ValueError('Duplicate ablation names')
    planned = estimate_ablation_fits(names, outer_folds, seeds)
    print(planned.to_string(index=False))
    print('Total planned video model fits:', int(planned.video_model_fits.sum()))
    print('Separate role-selector and tabular fits are additional; decoded detections may be reused.')
    if not enabled:
        print('Ablations disabled. Set RUN_ABLATIONS=True only after inspecting the fit budget.')
        return {}
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    planned.to_csv(output_dir/'ablation_fit_budget.csv', index=False)
    previous = ACTIVE_ABLATION_NAME
    results = {}
    try:
        for name in names:
            set_active_ablation(name)
            (output_dir/f'{name}_feature_manifest.json').write_text(
                json.dumps(ablation_configuration_manifest(), indent=2), encoding='utf-8')
            results[name] = run_experiment(name)
    finally:
        set_active_ablation(previous)
    return results


ABLATION_REFERENCES = {
    'B_context_kicker': 'A_context', 'C_context_keeper': 'A_context',
    'D_all_visual': 'A_context', 'E_plus_pose': 'D_all_visual',
    'F_raw_motion': 'E_plus_pose', 'G_compensated_motion': 'F_raw_motion',
    'H_plus_metadata': 'G_compensated_motion', 'production': 'H_plus_metadata',
}


def compare_ablation_metrics(results, output_dir, component='ensemble'):
    """Compare matched folds and pooled rows using ComponentOOF export results."""
    records = []
    for variant, reference in ABLATION_REFERENCES.items():
        if variant not in results or reference not in results:
            continue
        variant_table = results[variant]['metrics']
        reference_table = results[reference]['metrics']
        variant_table = variant_table[variant_table.component == component].copy()
        reference_table = reference_table[reference_table.component == component].copy()
        for table in (variant_table, reference_table): table['fold'] = table['fold'].astype(str)
        if variant_table.empty or reference_table.empty:
            raise ValueError(f'No {component} metrics for ablation comparison')
        if set(variant_table.fold) != set(reference_table.fold):
            raise ValueError('Cannot compare experiments with different evaluated folds')
        merged = variant_table.merge(reference_table, on='fold', suffixes=('_variant', '_reference'),
                                     validate='one_to_one')
        if not (merged.clips_variant == merged.clips_reference).all():
            raise ValueError('Cannot compare experiments with different evaluated sample counts')
        for row in merged.to_dict('records'):
            record = dict(variant=variant, reference=reference, component=component,
                          fold=row['fold'], clips=int(row['clips_variant']))
            for metric in ['weighted_accuracy', 'weighted_cross_entropy',
                           'kick_direction_accuracy', 'goalkeeper_direction_accuracy']:
                record[metric + '_reference'] = row[metric + '_reference']
                record[metric + '_variant'] = row[metric + '_variant']
                record[metric + '_delta'] = row[metric + '_variant'] - row[metric + '_reference']
            records.append(record)
    table = pd.DataFrame(records)
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    table.to_csv(output_dir / 'ablation_comparisons.csv', index=False)
    return table


def ablation_feature_smoke_test():
    """CPU checks: excluded columns cannot affect features, including derived summaries."""
    rng = np.random.default_rng(19)
    cache = dict(numeric=rng.normal(size=(5, FRAME_FEATURE_WIDTH)).astype(np.float32))
    meta = np.array([1, 0, 0, 1], dtype=np.float32)
    for name, config in ABLATION_CONFIGS.items():
        baseline = ablation_numeric_features(cache, meta, config)
        kept = {entry['raw_column'] for entry in numeric_feature_manifest(config)
                if entry['raw_column'] is not None}
        excluded = sorted(set(range(FRAME_FEATURE_WIDTH)) - kept)
        changed = cache['numeric'].copy()
        changed[:, excluded] = 10000 + rng.normal(size=(len(changed), len(excluded)))
        changed_meta = meta if config['metadata'] else 1-meta
        np.testing.assert_array_equal(baseline, ablation_numeric_features(dict(numeric=changed), changed_meta, config))
        assert len(baseline) == ablation_configuration_manifest(name)['numeric_dimension']
    assert len(ablation_numeric_features(cache, meta, ABLATION_CONFIGS['A_context'])) == 0
    assert ablation_stream_ids(ABLATION_CONFIGS['B_context_kicker']) == (0, 1)
    assert not set(RAW_MOTION) & {e['raw_column'] for e in numeric_feature_manifest(ABLATION_CONFIGS['G_compensated_motion'])}
    production = ablation_numeric_features(cache, meta, ABLATION_CONFIGS['production'])
    x = cache['numeric']
    np.testing.assert_allclose(production, np.r_[x.mean(0), x.std(0), x[-1], x[-1]-x[0], meta])
    # Empty features remain genuinely absent; do not insert a learnable constant branch.
    assert fit_feature_scaler(np.empty((3, 0))).transform(np.empty((2, 0))).shape == (2, 0)
    return dict(status='PASS', variants_checked=len(ABLATION_CONFIGS),
                checks=['stream selection', 'excluded raw/derived features', 'metadata exclusion',
                        'visual-only empty branch', 'production feature compatibility'],
                limitation='Feature smoke only; video retraining and GPU forward/backward require Colab.')


def ablation_architecture_smoke_test():
    """Train tiny synthetic CPU variants through the actual configurable class and backbone adapter.

    Does not download VideoMAE or estimate video prediction performance. Covers both training policies.
    """
    import torch
    from types import SimpleNamespace

    class TinyBackbone(torch.nn.Module):
        def __init__(self):
            super().__init__(); self.projection = torch.nn.Linear(3, 4)

        def gradient_checkpointing_enable(self): pass

        def forward(self, pixel_values):
            pooled = pixel_values.mean(dim=(1, 3, 4))
            return SimpleNamespace(last_hidden_state=self.projection(pooled)[:, None, :])

    class TinyConfig(SimpleNamespace):
        def to_dict(self): return dict(vars(self))

    class TinyPretrained:
        def __init__(self, config=None):
            self.videomae = TinyBackbone(); self.fc_norm = torch.nn.LayerNorm(4)
            self.config = TinyConfig(hidden_size=4, num_frames=2, image_size=4)

        @classmethod
        def from_pretrained(cls, name, **kwargs):
            return cls()

    tiny_spec = dict(backbone_spec('videomae'), checkpoint='synthetic-cpu-smoke', frames=2, size=4, feature_dim=4)
    overrides = dict(VideoMAEForVideoClassification=TinyPretrained, VIDEO_ID='synthetic-cpu-smoke',
                     BACKBONE_SPEC=tiny_spec, TRAINING_POLICY='finetune',
                     N_FRAMES=2, SIZE=4, CLASSES=[['left', 'middle', 'right'],
                                                ['left', 'stands in the middle', 'right']])
    previous = {key: globals().get(key) for key in overrides}
    previous_name = ACTIVE_ABLATION_NAME
    results = []
    try:
        globals().update(overrides)
        with torch.random.fork_rng(devices=[]):
            # CPU generator only: do not alter a Colab CUDA RNG stream.
            torch.random.default_generator.manual_seed(42)
            for name, policy in [('A_context', 'finetune'), ('B_context_kicker', 'finetune'), ('B_context_kicker', 'frozen_probe')]:
                globals()['TRAINING_POLICY'] = policy
                set_active_ablation(name)
                model = make_ablation_model_class()(0)
                video = torch.randn(2, 3, 2, 3, 4, 4); numeric = torch.empty(2, 0)
                model.eval(); baseline = model(video, numeric)
                changed = video.clone()
                excluded = sorted(set(range(3)) - set(ablation_stream_ids()))
                changed[:, excluded] += 10000
                for p, q in zip(baseline, model(changed, numeric)):
                    torch.testing.assert_close(p, q, rtol=0, atol=0)
                assert not list(model.numeric.parameters())
                assert model.heads[0].in_features == len(ablation_stream_ids()) * 4
                optimizer = torch.optim.AdamW(apply_training_policy(model, policy), lr=.01)
                before = model.backbone.projection.weight.detach().clone()
                head_before = model.heads[0].weight.detach().clone()
                model.train(); logits = model(video, numeric); y = torch.tensor([0, 2])
                loss = .7 * torch.nn.functional.cross_entropy(logits[0], y) + .3 * torch.nn.functional.cross_entropy(logits[1], y)
                loss.backward(); optimizer.step()
                assert not torch.equal(head_before, model.heads[0].weight)
                if policy == 'finetune':
                    assert not torch.equal(before, model.backbone.projection.weight)
                else:
                    assert torch.equal(before, model.backbone.projection.weight) and not model.backbone.training
                results.append(dict(variant=name, policy=policy, status='PASS', numeric_dimension=0,
                                    streams=list(ablation_stream_ids())))
    finally:
        set_active_ablation(previous_name)
        for key, value in previous.items():
            if value is None: globals().pop(key, None)
            else: globals()[key] = value
    return dict(status='PASS', variants=results,
                limitation='Tiny synthetic CPU architectures only; no pretrained-video performance estimate.')


# Cheap synthetic checks: no videos, labels from private clips, pretrained weights, or GPU needed.
RUN_DIAGNOSTICS_SMOKE = RUN_MODE in {'smoke', 'train', 'resume'}  # pure definitions in inference/reports modes
DIAGNOSTICS_SMOKE_OUTPUT = Path(SCRATCH_ROOT) / 'diagnostics_smoke'
if RUN_DIAGNOSTICS_SMOKE:
    smoke_output=DIAGNOSTICS_SMOKE_OUTPUT
    smoke_component_reports(smoke_output/'components')
    tracking_diagnostics_smoke(smoke_output/'tracking')
    print(ablation_feature_smoke_test())
    print(ablation_architecture_smoke_test())
    print('DIAGNOSTICS CPU SMOKE: PASS. Synthetic scores are not performance estimates.',smoke_output)


## Configuration and smoke mode
`RUN_MODE='smoke'` (the default in both notebooks) is a cheap real-model check: 16 training and four held-out public clips from disjoint sources, role selectors, four video minibatches with the real backbone checkpoint, parameter-update checks for the selected training policy (encoder updates for `finetune`; unchanged encoder plus head updates for `frozen_probe`), checkpoint reload and ensembling. With `RUN_TIMING_BENCHMARK=True` it also measures warmed throughput and peak memory (`timing_estimate.json`). It does not enter cross-validation or write a submission. Run it on the hardware intended for the long run before scheduling all folds.


In [ ]:
# CELL: experiment_configuration
from ultralytics import YOLO
from transformers import VideoMAEForVideoClassification, VideoMAEImageProcessor
from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from PIL import Image, ImageEnhance
import cv2, copy, itertools, gc, pickle, subprocess, time
from collections import deque

# User-facing switches are in Settings. These are fixed protocol constants (part of the experiment identity).
TRAINING_MODES = {'smoke', 'train', 'resume'}
VALIDATION_FRACTION = .20
TEST_FRACTION = .05
TIMING_WARMUP_BATCHES = 8
TIMING_MEASURED_BATCHES = 24  # Three full accumulation groups at the default settings.
TIMING_INFERENCE_CLIPS = 8
POSE_WEIGHTS = 'yolo11m-pose.pt'
DETECTOR_CONF = .08  # Retain small/distant keepers; learned roles and tracking filter candidates.
DETECTOR_IMGSZ = 1280
WINDOW = 1.6
VIDEO_EPOCHS = 6  # Maximum epochs; inner validation selects outer epochs.
VIDEO_SEEDS = [42, 137]
OUTER_FOLDS = 5
ROLE_FOLDS = 3
GRAD_ACCUM = 8
ROLE_MIN_PROB = .35
TRACK_MAX_GAP_SECONDS = .5
TRACK_MAX_JUMP_HEIGHTS = 1.25
TRACK_SPEED_HEIGHTS_PER_SECOND = 6.0
AUDIT_HIGH_LOSS = 3
AUDIT_CORRECT = 2
ABLATION_VARIANTS = ['A_context','B_context_kicker','C_context_keeper','D_all_visual',
                     'E_plus_pose','F_raw_motion','G_compensated_motion','H_plus_metadata']
HEATMAP_TOP_K = 4
HEATMAP_GRID = 5
HEATMAP_FILL = 'blur'  # Compare with 'mean' to check masking sensitivity.
HEATMAP_SEEDS = None  # None explains the full video-seed ensemble.
SMOKE_MAX_BATCHES = 4
CAMERA_CACHE_VERSION = 'similarity-jpeg-v1'

# Backbone: frame count and crop resolution come from the adapter, not VideoMAE defaults.
BACKBONE_SPEC = backbone_spec(BACKBONE, BACKBONE_REVISION)
LEGACY_IDENTITY = backbone_identity(backbone_spec('videomae'), 'finetune')
SAVED_INFERENCE_MANIFEST = None
OUTPUT = Path(OUTPUT_ROOT)   # historical name used by split/cache helpers
if RUN_MODE == 'inference':
    # Inference reconstructs the SAVED model identity; current notebook defaults never redefine it.
    if not TRAINED_RUN_DIR.is_dir(): raise FileNotFoundError(f'TRAINED_RUN_DIR not found: {TRAINED_RUN_DIR}')
    if is_bundle(TRAINED_RUN_DIR):
        verify_bundle(TRAINED_RUN_DIR, VERIFY_CHECKSUMS); print('Bundle verified:', TRAINED_RUN_DIR)
    _manifest_path = TRAINED_RUN_DIR/'inference_manifest.json'
    if _manifest_path.exists():
        SAVED_INFERENCE_MANIFEST = json.loads(_manifest_path.read_text())
        _saved_cfg = SAVED_INFERENCE_MANIFEST['config']
        BACKBONE_SPEC = SAVED_INFERENCE_MANIFEST['backbone']['spec']
        TRAINING_POLICY = SAVED_INFERENCE_MANIFEST['backbone']['training_policy']
    else:
        _source = TRAINED_RUN_DIR/'experiment_manifest.json'
        if not _source.exists():
            raise FileNotFoundError(f'{_manifest_path} not found and {_source} missing; inference needs a finished '
                                    'nested_cv run (or export a holdout run while it is the active training RUN).')
        _saved_cfg = json.loads(_source.read_text())['config']
        if 'backbone' in _saved_cfg:
            BACKBONE_SPEC = backbone_spec(_saved_cfg['backbone']['name'], _saved_cfg['backbone']['revision'])
            TRAINING_POLICY = _saved_cfg['training_policy']
        else:
            BACKBONE_SPEC = dict(backbone_spec('videomae'), checkpoint=_saved_cfg['video']); TRAINING_POLICY = 'finetune'
    N_FRAMES, WINDOW, SIZE = int(_saved_cfg['frames']), float(_saved_cfg['window']), int(_saved_cfg['size'])
    DETECTOR_CONF, DETECTOR_IMGSZ, POSE_WEIGHTS = _saved_cfg['detector_conf'], _saved_cfg['detector_imgsz'], _saved_cfg['pose']
    TRACK_MAX_GAP_SECONDS = _saved_cfg['tracking']['max_gap']; TRACK_MAX_JUMP_HEIGHTS = _saved_cfg['tracking']['max_jump']
    TRACK_SPEED_HEIGHTS_PER_SECOND = _saved_cfg['tracking']['speed']; ROLE_MIN_PROB = _saved_cfg['role_min_prob']
    VIDEO_SEEDS = list(_saved_cfg['seeds'])
    if BACKBONE != BACKBONE_SPEC['name']:
        print(f'NOTE: Settings say BACKBONE={BACKBONE!r}, but the saved run uses {BACKBONE_SPEC["name"]!r}. '
              'The saved run wins; inference never converts a run to another backbone.')
    print('Loaded saved preprocessing/model settings from', TRAINED_RUN_DIR)
VIDEO_ID = BACKBONE_SPEC['checkpoint']
if RUN_MODE != 'inference':
    N_FRAMES = BACKBONE_SPEC['frames']; SIZE = BACKBONE_SPEC['size']
BACKBONE_IDENTITY = backbone_identity(BACKBONE_SPEC, TRAINING_POLICY)
# The original VideoMAE fine-tuning configuration keeps its exact CFG, so old runs still resume.
LEGACY_DEFAULT_BACKBONE = BACKBONE_IDENTITY == LEGACY_IDENTITY
OPTIMIZER_SETTINGS = dict(name='AdamW', weight_decay=.05, policy=TRAINING_POLICY,
                          learning_rates=dict(finetune=dict(backbone=1e-5, heads=1e-4), frozen_probe=dict(heads=1e-3))[TRAINING_POLICY],
                          gradient_clip=1.0, amp='float16')
if EVALUATION_MODE == 'development' and RUN_MODE != 'inference':
    VIDEO_SEEDS = list(DEV_SEEDS); VIDEO_EPOCHS = int(DEV_EPOCHS)
FULL_TRAIN_SETTINGS=dict(epochs=VIDEO_EPOCHS,seeds=list(VIDEO_SEEDS),accumulation=GRAD_ACCUM)
if QUICK_SMOKE:
    VIDEO_EPOCHS = 1
    VIDEO_SEEDS = [42]
    ROLE_FOLDS = 2
    GRAD_ACCUM = 2
if RUN_MODE == 'inference' and DEVICE != 'cuda':
    print('WARNING: no GPU. Inference will work but uncached detection/video encoding will be slow.')

CFG = dict(dataset=DATA_ID,video=VIDEO_ID,pose=POSE_WEIGHTS,frames=N_FRAMES,
           detector_conf=DETECTOR_CONF,detector_imgsz=DETECTOR_IMGSZ,
           window=WINDOW,size=SIZE,epochs=VIDEO_EPOCHS,seeds=VIDEO_SEEDS,
           outer_folds=OUTER_FOLDS,role_folds=ROLE_FOLDS,role_min_prob=ROLE_MIN_PROB,
           version='nested-components-v6',gradient_accumulation=GRAD_ACCUM,evaluation_mode=EVALUATION_MODE,
           validation_fraction=VALIDATION_FRACTION,test_fraction=TEST_FRACTION,split_seed=SEED,targets=TARGETS,metric_weights=WEIGHTS.tolist(),
           tracking=dict(max_gap=TRACK_MAX_GAP_SECONDS,max_jump=TRACK_MAX_JUMP_HEIGHTS,speed=TRACK_SPEED_HEIGHTS_PER_SECOND),
           smoke=QUICK_SMOKE,smoke_max_batches=SMOKE_MAX_BATCHES)
if not LEGACY_DEFAULT_BACKBONE:
    # New backbones/policies get distinct runs, crop caches (size/frames) and checkpoints.
    CFG.update(backbone=BACKBONE_IDENTITY, training_policy=TRAINING_POLICY, optimizer=OPTIMIZER_SETTINGS)
if EVALUATION_MODE == 'development':
    CFG.update(development=dict(train_source_fraction=DEV_TRAIN_SOURCE_FRACTION,
        validation_source_fraction=DEV_VALIDATION_SOURCE_FRACTION, seeds=list(DEV_SEEDS), epochs=int(DEV_EPOCHS),
        scope='development inner-validation subset; not complete OOF performance'))
if RUN_MODE == 'inference':
    CFG = _saved_cfg; RUN = TRAINED_RUN_DIR
    EXPERIMENT_ID = stable_hash(CFG)
    if DATA_ID is not None and _saved_cfg.get('dataset') != DATA_ID:
        print('NOTE: dataset ZIP differs from the training ZIP; training-time crop/detector caches will not be reused.')
else:
    # Identity = computation only. Platform/paths/attempts choose WHERE it is written, never WHAT it is.
    EXPERIMENT_ID = hashlib.sha256(json.dumps(CFG,sort_keys=True).encode()).hexdigest()[:16]
    RUN = OUTPUT/'runs'/PLATFORM/f'{EXPERIMENT_NAME}__learned_{EXPERIMENT_ID}'
    _legacy = OUTPUT/f'learned_{EXPERIMENT_ID}'          # layout before platform-separated runs (Colab)
    if PLATFORM == 'colab' and _legacy.is_dir() and not RUN.exists():
        RUN = _legacy; print('Using the existing pre-update run folder', RUN)
    if RUN_MODE == 'resume':
        if TRAINED_RUN_DIR is not None and TRAINED_RUN_DIR.resolve() != RUN.resolve():
            saved = TRAINED_RUN_DIR/'experiment_manifest.json'
            hint = ''
            if saved.exists():
                old = json.loads(saved.read_text())['config']
                hint = ' Differing settings: ' + ', '.join(sorted(k for k in set(old)|set(CFG)
                    if k in CFG and old.get(k) != CFG.get(k)))
            raise ValueError(f'Current settings map to {RUN.name}, not {TRAINED_RUN_DIR.name}.{hint} '
                             'Restore the original settings to resume (backbone/policy changes are new runs).')
        if not (RUN/'stages').is_dir():
            raise ValueError(f'RUN_MODE="resume" but {RUN} has no saved stages. Use RUN_MODE="train" to start it.')
    RUN.mkdir(parents=True, exist_ok=True)
    write_attempt_record(RUN, dict(execution=EXECUTION_SETTINGS, experiment=EXPERIMENT_SETTINGS,
                                   experiment_identity=EXPERIMENT_ID, config=CFG), ENVIRONMENT_INFO)
print('RUN:', RUN, '| experiment identity:', EXPERIMENT_ID, '| attempt:', ATTEMPT_ID)
rows=pd.concat([train,test],ignore_index=True)
if NEEDS_DATASET:
    # Shared detector cache excludes learned role/model parameters and target labels.
    # Click timestamps affect only separate annotation-nearest detector frames, never sampled model inputs.
    _click_table=pd.read_csv(find_one({'public_position_clicks.csv'}))
    _time_columns=['clip','kicker_click_time_seconds','keeper_click_time_seconds']
    _click_times_hash=hashlib.sha256(_click_table[_time_columns].sort_values('clip').to_csv(index=False).encode()).hexdigest()
    clicks=_click_table.set_index('clip')
    assert clicks.index.is_unique
else:
    _click_times_hash=None
    clicks=pd.DataFrame(columns=['kicker_click_time_seconds','keeper_click_time_seconds','kicker_x_norm','kicker_y_norm',
                                 'keeper_x_norm','keeper_y_norm']).rename_axis('clip')
RAW_CFG=dict(dataset=DATA_ID,pose=POSE_WEIGHTS,conf=DETECTOR_CONF,imgsz=DETECTOR_IMGSZ,
    frames=N_FRAMES,window=WINDOW,click_timestamps=_click_times_hash,camera_cache_version=CAMERA_CACHE_VERSION)
RAW=OUTPUT/('detector_cache_'+hashlib.sha256(json.dumps(RAW_CFG,sort_keys=True).encode()).hexdigest()[:16])
if NEEDS_DATASET:
    RAW.mkdir(parents=True, exist_ok=True)
    atomic_write_text(RAW/'manifest.json',json.dumps(RAW_CFG,indent=2))
STAGES=RUN/'stages'
if RUN_MODE != 'inference': STAGES.mkdir(exist_ok=True)

# Choose annotated examples by source only, without selecting on target outcomes.
# Four training sources x four clips; two held-out sources x two clips.
if QUICK_SMOKE:
    eligible=train[train['clip'].isin(clicks.index)]
    sources=eligible.groupby('source_id').size()
    sources=sources[sources>=4].index.to_numpy()
    if len(sources)<6: raise ValueError('Smoke test needs six sources with at least four clicked clips each.')
    selected=np.random.default_rng(SEED).permutation(sources)[:6]
    smoke_train=np.concatenate([eligible[eligible.source_id==g].index.to_numpy()[:4] for g in selected[:4]])
    smoke_hold=np.concatenate([eligible[eligible.source_id==g].index.to_numpy()[:2] for g in selected[4:]])
    assert not set(train.source_id.iloc[smoke_train]) & set(train.source_id.iloc[smoke_hold])
    print('Smoke subset:',len(smoke_train),'training clips,',len(smoke_hold),'held-out clips; 6 disjoint sources')

def make_dataset_splits(frame, validation_fraction, test_fraction, seed):
    # Split whole sources; never choose a partition by validation/test outcomes.
    if not 0 < test_fraction < validation_fraction < 1-test_fraction:
        raise ValueError('Require 0 < test fraction < validation fraction and a positive training fraction.')
    if frame.source_id.isna().any(): raise ValueError('Missing source_id prevents safe splitting.')
    ids=np.arange(len(frame))
    remaining,held=next(GroupShuffleSplit(n_splits=1,test_size=test_fraction,random_state=seed).split(
        ids,groups=frame.source_id))
    a,b=next(GroupShuffleSplit(n_splits=1,test_size=validation_fraction/(1-test_fraction),
        random_state=seed+1).split(remaining,groups=frame.source_id.iloc[remaining]))
    parts={'training':remaining[a], 'validation':remaining[b], 'test':held}
    assert len(np.unique(np.concatenate(list(parts.values()))))==len(frame)
    for left,right in itertools.combinations(parts.values(),2):
        assert not set(frame.source_id.iloc[left]) & set(frame.source_id.iloc[right])
    if not len(parts['test']) < len(parts['validation']) < len(parts['training']):
        raise ValueError('Source sizes prevent training > validation > test. Adjust fractions before training.')
    return parts

if EVALUATION_MODE=='holdout' and RUN_MODE in {'train', 'resume'}:
    split_indices=make_dataset_splits(train,VALIDATION_FRACTION,TEST_FRACTION,SEED)
    for j,classes in enumerate(CLASSES):
        if len(np.unique(Y[split_indices['training'],j]))!=len(classes):
            raise ValueError(f'Training split is missing a class for {TARGETS[j]}; adjust split settings.')
    split_manifest=train[['clip','source_id']].copy()
    for name,ids in split_indices.items(): split_manifest.loc[ids,'split']=name
    split_manifest.to_csv(RUN/'dataset_splits.csv',index=False)
    split_summary=pd.DataFrame([dict(split=name,clips=len(ids),sources=train.source_id.iloc[ids].nunique(),
        fraction=len(ids)/len(train)) for name,ids in split_indices.items()])
    display(split_summary)
    print('Fractions apply to source groups, so actual clip percentages may differ.')
    print('The labeled test split is separate from the 900 unlabeled competition clips.')

# Normalization constants: VideoMAE reads its checkpoint processor; V-JEPA uses the official ImageNet values.
if RUN_MODE == 'inference' and SAVED_INFERENCE_MANIFEST is not None:
    IMAGE_MEAN=np.array(SAVED_INFERENCE_MANIFEST['preprocessing']['image_mean'],dtype=np.float32)
    IMAGE_STD=np.array(SAVED_INFERENCE_MANIFEST['preprocessing']['image_std'],dtype=np.float32)
elif BACKBONE_SPEC['family'] == 'videomae':
    processor=VideoMAEImageProcessor.from_pretrained(VIDEO_ID)  # small preprocessing config, not weights
    IMAGE_MEAN=np.array(processor.image_mean,dtype=np.float32)
    IMAGE_STD=np.array(processor.image_std,dtype=np.float32)
else:
    IMAGE_MEAN=np.array(IMAGENET_MEAN,dtype=np.float32); IMAGE_STD=np.array(IMAGENET_STD,dtype=np.float32)
print(f'Backbone {BACKBONE_SPEC["name"]}: {N_FRAMES} frames, {SIZE}px crops, policy={TRAINING_POLICY}')
# The pose detector loads lazily (get_pose_model) only when an uncached clip must be decoded.


Label-free extraction and camera model
Cache detector candidates, poses, timestamps, sampled RGB frames, and background transforms.
Click timestamps select an additional exact annotation-nearest frame for role training only.
They do not alter the classification frame sampling. All videos use the same sampling rule.
A similarity transform maps previous-frame pixels to current-frame pixels. It models
translation, in-plane rotation and uniform zoom. Person masks exclude foreground motion.
RANSAC rejects outliers; invalid fits carry quality=0 and are never treated as valid compensation.


In [ ]:
# CELL: extraction

def atomic_pickle(value,path):
    path=Path(path); tmp=path.with_name(f'.{path.name}.{os.getpid()}.tmp')  # unique per writer process
    if isinstance(value,dict) and 'video' in value:
        value=dict(value)
        video=value.pop('video')
        value['video_jpeg']=[[encode_jpeg(frame.transpose(1,2,0)) for frame in stream] for stream in video]
    with tmp.open('wb') as f: pickle.dump(value,f,protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)

def encode_jpeg(rgb):
    ok,buffer=cv2.imencode('.jpg',cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR),[cv2.IMWRITE_JPEG_QUALITY,95])
    if not ok: raise ValueError('JPEG cache encoding failed')
    return buffer.tobytes()

def decode_jpeg(buffer):
    image=cv2.imdecode(np.frombuffer(buffer,dtype=np.uint8),cv2.IMREAD_COLOR)
    if image is None: raise ValueError('Corrupt JPEG cache')
    return cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

def decode_frames(path,click_times=()):
    recent=deque(); nearest=[None]*len(click_times); distances=[float('inf')]*len(click_times)
    with av.open(str(path)) as container:
        stream=container.streams.video[0]; rate=float(stream.average_rate or 25)
        for i,frame in enumerate(container.decode(stream)):
            t=float(frame.time) if frame.time is not None else i/rate
            im=frame.to_image(); im.thumbnail((DETECTOR_IMGSZ,DETECTOR_IMGSZ))
            rgb=np.asarray(im).copy()
            recent.append((t,rgb))
            while len(recent)>2 and recent[1][0]<t-WINDOW-.1: recent.popleft()
            for j,ct in enumerate(click_times):
                if abs(t-ct)<distances[j]: nearest[j]=(t,rgb); distances[j]=abs(t-ct)
    if not recent: raise ValueError(f'No frames: {path}')
    ts=np.array([a[0] for a in recent]); wanted=ts[-1]-np.linspace(np.sqrt(WINDOW),0,N_FRAMES)**2
    ids=np.abs(ts[:,None]-wanted).argmin(0)
    return [recent[i] for i in ids],nearest

def appearance_features(rgb,box):
    h,w=rgb.shape[:2]; a=np.clip(box,0,1)
    x1,y1,x2,y2=(a*np.array([w,h,w,h])).astype(int)
    patch=rgb[max(0,y1):max(y1+1,y2),max(0,x1):max(x1+1,x2)]
    if not patch.size: return np.zeros(24,dtype=np.float32)
    hsv=cv2.cvtColor(patch,cv2.COLOR_RGB2HSV)
    return np.concatenate([np.histogram(hsv[:,:,i],bins=8,range=(0,180 if i==0 else 256))[0]/(patch.shape[0]*patch.shape[1]) for i in range(3)])

_POSE_MODEL=None
def get_pose_model():
    # Loaded lazily: cached clips (and reports mode) never load or download the detector.
    global _POSE_MODEL
    if _POSE_MODEL is None: _POSE_MODEL=YOLO(POSE_WEIGHTS)
    return _POSE_MODEL

def release_pose_model():
    global _POSE_MODEL
    _POSE_MODEL=None; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

def detect(rgb):
    # PIL input avoids RGB/BGR ambiguity in Ultralytics numpy inputs.
    result=get_pose_model().predict(Image.fromarray(rgb),imgsz=DETECTOR_IMGSZ,conf=DETECTOR_CONF,verbose=False,
                                    device=0 if DEVICE=='cuda' else 'cpu')[0]
    boxes=result.boxes.xyxyn.cpu().numpy().astype(np.float32)
    joints=result.keypoints.xyn.cpu().numpy().astype(np.float32)
    confidence=result.keypoints.conf.cpu().numpy().astype(np.float32)
    features=[]
    for i,box in enumerate(boxes):
        center=(box[:2]+box[2:])/2; size=np.maximum(box[2:]-box[:2],1e-5)
        rel=(joints[i]-center)/size; rel[confidence[i]<.25]=0
        features.append(np.r_[box,center,size,size.prod(),float(result.boxes.conf[i]),
            rel.ravel(),confidence[i],appearance_features(rgb,box)])
    return dict(boxes=boxes,joints=joints,confidence=confidence,
                features=np.asarray(features,dtype=np.float32).reshape(len(boxes),85))

def background_mask(shape,boxes):
    h,w=shape[:2]; mask=np.full((h,w),255,dtype=np.uint8)
    # Exclude screen edges, often occupied by stationary graphics.
    mask[:int(.07*h)]=0; mask[int(.93*h):]=0
    for b in boxes:
        x1,y1,x2,y2=b*np.array([w,h,w,h]); pad=.08*max(x2-x1,y2-y1)
        cv2.rectangle(mask,(max(0,int(x1-pad)),max(0,int(y1-pad))),
                      (min(w-1,int(x2+pad)),min(h-1,int(y2+pad))),0,-1)
    return mask

def fit_similarity(src,dst):
    identity=np.eye(3,dtype=np.float32)
    if len(src)<12: return identity,0.
    affine,inliers=cv2.estimateAffinePartial2D(np.asarray(src,dtype=np.float32),np.asarray(dst,dtype=np.float32),
         method=cv2.RANSAC,ransacReprojThreshold=3,maxIters=3000,confidence=.995)
    if affine is None or inliers is None: return identity,0.
    keep=inliers.ravel().astype(bool); ratio=keep.mean()
    scale=float(np.hypot(affine[0,0],affine[1,0])); angle=float(np.arctan2(affine[1,0],affine[0,0]))
    residual=np.linalg.norm(np.asarray(src)@affine[:,:2].T+affine[:,2]-dst,axis=1)
    if keep.sum()<10 or ratio<.5 or not .65<scale<1.5 or abs(angle)>.4 or np.median(residual[keep])>2.5:
        return identity,0.
    matrix=identity.copy(); matrix[:2]=affine
    return matrix,float(ratio)

def camera_motion(previous,current,prev_boxes,cur_boxes):
    identity=np.eye(3,dtype=np.float32)
    if previous.shape!=current.shape: return identity,0.
    a=cv2.cvtColor(previous,cv2.COLOR_RGB2GRAY); b=cv2.cvtColor(current,cv2.COLOR_RGB2GRAY)
    mask=background_mask(previous.shape,prev_boxes); nextmask=background_mask(current.shape,cur_boxes)
    p=cv2.goodFeaturesToTrack(a,maxCorners=1000,qualityLevel=.01,minDistance=8,mask=mask)
    if p is None or len(p)<12: return identity,0.
    q,status,_=cv2.calcOpticalFlowPyrLK(a,b,p,None,winSize=(31,31),maxLevel=4)
    if q is None: return identity,0.
    back,status2,_=cv2.calcOpticalFlowPyrLK(b,a,q,None,winSize=(31,31),maxLevel=4)
    if back is None: return identity,0.
    src=p.reshape(-1,2); dst=q.reshape(-1,2)
    good=(status.ravel()>0)&(status2.ravel()>0)&(np.linalg.norm(src-back.reshape(-1,2),axis=1)<1.5)
    h,w=a.shape; inside=(dst[:,0]>=0)&(dst[:,0]<w)&(dst[:,1]>=0)&(dst[:,1]<h)
    good &= inside
    ix=np.clip(dst[:,0].astype(int),0,w-1); iy=np.clip(dst[:,1].astype(int),0,h-1)
    good &= nextmask[iy,ix]>0
    src,dst=src[good],dst[good]
    # Require spatial coverage, not merely one small moving patch.
    if len(src)<12 or np.ptp(src[:,0])<.2*w or np.ptp(src[:,1])<.15*h: return identity,0.
    return fit_similarity(src,dst)

def extract_raw(clip,path=None,cache_path=None):
    # Default: dataset clip in the shared detector cache. Inference of other videos passes an explicit
    # path and content-keyed cache_path; those never receive annotation clicks.
    dest=RAW/(clip+'.pkl') if cache_path is None else Path(cache_path)
    if dest.exists():
        with dest.open('rb') as f: value=pickle.load(f)
        value['frames']=[decode_jpeg(frame) for frame in value.pop('frames_jpeg')]
        return value
    annotation=clicks.loc[clip] if cache_path is None and clip in clicks.index else None
    ctimes=[] if annotation is None else [float(annotation[f'{r}_click_time_seconds']) for r in ['kicker','keeper']]
    sampled,clicked=decode_frames(video_paths[clip] if path is None else path,ctimes)
    detections=[detect(rgb) for t,rgb in sampled]
    matrices=[np.eye(3,dtype=np.float32)]; quality=[0.]
    for i in range(1,N_FRAMES):
        if sampled[i][0]==sampled[i-1][0]: matrix,q=np.eye(3,dtype=np.float32),0.
        else: matrix,q=camera_motion(sampled[i-1][1],sampled[i][1],detections[i-1]['boxes'],detections[i]['boxes'])
        matrices.append(matrix); quality.append(q)
    value=dict(times=np.array([x[0] for x in sampled]),frames=[x[1] for x in sampled],
               detections=detections,camera=np.stack(matrices),camera_quality=np.array(quality),
               clicked=[dict(time=t,**detect(rgb)) for t,rgb in clicked])
    value['frames_jpeg']=[encode_jpeg(frame) for frame in value.pop('frames')]
    atomic_pickle(value,dest)
    # Identical first-run and cache-hit inputs for downstream crop processing.
    value['frames']=[decode_jpeg(frame) for frame in value.pop('frames_jpeg')]
    return value

if RUN_MODE in TRAINING_MODES:
    work_clips=rows['clip'].iloc[np.r_[smoke_train,smoke_hold]] if QUICK_SMOKE else rows['clip']
    for clip in tqdm(work_clips,desc='Label-free detections, exact click frames, camera motion'): extract_raw(clip)
    release_pose_model()
    if QUICK_SMOKE: print('Smoke extraction complete; continue through role training, fine-tuning, and inference below.')
else:
    print(f'{RUN_MODE} mode: no bulk extraction here (inference decodes only uncached requested clips).')



Supervised role selector and cross-fitting
For each annotated frame, a click identifies the containing detection nearest its box center.
Ambiguous/missed matches are skipped. Other people become negative examples for that role.
Two CatBoost binary models learn role likelihoods from box geometry, pose and color features.
Click coordinates are supervision only, never inference features. Distinct role assignment is
chosen by model probabilities, with camera-aware continuity; there is no positional role prior.
Low-confidence/unavailable roles fall back to the full frame and explicit missingness indicators.
Training clips use source-cross-fitted role predictions; held-out clips use the complete allowed
training partition's selector. Inner selection repeats this entire procedure independently.


In [ ]:
# CELL: roles_and_crops

def role_examples(indices,role):
    features=[]; labels=[]; groups=[]; sample_weights=[]; audit=[]
    name=['kicker','keeper'][role]
    for i in indices:
        clip=rows['clip'].iloc[i]
        if clip not in clicks.index: continue
        raw=extract_raw(clip); d=raw['clicked'][role]
        point=clicks.loc[clip,[f'{name}_x_norm',f'{name}_y_norm']].to_numpy(dtype=float)
        selected,method=match_click_candidate(d,point)
        audit.append(dict(clip=clip,source_id=rows.source_id.iloc[i],role=name,
                          match=method,candidates=len(d['boxes'])))
        if selected is None: continue
        y=np.zeros(len(d['boxes']),dtype=int); y[selected]=1
        features.extend(d['features']); labels.extend(y); groups.extend([rows.source_id.iloc[i]]*len(y))
        # Crowded clips should not dominate training solely by having more negatives.
        sample_weights.extend([1/len(y)]*len(y))
    return np.asarray(features,dtype=np.float32),np.asarray(labels),groups,np.asarray(sample_weights),audit


def fit_roles(indices,path):
    source_ids=set(rows.source_id.iloc[indices])
    if path.exists() and not QUICK_SMOKE:
        saved=joblib.load(path)
        assert set(saved['sources'])==source_ids
        return saved
    models=[]; counts=[]
    for role in range(2):
        x,y,_,sample_weights,match_audit=role_examples(indices,role)
        if len(y)<(8 if QUICK_SMOKE else 30) or len(np.unique(y))<2:
            raise ValueError(f'Insufficient matched role examples for role {role}: {len(y)} candidates, {int(y.sum())} positives. Inspect detector matches or enlarge the smoke subset.')
        model=CatBoostClassifier(iterations=30 if QUICK_SMOKE else 350,depth=4,learning_rate=.04,l2_leaf_reg=8,
             loss_function='Logloss',verbose=False,random_seed=42,thread_count=4)
        model.fit(x,y,sample_weight=sample_weights); models.append(model)
        pd.DataFrame(match_audit).to_csv(path.with_name(path.stem+f'_role{role}_matches.csv'),index=False)
        counts.append(dict(candidates=len(y),positives=int(y.sum()),annotated_clips=len(match_audit)))
    result=dict(models=models,sources=sorted(source_ids),counts=counts)
    joblib.dump(result,path); return result

def transform_points(points,matrix):
    return np.asarray(points)@matrix[:2,:2].T+matrix[:2,2]

def assign_roles(detection,selector,previous,matrix,quality,shape):
    n=len(detection['boxes'])
    if not n: return [None,None],np.zeros(2)
    probs=np.stack([m.predict_proba(detection['features'])[:,1] for m in selector['models']])
    centers=(detection['boxes'][:,:2]+detection['boxes'][:,2:])/2
    h,w=shape[:2]; score=np.log(np.maximum(probs,1e-8))
    for role in range(2):
        if previous[role] is not None and quality>0:
            expected=transform_points(previous[role]*[w,h],matrix)/[w,h]
            score[role]-=2*np.linalg.norm(centers-expected,axis=1)
    choices=[None]+list(range(n)); best=None; best_score=-float('inf')
    for a,b in itertools.product(choices,repeat=2):
        if a is not None and a==b: continue
        pair=(a,b)
        if any(j is not None and probs[r,j]<ROLE_MIN_PROB for r,j in enumerate(pair)): continue
        total=sum(np.log(ROLE_MIN_PROB) if j is None else score[r,j] for r,j in enumerate(pair))
        if total>best_score: best=pair; best_score=total
    return list(best),np.array([0. if j is None else probs[r,j] for r,j in enumerate(best)])

def match_click_candidate(detection, point, near_limit=0.35):
    boxes=detection['boxes']
    if not len(boxes): return None,'no_detection'
    point=np.asarray(point,dtype=float)
    centers=(boxes[:,:2]+boxes[:,2:])/2
    distances=np.linalg.norm(centers-point,axis=1)/np.maximum(boxes[:,3]-boxes[:,1],.02)
    inside=np.all(point>=boxes[:,:2]-.01,axis=1)&np.all(point<=boxes[:,2:]+.01,axis=1)
    candidates=np.flatnonzero(inside)
    method='inside'
    if not len(candidates):
        candidates=np.flatnonzero(distances<=near_limit); method='near'
    if not len(candidates): return None,'unmatched'
    order=candidates[np.argsort(distances[candidates])]
    if len(order)>1 and distances[order[1]]-distances[order[0]]<.10:
        return None,'ambiguous'
    return int(order[0]),method


def bidirectional_role_tracks(raw,selector):
    count=len(raw['detections']); tracks=np.full((count,2),-1,dtype=int)
    confidence=np.zeros((count,2),dtype=np.float32)
    nominal=int(np.abs(raw['times']-(raw['times'][-1]-.23)).argmin())
    probs=[]
    for d in raw['detections']:
        probs.append(np.stack([m.predict_proba(d['features'])[:,1] for m in selector['models']])
                     if len(d['boxes']) else np.zeros((2,0)))
    # Prefer an automatic anchor near the public annotation timing; no click is read here.
    anchors=sorted(range(count),key=lambda t:abs(raw['times'][t]-raw['times'][nominal]))
    best=None
    for t in anchors:
        selected,p=assign_roles(raw['detections'][t],selector,[None,None],np.eye(3),0,raw['frames'][t].shape)
        found=sum(j is not None for j in selected)
        key=(found,float(p.sum())-.25*abs(raw['times'][t]-raw['times'][nominal]))
        if found and (best is None or key>best[0]): best=(key,t,selected,p)
        if found==2 and abs(raw['times'][t]-raw['times'][nominal])<.12 and p.min()>=.6: break
    if best is None: return tracks,confidence,nominal
    _,anchor,seeds,seed_probs=best
    for r,j in enumerate(seeds):
        if j is not None: tracks[anchor,r]=j; confidence[anchor,r]=seed_probs[r]
    for step in [1,-1]:
        states=[]
        for r,j in enumerate(seeds):
            if j is None: states.append(None); continue
            d=raw['detections'][anchor]; b=d['boxes'][j]; h,w=raw['frames'][anchor].shape[:2]
            states.append(dict(center=(b[:2]+b[2:])/2*[w,h],height=max((b[3]-b[1])*h,2.),
                appearance=d['features'][j,-24:].copy(),last_time=raw['times'][anchor]))
        for t in range(anchor+step,count if step==1 else -1,step):
            previous=t-step; h,w=raw['frames'][t].shape[:2]
            edge=t if step==1 else previous
            good=raw['camera_quality'][edge]>0
            matrix=raw['camera'][edge] if step==1 else np.linalg.inv(raw['camera'][edge])
            d=raw['detections'][t]; boxes=d['boxes']; centers=(boxes[:,:2]+boxes[:,2:])/2*[w,h]
            scores=[]
            for r,state in enumerate(states):
                score=np.full(len(boxes),-np.inf)
                if state is not None:
                    if good:
                        state['center']=transform_points(state['center'],matrix)
                        state['height']*=float(np.hypot(matrix[0,0],matrix[1,0]))
                    gap=abs(raw['times'][t]-state['last_time'])
                    if gap<=TRACK_MAX_GAP_SECONDS:
                        distances=np.linalg.norm(centers-state['center'],axis=1)/max(state['height'],2.)
                        allowed=min(TRACK_MAX_JUMP_HEIGHTS,.15+TRACK_SPEED_HEIGHTS_PER_SECOND*gap)
                        appearance=np.abs(d['features'][:,-24:]-state['appearance']).sum(axis=1)/3 if len(boxes) else np.zeros(0)
                        valid=(distances<=allowed)&(probs[t][r]>=.15)
                        score[valid]=-distances[valid]-.4*appearance[valid]+.35*np.log(np.maximum(probs[t][r,valid],1e-8))
                scores.append(score)
            choices=[None]+list(range(len(boxes)))
            selected=(None,None); value=-3.2
            for pair in itertools.product(choices,repeat=2):
                if pair[0] is not None and pair[0]==pair[1]: continue
                candidate=sum(-1.6 if j is None else scores[r][j] for r,j in enumerate(pair))
                if candidate>value: value=candidate; selected=pair
            for r,j in enumerate(selected):
                if j is None: continue
                b=boxes[j]; states[r]=dict(center=centers[j],height=max((b[3]-b[1])*h,2.),
                    appearance=d['features'][j,-24:].copy(),last_time=raw['times'][t])
                tracks[t,r]=j; confidence[t,r]=probs[t][r,j]
    return tracks,confidence,anchor


def letterbox(rgb):
    image=Image.fromarray(rgb); image.thumbnail((SIZE,SIZE))
    canvas=Image.new('RGB',(SIZE,SIZE),(123,116,103))
    canvas.paste(image,((SIZE-image.width)//2,(SIZE-image.height)//2))
    return np.asarray(canvas)

def role_crop(rgb,box):
    h,w=rgb.shape[:2]; center=(box[:2]+box[2:])/2; size=(box[2:]-box[:2])*[1.7,1.35]
    lo=np.maximum(0,center-size/2); hi=np.minimum(1,center+size/2)
    x1,y1=(lo*[w,h]).astype(int); x2,y2=(hi*[w,h]).astype(int)
    return rgb[y1:max(y1+1,y2),x1:max(x1+1,x2)]

def make_views(raw,selector):
    videos=[]; numeric=[]; traces=[]; previous=[None,None]; prior_points=[None,None]
    track_ids,track_confidence,anchor=bidirectional_role_tracks(raw,selector)
    for t,(rgb,d) in enumerate(zip(raw['frames'],raw['detections'])):
        matrix=raw['camera'][t]; quality=raw['camera_quality'][t]; h,w=rgb.shape[:2]
        selected=[None if j<0 else int(j) for j in track_ids[t]]
        prob=track_confidence[t]
        dt=raw['times'][t]-raw['times'][t-1] if t else 0.
        frame_views=[letterbox(rgb)]; values=[]; trace=[]
        for role,j in enumerate(selected):
            if j is None:
                frame_views.append(letterbox(rgb)); values.append(np.zeros(146,dtype=np.float32))
                previous[role]=None; prior_points[role]=None; trace.append([0,0,0]); continue
            box=d['boxes'][j]; center=(box[:2]+box[2:])/2; size=np.maximum(box[2:]-box[:2],1e-5)
            xy=d['joints'][j]; conf=d['confidence'][j]; rel=(xy-center)/size; rel[conf<.25]=0
            points=np.vstack([center,xy]); valid=np.r_[True,conf>=.25]
            raw_velocity=np.zeros((18,2)); compensated=np.zeros((18,2)); motion_valid=np.zeros(18)
            if prior_points[role] is not None and dt>1e-6:
                before,before_valid=prior_points[role]; good=valid&before_valid
                raw_velocity[good]=(points-before)[good]/dt
                if quality>0:
                    expected=transform_points(before*[w,h],matrix)/[w,h]
                    compensated[good]=(points-expected)[good]/dt; motion_valid[good]=1
            # Raw and compensated velocities coexist; validity distinguishes failure from stillness.
            feature=np.r_[rel.ravel(),conf,center,size,prob[role],raw_velocity.ravel(),
                          compensated.ravel(),motion_valid]
            assert len(feature)==146
            values.append(feature.astype(np.float32)); frame_views.append(letterbox(role_crop(rgb,box)))
            previous[role]=center; prior_points[role]=(points,valid); trace.append([*center,prob[role]])
        scale=np.hypot(matrix[0,0],matrix[1,0]); angle=np.arctan2(matrix[1,0],matrix[0,0])
        camera=np.array([matrix[0,2]/w,matrix[1,2]/h,np.log(max(scale,1e-6)),angle,quality,
                         dt,raw['times'][t]-raw['times'][-1]],dtype=np.float32)
        videos.append(np.stack(frame_views)); numeric.append(np.r_[*values,camera]); traces.append(trace)
    result=dict(video=np.stack(videos).transpose(1,0,4,2,3),numeric=np.asarray(numeric,dtype=np.float32),
                traces=np.asarray(traces,dtype=np.float32),track_ids=track_ids,anchor_frame=anchor)
    assert result['video'].shape==(3,N_FRAMES,3,SIZE,SIZE)
    assert np.isfinite(result['numeric']).all()
    return result

def stage_identity(training,selection,name):
    """Fitted-stage identity: training clips, checkpoint-selection clips and configuration.
    The evaluation (prediction) list is deliberately excluded: new prediction clips never retrain."""
    return dict(name=name,training=sorted(map(int,training)),
                selection=None if selection is None else sorted(map(int,selection)),config=CFG)

def resolve_stage_folder(training,selection,name):
    identity=stage_identity(training,selection,name)
    tag='s'+hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:11]
    folder=STAGES/(name+'_'+tag)
    if folder.exists(): return folder
    for candidate in sorted(STAGES.glob(name+'_*')):
        marker=candidate/'stage_identity.json'
        if marker.exists():
            if json.loads(marker.read_text()).get('identity_tag')==tag: return candidate
            continue
        # Legacy folders also hashed the evaluation list. Reuse one only when its saved fit manifests prove
        # the same training clips, selection clips and configuration (so the fitted models are identical).
        manifests=sorted(candidate.glob('video_seed*_fit_manifest.json')); provenance=candidate/'provenance.json'
        if not manifests or not provenance.exists(): continue
        fits=[json.loads(m.read_text()) for m in manifests]
        crossfit=sorted(i for e in json.loads(provenance.read_text()) if e['kind']=='training_crossfit'
                        for i in e['prediction_indices'])
        if (crossfit==identity['training'] and all(f['config']==CFG and sorted(f['training_indices'])==identity['training']
                and sorted(f['validation_indices'])==(identity['selection'] or []) for f in fits)):
            marker.write_text(json.dumps(dict(identity_tag=tag,legacy_folder=True,
                note='Legacy stage reused: identical fitted identity; only the evaluation list differed.'),indent=2))
            print(f'Reusing existing stage {candidate.name}: same fitted identity, new prediction clips only.')
            return candidate
    return folder

def build_stage(training,evaluation,name,selection=None):
    training=np.asarray(training); evaluation=np.asarray(evaluation)
    assert not set(training)&set(evaluation)
    heldout_public=evaluation[evaluation<len(train)]
    assert not set(rows.source_id.iloc[training])&set(rows.source_id.iloc[heldout_public])
    # Private shared sources are allowed at final inference; outer/inner validation must be disjoint.
    folder=resolve_stage_folder(training,selection,name); folder.mkdir(exist_ok=True)
    if not (folder/'stage_identity.json').exists():
        identity=stage_identity(training,selection,name)
        (folder/'stage_identity.json').write_text(json.dumps(dict(identity_tag=folder.name.rsplit('_',1)[1],
            training_clips=len(identity['training']),selection_clips=None if selection is None else len(identity['selection']),
            note='Prediction clips are not part of this identity; changing them reuses the fitted models.'),indent=2))
    mapping={}; manifests=[]
    folds=GroupKFold(n_splits=ROLE_FOLDS).split(training,groups=rows.source_id.iloc[training])
    for k,(a,b) in enumerate(folds):
        fit_idx,predict_idx=training[a],training[b]
        assert not set(rows.source_id.iloc[fit_idx])&set(rows.source_id.iloc[predict_idx])
        selector=fit_roles(fit_idx,folder/f'roles_crossfit_{k}.joblib')
        for i in tqdm(predict_idx,desc=f'{name}: cross-fit crops {k}',leave=False):
            dest=folder/f'{i}.pkl'
            if not dest.exists(): atomic_pickle(make_views(extract_raw(rows['clip'].iloc[i]),selector),dest)
            mapping[int(i)]=dest
        manifests.append(dict(kind='training_crossfit',fit_sources=selector['sources'],prediction_indices=predict_idx.tolist()))
    selector=fit_roles(training,folder/'roles_all_training.joblib')
    for i in tqdm(evaluation,desc=f'{name}: held-out crops',leave=False):
        dest=folder/f'{i}.pkl'
        if not dest.exists(): atomic_pickle(make_views(extract_raw(rows['clip'].iloc[i]),selector),dest)
        mapping[int(i)]=dest
    # One evaluation entry accumulates every clip ever predicted from this fitted stage.
    previous=[]
    if (folder/'provenance.json').exists():
        previous=[i for e in json.loads((folder/'provenance.json').read_text()) if e['kind']=='evaluation'
                  for i in e['prediction_indices']]
    evaluated=sorted(set(map(int,previous))|set(map(int,evaluation)))
    assert not set(evaluated)&set(map(int,training))
    manifests.append(dict(kind='evaluation',fit_sources=selector['sources'],prediction_indices=evaluated))
    (folder/'provenance.json').write_text(json.dumps(manifests,indent=2))
    return folder,mapping,selector


End-to-end multi-view video model and pose/motion specialist
A shared backbone (selected by `BACKBONE`; VideoMAE by default) processes context, kicker and keeper streams; two heads fuse their pooled embeddings with numeric pose/motion metadata. With `TRAINING_POLICY='finetune'` all encoder parameters receive gradients (original policy); `'frozen_probe'` freezes the encoder and trains only the pooled-feature norm, numeric MLP and heads. Each stage writes `backbone_serialization.json` (checkpoint, revision, architecture, pooling, trainable-parameter counts).
Use batch size one, gradient accumulation, mixed precision, and gradient checkpointing (through each backbone's own supported interface). Inner validation selects epochs; outer refits train for those fixed epochs. Two seeds are averaged. Appearance jitter is consistent across frames; no reflection or temporal reversal changes directional labels.
Numeric scaling, CatBoost training and class priors use only the current training subset; the prior is saved as `class_prior.json` so inference never needs labels. Checkpoints and role provenance are saved per stage. Resume checkpoints record the backbone/policy identity and refuse a mismatched configuration.


In [ ]:
# CELL: model_and_training
metadata=(np.column_stack([(rows.preferred_foot=='left'),(rows.preferred_foot=='right'),
                          (rows.gender=='F'),(rows.gender=='M')]).astype(np.float32) if RUN_MODE!='reports' else None)

def load_view(mapping,i,decode_video=False):
    with Path(mapping[int(i)]).open('rb') as f: value=pickle.load(f)
    if decode_video:
        value['video']=np.stack([[decode_jpeg(frame).transpose(2,0,1) for frame in stream]
                                for stream in value['video_jpeg']])
    return value

def numeric_summary(value):
    x=value['numeric']
    return np.r_[x.mean(0),x.std(0),x[-1],x[-1]-x[0]]

class ClipDataset(torch.utils.data.Dataset):
    def __init__(self,indices,mapping,scaler,augment=False):
        self.indices=list(indices); self.mapping=mapping; self.scaler=scaler; self.augment=augment
    def __len__(self): return len(self.indices)
    def __getitem__(self,index):
        i=self.indices[index]; d=load_view(self.mapping,i,decode_video=True)
        x=d['video'].astype(np.float32)/255.
        if self.augment:
            gain=np.random.uniform(.85,1.15); shift=np.random.uniform(-.04,.04)
            x=np.clip(x*gain+shift,0,1)
        x=(x-IMAGE_MEAN[None,None,:,None,None])/IMAGE_STD[None,None,:,None,None]
        n=ablation_numeric_features(d,metadata[i])[None]
        n=self.scaler.transform(n)[0].astype(np.float32)
        y=Y[i] if i<len(train) else np.zeros(len(TARGETS),dtype=np.int64)
        return torch.from_numpy(x),torch.from_numpy(n),torch.tensor(y,dtype=torch.long)

# MultiViewVideo is created at the end of this cell from make_ablation_model_class(); the
# backbone (VideoMAE / V-JEPA 2.1 / V-JEPA 2) comes from the adapter selected in Run settings.

def direction_loss(logits, labels):
    assert len(logits) == len(TARGETS) == 2
    assert labels.ndim == 2 and labels.shape[1] == 2
    return (
        float(WEIGHTS[0]) * torch.nn.functional.cross_entropy(logits[0], labels[:, 0])
        + float(WEIGHTS[1]) * torch.nn.functional.cross_entropy(logits[1], labels[:, 1])
    )

@torch.inference_mode()
def evaluate_video_loss(model,scaler,indices,mapping):
    model.eval(); total=np.zeros(len(TARGETS)); count=0
    loader=torch.utils.data.DataLoader(ClipDataset(indices,mapping,scaler),batch_size=1,num_workers=0)
    for v,n,y in loader:
        with torch.autocast('cuda',dtype=torch.float16): logits=model(v.to(DEVICE),n.to(DEVICE))
        for j,p in enumerate(logits):
            total[j]+=torch.nn.functional.cross_entropy(p.float(),y[:,j].to(DEVICE),reduction='sum').item()
        count+=len(y)
    if not count: raise ValueError('Cannot evaluate an empty split')
    losses=total/count
    if not np.isfinite(losses).all(): raise FloatingPointError('Non-finite evaluation loss')
    return dict(loss=float(WEIGHTS@losses),**{t+'_loss':float(x) for t,x in zip(TARGETS,losses)})

def update_loss_plot(history,folder,seed,handle=None):
    # Update just this figure, preserving progress bars and earlier cell output.
    import matplotlib.pyplot as plt
    from IPython.display import display
    from matplotlib.ticker import MaxNLocator
    data=pd.DataFrame(history)
    with plt.ioff():
        fig,ax=plt.subplots(figsize=(9,5))
        ax.plot(data.epoch,data.train_loss,marker='o',label='Training')
        ax.plot(data.epoch,data.validation_loss,marker='o',linestyle='--',label='Validation')
        best=data.loc[data.validation_loss.idxmin()]
        ax.scatter([best.epoch],[best.validation_loss],s=110,marker='*',color='green',
                   zorder=5,label=f'Best validation: epoch {int(best.epoch)}')
        ax.set(xlabel='Epoch',ylabel='Weighted cross-entropy (70% kick / 30% keeper)',
               title=f'Seed {seed} | epoch {int(data.epoch.iloc[-1])}/{VIDEO_EPOCHS} | '
                     f'train {data.train_loss.iloc[-1]:.4f} | validation {data.validation_loss.iloc[-1]:.4f}')
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
        try:
            fig.savefig(folder/f'loss_curves_seed{seed}.png',dpi=150)
            if handle is None: handle=display(fig,display_id=True)
            else: handle.update(fig)
        finally:
            plt.close(fig)
    return handle

def atomic_torch_save(state,path):
    # A failed write leaves the last complete checkpoint untouched.
    path=Path(path); temporary=path.with_name(f'.{path.name}.{os.getpid()}.tmp')  # unique per writer process
    with temporary.open('wb') as stream:
        torch.save(state,stream)
        stream.flush(); os.fsync(stream.fileno())
    temporary.replace(path)

def capture_training_rng():
    numpy_state=np.random.get_state()
    return dict(python=random.getstate(),numpy=dict(name=numpy_state[0],
        keys=numpy_state[1].tolist(),position=numpy_state[2],has_gauss=numpy_state[3],
        cached_gaussian=numpy_state[4]),torch=torch.get_rng_state(),
        cuda=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [])

def restore_training_rng(state):
    random.setstate(state['python'])
    n=state['numpy']
    np.random.set_state((n['name'],np.asarray(n['keys'],dtype=np.uint32),
                         n['position'],n['has_gauss'],n['cached_gaussian']))
    torch.set_rng_state(state['torch'])
    if state['cuda'] and torch.cuda.is_available(): torch.cuda.set_rng_state_all(state['cuda'])

def fit_video(indices,mapping,folder,seed,validation_indices=None,epochs_override=None):
    planned_epochs=VIDEO_EPOCHS if epochs_override is None else int(epochs_override)
    assert planned_epochs>0
    checkpoint=folder/f'video_seed{seed}.pt'; scaler_path=folder/'numeric_scaler.joblib'
    resume_path=folder/f'video_seed{seed}_latest.pt'
    training_manifest=dict(seed=int(seed),training_indices=list(map(int,indices)),
        training_sources=sorted(set(rows.source_id.iloc[indices])),
        validation_indices=[] if validation_indices is None else list(map(int,validation_indices)),
        feature_manifest=numeric_feature_manifest(),streams=list(ablation_stream_ids()),
        planned_epochs=planned_epochs,gradient_accumulation=GRAD_ACCUM,
        config=CFG)
    manifest_path=folder/f'video_seed{seed}_fit_manifest.json'
    if manifest_path.exists():
        assert json.loads(manifest_path.read_text())==training_manifest, 'Checkpoint training provenance changed.'
    manifest_path.write_text(json.dumps(training_manifest,indent=2))
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    raw=np.stack([ablation_numeric_features(load_view(mapping,i),metadata[i]) for i in indices])
    scaler=fit_feature_scaler(raw); joblib.dump(scaler,scaler_path)
    model=MultiViewVideo(raw.shape[1]).to(DEVICE)
    serialization=dict(model.adapter.serialization(model.architecture),streams=list(ablation_stream_ids()),
                       parameters=trainable_parameter_report(model))
    (folder/'backbone_serialization.json').write_text(json.dumps(serialization,indent=2,default=str))
    optimizer=torch.optim.AdamW(apply_training_policy(model,TRAINING_POLICY),weight_decay=.05)
    amp=torch.amp.GradScaler('cuda')
    # Fine-tuning: a backbone parameter probe proves real encoder updates. Frozen probe: the encoder must
    # stay bit-identical while a head parameter changes.
    probe_name,probe=next((name,p) for name,p in model.named_parameters() if p.requires_grad and p.ndim>1
                          and (name.startswith('backbone.') or TRAINING_POLICY=='frozen_probe'))
    frozen_before=(next(p for p in model.backbone.parameters() if p.ndim>1).detach().float().cpu().clone()
                   if QUICK_SMOKE and TRAINING_POLICY=='frozen_probe' else None)
    probe_before=probe.detach().float().cpu().clone() if QUICK_SMOKE else None
    history=[]; best_validation=float('inf'); best_state=None; best_epoch=None; loss_plot=None
    start_epoch=0
    updates=0; gradient_observed=False; losses=[]
    loader=torch.utils.data.DataLoader(ClipDataset(indices,mapping,scaler,True),batch_size=1,shuffle=True,num_workers=0)
    batch_limit=min(len(loader),SMOKE_MAX_BATCHES) if QUICK_SMOKE else len(loader)
    if resume_path.exists() and not QUICK_SMOKE:
        saved=torch.load(resume_path,map_location='cpu',weights_only=True)
        assert saved['format_version']==1 and saved['seed']==seed
        assert saved['training_indices']==list(map(int,indices))
        assert saved['validation_indices']==([] if validation_indices is None else list(map(int,validation_indices)))
        assert saved['planned_epochs']==planned_epochs, 'Keep the same epoch setting when resuming.'
        saved_identity=saved.get('model_identity',LEGACY_IDENTITY)  # format 1 checkpoints predate the field
        if saved_identity!=BACKBONE_IDENTITY:
            raise ValueError(f'Checkpoint {resume_path} was trained with {saved_identity}; current settings are '
                             f'{BACKBONE_IDENTITY}. Backbone/policy changes must start a new run.')
        model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
        amp.load_state_dict(saved['amp'])
        start_epoch=int(saved['completed_epochs']); history=saved['history']
        best_validation=saved['best_validation']; best_state=saved['best_model']; best_epoch=saved['best_epoch']
        updates=saved['optimizer_updates']
        assert len(history)==start_epoch and 0<=start_epoch<=planned_epochs
        pd.DataFrame(history).to_csv(folder/f'loss_history_seed{seed}.csv',index=False)
        if validation_indices is not None and history:
            loss_plot=update_loss_plot(history,folder,seed,loss_plot)
        restore_training_rng(saved['rng'])
        del saved
        print(f'Restored seed {seed}: {start_epoch}/{planned_epochs} epochs complete. '
              + (f'Resuming at epoch {start_epoch+1}.' if start_epoch<planned_epochs else 'Training already complete.'))
    elif checkpoint.exists() and not QUICK_SMOKE:
        print(f'Seed {seed}: old weights-only checkpoint has no optimizer state; starting a fresh training run.')
    torch.cuda.synchronize()
    training_started=time.perf_counter()
    for epoch in range(start_epoch,planned_epochs):
        model.train(); optimizer.zero_grad(); total=0.
        for step,(v,n,y) in enumerate(tqdm(loader,desc=f'Video seed {seed} epoch {epoch+1}',leave=False)):
            v,n,y=v.to(DEVICE),n.to(DEVICE),y.to(DEVICE)
            with torch.autocast('cuda',dtype=torch.float16):
                logits=model(v,n)
                raw_loss=direction_loss(logits,y)
                # Correct normalization for the final short accumulation group.
                group_size=min(GRAD_ACCUM,batch_limit-(step//GRAD_ACCUM)*GRAD_ACCUM)
                loss=raw_loss/group_size
            if not torch.isfinite(raw_loss): raise FloatingPointError('Non-finite video training loss')
            amp.scale(loss).backward(); total+=raw_loss.detach().item(); losses.append(raw_loss.detach().item())
            if (step+1)%GRAD_ACCUM==0 or step+1==batch_limit:
                amp.unscale_(optimizer)
                if QUICK_SMOKE:
                    if probe.grad is not None:
                        gradient_observed |= bool(torch.isfinite(probe.grad).all() and probe.grad.abs().sum()>0)
                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                scale_before=amp.get_scale()
                amp.step(optimizer); amp.update()
                if amp.get_scale()>=scale_before: updates+=1
                optimizer.zero_grad()
            if step+1>=batch_limit: break
        record=dict(epoch=epoch+1,optimization_loss=total/batch_limit)
        if validation_indices is not None:
            # Eval-mode, unaugmented training loss is comparable to validation loss.
            for label,ids in [('train',indices),('validation',validation_indices)]:
                record.update({label+'_'+k:v for k,v in evaluate_video_loss(model,scaler,ids,mapping).items()})
            if record['validation_loss'] < best_validation:
                best_validation=record['validation_loss']
                best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
                best_epoch=epoch+1
        history.append(record)
        # The latest checkpoint is authoritative for history and best-model selection.
        # Store the best weights here too so an interrupted save cannot mismatch epochs.
        atomic_torch_save(dict(format_version=1,seed=seed,planned_epochs=planned_epochs,
            completed_epochs=epoch+1,training_indices=list(map(int,indices)),
            validation_indices=[] if validation_indices is None else list(map(int,validation_indices)),
            model={k:v.detach().cpu() for k,v in model.state_dict().items()},
            optimizer=optimizer.state_dict(),amp=amp.state_dict(),rng=capture_training_rng(),
            history=history,best_validation=best_validation,best_epoch=best_epoch,
            best_model=best_state,optimizer_updates=updates,model_identity=BACKBONE_IDENTITY),resume_path)
        print(f'Checkpoint saved: seed {seed}, epoch {epoch+1}/{planned_epochs} -> {resume_path}')
        pd.DataFrame(history).to_csv(folder/f'loss_history_seed{seed}.csv',index=False)
        print(record)
        if validation_indices is not None:
            loss_plot=update_loss_plot(history,folder,seed,loss_plot)
    torch.cuda.synchronize()
    training_seconds=time.perf_counter()-training_started
    if QUICK_SMOKE:
        delta=float((probe.detach().float().cpu()-probe_before).abs().max())
        frozen_delta=(float((next(p for p in model.backbone.parameters() if p.ndim>1).detach().float().cpu()
                             -frozen_before).abs().max()) if frozen_before is not None else None)
        checks=dict(training_policy=TRAINING_POLICY,backbone=BACKBONE_SPEC['name'],probe_parameter=probe_name,
                    probe_max_change=delta,frozen_backbone_max_change=frozen_delta,
                    finite_nonzero_probe_gradient=gradient_observed,optimizer_updates=updates,
                    trainable_parameters=trainable_parameter_report(model),
                    batches=len(losses),losses=losses,training_seconds=training_seconds,
                    seconds_per_training_batch=training_seconds/len(losses))
        (folder/'video_training_checks.json').write_text(json.dumps(checks,indent=2))
        assert gradient_observed and updates>0 and delta>0, f'Training update check failed: {checks}'
        assert frozen_delta in (None,0.0), f'Frozen encoder changed: {checks}'
    if validation_indices is None:
        atomic_torch_save(dict(model={k:v.detach().cpu() for k,v in model.state_dict().items()}),checkpoint)
    else:
        assert best_state is not None
        atomic_torch_save(dict(model=best_state,best_epoch=best_epoch,validation_loss=best_validation),checkpoint)
        model.load_state_dict(best_state)
    return model,scaler

@torch.inference_mode()
def predict_video(model,scaler,indices,mapping):
    model.eval(); out=[[] for _ in TARGETS]
    loader=torch.utils.data.DataLoader(ClipDataset(indices,mapping,scaler),batch_size=1,num_workers=0)
    torch.cuda.synchronize(); started=time.perf_counter()
    for v,n,_ in tqdm(loader,desc='Video inference',leave=False):
        with torch.autocast('cuda',dtype=torch.float16): pred=model(v.to(DEVICE),n.to(DEVICE))
        for j,p in enumerate(pred): out[j].append(p.float().softmax(1).cpu().numpy())
    torch.cuda.synchronize()
    model.last_inference_seconds_per_clip=(time.perf_counter()-started)/len(indices)
    return [np.concatenate(p) for p in out]

def fit_predict_stage(training,evaluation,name):
    folder,mapping,selector=build_stage(training,evaluation,name)
    neural=[np.zeros((len(evaluation),len(c))) for c in CLASSES]
    for seed in VIDEO_SEEDS:
        model,scaler=fit_video(training,mapping,folder,seed)
        for j,p in enumerate(predict_video(model,scaler,evaluation,mapping)): neural[j]+=p/len(VIDEO_SEEDS)
        del model; gc.collect(); torch.cuda.empty_cache()
    x=np.stack([ablation_numeric_features(load_view(mapping,i),metadata[i]) for i in training])
    z=np.stack([ablation_numeric_features(load_view(mapping,i),metadata[i]) for i in evaluation])
    predictions=[]; trees=[]
    for j,c in enumerate(CLASSES):
        tree=CatBoostClassifier(iterations=30 if QUICK_SMOKE else 600,depth=4,learning_rate=.035,l2_leaf_reg=8,
             loss_function='MultiClass',verbose=False,random_seed=42,thread_count=4)
        tree.fit(x,Y[training,j]); prob=np.zeros((len(evaluation),len(c)))
        prob[:,np.asarray(tree.classes_,dtype=int)]=tree.predict_proba(z)
        prior=np.bincount(Y[training,j],minlength=len(c))/len(training)
        predictions.append([neural[j],prob,np.tile(prior,(len(evaluation),1))]); trees.append(tree)
    joblib.dump(trees,folder/'pose_camera_trees.joblib')
    return predictions,folder,mapping,selector

def select_mixture(predictions,truth):
    grid=[np.array([a,b,4-a-b])/4 for a in range(5) for b in range(5-a)]
    return max(grid,key=lambda w:(accuracy_score(truth,sum(x*p for x,p in zip(w,predictions)).argmax(1)),-np.square(w).sum()))


# Stream-specific heads are retrained for each ablation; production keeps the original architecture.
MultiViewVideo=make_ablation_model_class()


Timing benchmark: run this on the GPU intended for full training
The smoke run benchmarks 8 warm-up plus 24 measured training minibatches by default,
then warmed inference. It uses the full run's gradient accumulation and optimizer settings.
Only training-source clips are used for these extra updates. No benchmark weights are saved.
Each timing includes loading a cache from OUTPUT_ROOT, JPEG decoding, preprocessing and device transfers.
The output shows training/inference hours, observed data-loading share and peak GPU memory.
The 25% and 50% allowances are planning scenarios, not confidence bounds. Preprocessing,
role/tree fitting, model loading and checkpoint writes are not included in the base estimate.
Repeated access to a small sample can make storage caching look better than a full run.
Set RUN_TIMING_BENCHMARK=False if you only want the short execution smoke test.


In [ ]:
# CELL: timing_benchmark

def full_video_workload():
    if EVALUATION_MODE=='holdout':
        ntrain=len(split_indices['training']); nval=len(split_indices['validation'])
        seeds=len(FULL_TRAIN_SETTINGS['seeds']); epochs=FULL_TRAIN_SETTINGS['epochs']
        return dict(training_batches=seeds*epochs*ntrain,
            inference_clips=seeds*(epochs*(ntrain+nval)+nval+len(split_indices['test'])),video_fits=seeds)
    # Maximum-budget estimate: inner validation can choose fewer outer epochs.
    batches=0; inference=0; fits=0
    for item in saved_diagnostic_splits()['folds']:
        ninner=len(item['inner_training']); nhold=len(item['inner_validation'])
        ntrain=len(item['training']); nval=len(item['validation'])
        seeds=len(FULL_TRAIN_SETTINGS['seeds']); epochs=FULL_TRAIN_SETTINGS['epochs']
        batches+=seeds*epochs*(ninner+ntrain)
        inference+=seeds*(epochs*(ninner+nhold)+nhold+nval+(len(test) if PREDICT_PRIVATE else 0))
        fits+=2*seeds
    return dict(training_batches=int(batches),inference_clips=int(inference),video_fits=int(fits))


def summarize_benchmark(training_seconds,data_seconds,inference_seconds,workload):
    training_seconds=np.asarray(training_seconds,dtype=float)
    data_seconds=np.asarray(data_seconds,dtype=float)
    inference_seconds=np.asarray(inference_seconds,dtype=float)
    assert len(training_seconds)>0 and len(inference_seconds)>0
    assert len(training_seconds)==len(data_seconds)
    assert np.isfinite(training_seconds).all() and np.isfinite(inference_seconds).all()
    assert np.isfinite(data_seconds).all() and (data_seconds>=0).all()
    assert (training_seconds>0).all() and (inference_seconds>0).all()
    assert (data_seconds<=training_seconds).all()
    train_hours=float(training_seconds.mean()*workload['training_batches']/3600)
    infer_hours=float(inference_seconds.mean()*workload['inference_clips']/3600)
    subtotal=train_hours+infer_hours
    return dict(workload=workload,measured_training_batches=len(training_seconds),
        measured_inference_clips=len(inference_seconds),
        seconds_per_training_batch=float(training_seconds.mean()),
        training_batch_p50_seconds=float(np.median(training_seconds)),
        training_batch_p90_seconds=float(np.quantile(training_seconds,.9)),
        seconds_per_inference_clip=float(inference_seconds.mean()),
        training_data_loading_fraction=float(data_seconds.sum()/training_seconds.sum()),
        training_hours=train_hours,inference_hours=infer_hours,video_subtotal_hours=subtotal,
        planning_with_25_percent_allowance_hours=subtotal*1.25,
        planning_with_50_percent_allowance_hours=subtotal*1.5)

def benchmark_video(model,scaler,training_indices,mapping):
    accumulation=FULL_TRAIN_SETTINGS['accumulation']
    assert TIMING_WARMUP_BATCHES>=accumulation and TIMING_WARMUP_BATCHES%accumulation==0
    assert TIMING_MEASURED_BATCHES>=accumulation and TIMING_MEASURED_BATCHES%accumulation==0
    assert TIMING_INFERENCE_CLIPS>0
    dataset=ClipDataset(training_indices,mapping,scaler,True)
    loader=torch.utils.data.DataLoader(dataset,batch_size=1,shuffle=True,num_workers=0)
    iterator=iter(loader)
    optimizer=torch.optim.AdamW(apply_training_policy(model,TRAINING_POLICY),weight_decay=.05)
    amp=torch.amp.GradScaler('cuda')
    model.train(); optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
    measured=[]; loading=[]; updates=0
    total=TIMING_WARMUP_BATCHES+TIMING_MEASURED_BATCHES
    for step in tqdm(range(total),desc='Timing benchmark: warm-up + measured training'):
        torch.cuda.synchronize(); started=time.perf_counter()
        try: v,n,y=next(iterator)
        except StopIteration:
            iterator=iter(loader); v,n,y=next(iterator)
        data_elapsed=time.perf_counter()-started
        v,n,y=v.to(DEVICE),n.to(DEVICE),y.to(DEVICE)
        with torch.autocast('cuda',dtype=torch.float16):
            logits=model(v,n)
            loss=direction_loss(logits,y)
            loss=loss/accumulation
        if not torch.isfinite(loss): raise FloatingPointError('Non-finite timing benchmark loss')
        amp.scale(loss).backward()
        if (step+1)%accumulation==0:
            amp.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            previous_scale=amp.get_scale(); amp.step(optimizer); amp.update()
            if step>=TIMING_WARMUP_BATCHES and amp.get_scale()>=previous_scale: updates+=1
            optimizer.zero_grad(set_to_none=True)
        torch.cuda.synchronize(); elapsed=time.perf_counter()-started
        if step>=TIMING_WARMUP_BATCHES:
            measured.append(elapsed); loading.append(data_elapsed)
    assert updates==TIMING_MEASURED_BATCHES//accumulation, 'Skipped optimizer updates; benchmark is not representative.'
    peak_gib=torch.cuda.max_memory_allocated()/1024**3
    del optimizer,amp; model.zero_grad(set_to_none=True)
    inference=[]; infer_dataset=ClipDataset(training_indices,mapping,scaler,False)
    model.eval()
    with torch.inference_mode():
        for step in range(2+TIMING_INFERENCE_CLIPS):
            torch.cuda.synchronize(); started=time.perf_counter()
            v,n,_=infer_dataset[step%len(infer_dataset)]
            with torch.autocast('cuda',dtype=torch.float16):
                outputs=model(v[None].to(DEVICE),n[None].to(DEVICE))
            outputs=[p.float().softmax(1).cpu().numpy() for p in outputs]
            torch.cuda.synchronize(); elapsed=time.perf_counter()-started
            assert all(np.isfinite(p).all() for p in outputs)
            if step>=2: inference.append(elapsed)
    result=summarize_benchmark(measured,loading,inference,full_video_workload())
    result.update(gpu=torch.cuda.get_device_name(),peak_training_gpu_allocated_gib=peak_gib,
        backbone=BACKBONE_IDENTITY,frames=N_FRAMES,resolution=SIZE,trainable_parameters=trainable_parameter_report(model),
        warmup_training_batches=TIMING_WARMUP_BATCHES,gradient_accumulation=accumulation,
        full_settings=FULL_TRAIN_SETTINGS,measured_batch_seconds=measured,
        caveat='Warmed small-subset maximum-epoch projection, not a full-run benchmark. Inner-selected outer epochs may reduce runtime. Includes inner loss evaluation; excludes checkpoint transfer overhead. Base estimate excludes preprocessing, role/tree fitting, model loading and checkpoint writes. Drive caching and GPU load can change throughput. Allowances are scenarios, not confidence bounds.')
    (RUN/'timing_estimate.json').write_text(json.dumps(result,indent=2))
    table=pd.DataFrame([
        ('Video training',result['training_hours']),('Video inference',result['inference_hours']),
        ('Video subtotal (excludes other stages)',result['video_subtotal_hours']),
        ('Subtotal + 25% planning allowance',result['planning_with_25_percent_allowance_hours']),
        ('Subtotal + 50% planning allowance',result['planning_with_50_percent_allowance_hours'])],columns=['Component','Estimated hours'])
    table.to_csv(RUN/'timing_estimate.csv',index=False)
    print(f"TIMING ESTIMATE: {result['gpu']}; {result['workload']['video_fits']} video fits")
    display(table.round(2))
    print(f"Data loading: {100*result['training_data_loading_fraction']:.1f}% of measured training time; peak allocated GPU memory: {peak_gib:.1f} GiB")
    print(result['caveat']); print('Saved:',RUN/'timing_estimate.json')
    return result



## Train, validate, and test
`EVALUATION_MODE='nested_cv'` runs the saved source-grouped inner/outer plan. Inner validation selects per-seed epochs, ensemble weights and reference components; outer validation is read only after prediction. With `FOLD_SELECTION=None` all folds run in this process and are then pooled. With `FOLD_SELECTION=[k]` only fold *k* runs (inner selection first, then its outer refit) and writes only `folds/fold<k>/`; pooled reports are produced by whichever job completes the last fold, after checking that every fold has the same experiment identity. `[]` only aggregates.

`EVALUATION_MODE='development'` is the bounded preset (one seed, few epochs, fixed label-free source-disjoint subset, saved split). Its validation subset also selects checkpoints, so results are **development-only screening**, never a full OOF result, and never a submission. `holdout` keeps the 75/20/5 workflow; its validation score is a tuning metric. All modes keep epoch-boundary checkpoint recovery and live loss curves where validation is permitted.


In [ ]:
# CELL: nested_orchestration
"""Nested source-grouped experiment orchestration, embedded into the notebook."""

def saved_diagnostic_splits():
    identity=dict(dataset=DATA_ID,seed=SEED,outer_folds=OUTER_FOLDS,
                  clips=train['clip'].tolist(),sources=train.source_id.tolist())
    split_id=hashlib.sha256(json.dumps(identity,sort_keys=True).encode()).hexdigest()[:16]
    folder=OUTPUT/'diagnostic_splits'; folder.mkdir(exist_ok=True)
    path=folder/f'{split_id}.json'
    if path.exists():
        plan=json.loads(path.read_text())
        assert plan['identity']==identity, 'Saved split identity does not match the dataset.'
    else:
        folds=[]
        splitter=StratifiedGroupKFold(n_splits=OUTER_FOLDS,shuffle=True,random_state=SEED)
        for fold,(tr,va) in enumerate(splitter.split(np.zeros(len(train)),Y[:,0],train.source_id)):
            # Only outer-training labels may be consulted to ensure inner training has all classes.
            for offset in range(100):
                a,b=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=SEED+offset).split(
                    tr,groups=train.source_id.iloc[tr]))
                inner,hold=tr[a],tr[b]
                if (train.source_id.iloc[inner].nunique()>=ROLE_FOLDS and
                    all(len(np.unique(Y[inner,j]))==len(c) for j,c in enumerate(CLASSES))): break
            else: raise ValueError('Cannot create inner training partition with all classes and enough sources.')
            folds.append(dict(fold=fold,training=tr.tolist(),validation=va.tolist(),
                              inner_training=inner.tolist(),inner_validation=hold.tolist()))
        plan=dict(split_id=split_id,identity=identity,folds=folds)
        atomic_write_text(path,json.dumps(plan,indent=2))
    coverage=np.zeros(len(train),dtype=int)
    for item in plan['folds']:
        tr,va,inner,hold=[np.asarray(item[k],dtype=int) for k in
                          ['training','validation','inner_training','inner_validation']]
        assert set(tr)|set(va)==set(range(len(train))) and not set(tr)&set(va)
        assert set(inner)|set(hold)==set(tr) and not set(inner)&set(hold)
        assert not set(train.source_id.iloc[tr]) & set(train.source_id.iloc[va])
        assert not set(train.source_id.iloc[inner]) & set(train.source_id.iloc[hold])
        coverage[va]+=1
    assert (coverage==1).all(), 'Saved outer validation partitions must cover each clip exactly once.'
    return plan


def diagnostic_stage(training,evaluation,name,validation=None,selected_epochs=None):
    # build_stage fits cross-fitted training crops and held-out evaluation crops using training sources only.
    folder,mapping,selector=build_stage(training,evaluation,name,selection=validation)
    component={}; selected={}
    for seed in VIDEO_SEEDS:
        epochs=VIDEO_EPOCHS if selected_epochs is None else int(selected_epochs[str(seed)])
        model,scaler=fit_video(training,mapping,folder,seed,validation_indices=validation,epochs_override=epochs)
        component[f'video_seed{seed}']=predict_video(model,scaler,evaluation,mapping)
        state=torch.load(folder/f'video_seed{seed}.pt',map_location='cpu',weights_only=True)
        selected[str(seed)]=int(state.get('best_epoch',epochs))
        del model,state; gc.collect(); torch.cuda.empty_cache()
    component['video_ensemble']=[sum(component[f'video_seed{s}'][j] for s in VIDEO_SEEDS)/len(VIDEO_SEEDS)
                                 for j in range(len(TARGETS))]
    x=np.stack([ablation_numeric_features(load_view(mapping,i),metadata[i]) for i in training])
    z=np.stack([ablation_numeric_features(load_view(mapping,i),metadata[i]) for i in evaluation])
    component['class_prior']=[]; trees=[]; priors=[]
    if x.shape[1]: component['pose_motion_tree']=[]
    for j,c in enumerate(CLASSES):
        prior=np.bincount(Y[training,j],minlength=len(c))/len(training)
        priors.append(prior.tolist())
        component['class_prior'].append(np.tile(prior,(len(evaluation),1)))
        if x.shape[1]:
            tree=CatBoostClassifier(iterations=30 if QUICK_SMOKE else 600,depth=4,learning_rate=.035,
                l2_leaf_reg=8,loss_function='MultiClass',verbose=False,random_seed=SEED,thread_count=4)
            tree.fit(x,Y[training,j]); prob=np.zeros((len(evaluation),len(c)))
            prob[:,np.asarray(tree.classes_,dtype=int)]=tree.predict_proba(z)
            component['pose_motion_tree'].append(prob); trees.append(tree)
    if trees: joblib.dump(trees,folder/'pose_camera_trees.joblib')
    # Training-only class prior persisted for inference; inference never needs labels to rebuild it.
    (folder/'class_prior.json').write_text(json.dumps(dict(prior=priors,class_order=CLASSES,
        training_clip_count=len(training),source='training_labels_at_fit',
        training_clips_sha=stable_hash(train['clip'].iloc[np.asarray(training)].tolist())),indent=2))
    provenance=dict(training_indices=list(map(int,training)),training_sources=sorted(set(rows.source_id.iloc[training])),
        evaluation_indices=list(map(int,evaluation)),checkpoint_selection_sources=[] if validation is None else
        sorted(set(rows.source_id.iloc[validation])),selected_epochs=selected,
        class_order=dict(zip(TARGETS,CLASSES)),features=numeric_feature_manifest(),
        components={name:dict(fit_sources=sorted(set(rows.source_id.iloc[training]))) for name in component},
        scaler_fit_sources=sorted(set(rows.source_id.iloc[training])),
        role_provenance_file=str(folder/'provenance.json'))
    public=np.asarray(evaluation)[np.asarray(evaluation)<len(train)]
    assert not set(rows.source_id.iloc[training]) & set(rows.source_id.iloc[public])
    (folder/'component_provenance.json').write_text(json.dumps(provenance,indent=2))
    return component,folder,mapping,selected


def select_diagnostic_mixture(components,truth):
    names=['video_ensemble']+(['pose_motion_tree'] if 'pose_motion_tree' in components else [])+['class_prior']
    grid=[np.asarray(x,dtype=float)/4 for x in itertools.product(range(5),repeat=len(names)) if sum(x)==4]
    weights=[]
    for j in range(len(TARGETS)):
        weights.append(max(grid,key=lambda w:(float((sum(v*components[n][j] for v,n in zip(w,names)).argmax(1)==truth[:,j]).mean()),
                                               -float(np.square(w).sum()))))
    return names,weights


def _nested_context(variant):
    """Switch RUN/STAGES/CFG to the variant's nested experiment; returns the restore tuple and split plan."""
    global RUN,STAGES,CFG
    saved=(RUN,STAGES,CFG)
    set_active_ablation(variant)
    plan=saved_diagnostic_splits()
    CFG=dict(CFG,diagnostics_version='nested-components-v1',split_id=plan['split_id'],
             ablation=variant,feature_manifest=numeric_feature_manifest(),gradient_accumulation=GRAD_ACCUM)
    if variant!='production':
        RUN=saved[0]/'ablations'/variant; RUN.mkdir(parents=True,exist_ok=True)
    STAGES=RUN/'stages'; STAGES.mkdir(exist_ok=True)
    fold_ids=np.full(len(train),-1,dtype=int)
    for item in plan['folds']: fold_ids[item['validation']]=item['fold']
    # Identical content from every fold job; atomic unique-temp writes never interleave.
    atomic_write_text(RUN/'folds.csv',train[['clip','source_id']].assign(fold=fold_ids).to_csv(index=False))
    atomic_write_text(RUN/'experiment_manifest.json',json.dumps(dict(config=CFG,split_plan=plan,
        classes=dict(zip(TARGETS,CLASSES)),seeds=VIDEO_SEEDS,experiment_identity=stable_hash(CFG),
        interpretation='Exploratory outer-fold diagnostics; repeated inspection is development feedback, not a final untouched estimate.'),indent=2))
    return saved,plan,fold_ids


def _restore_context(saved):
    global RUN,STAGES,CFG
    RUN,STAGES,CFG=saved
    set_active_ablation('production')


def fold_output_dir(fold):
    return RUN/'folds'/f'fold{int(fold)}'


def run_nested_fold(item,private_inference=False):
    """Inner selection, then the outer refit, for ONE saved fold. Writes only folds/fold<k>/ and its stages."""
    fold=int(item['fold']); out=fold_output_dir(fold)
    if (out/'COMPLETE').exists():
        result=json.loads((out/'result.json').read_text())
        if result['experiment_identity']!=stable_hash(CFG) or result['private_inference']!=bool(private_inference):
            raise ValueError(f'{out} was produced by a different configuration; it will not be mixed into this run.')
        print(f'Fold {fold}: already complete ({out}).'); return result
    out.mkdir(parents=True,exist_ok=True)
    with RunWriterLock(RUN,f'fold{fold}'):
        tr,va,inner,hold=[np.asarray(item[k],dtype=int) for k in
                          ['training','validation','inner_training','inner_validation']]
        ip,inner_folder,_,selected=diagnostic_stage(inner,hold,f'fold{fold}_inner',validation=hold)
        references=select_reference_components(ip,Y[hold],TARGETS,CLASSES,WEIGHTS)
        names,weights=select_diagnostic_mixture(ip,Y[hold])
        evaluation=np.r_[va,np.arange(len(train),len(rows))] if private_inference else va
        # Epoch counts are fixed using INNER validation before training on all outer-training sources.
        policy=hashlib.sha256(json.dumps(selected,sort_keys=True).encode()).hexdigest()[:8]
        op,folder,mapping,_=diagnostic_stage(tr,evaluation,f'fold{fold}_outer_{policy}',selected_epochs=selected)
        op['ensemble']=[sum(w*op[n][j] for w,n in zip(weights[j],names)) for j in range(len(TARGETS))]
        collect_tracking_diagnostics(va,mapping,fold,output_dir=RUN/f'tracking_fold{fold}')
        acc=[float((op['ensemble'][j][:len(va)].argmax(1)==Y[va,j]).mean()) for j in range(len(TARGETS))]
        result=dict(fold=fold,stage_folder=str(folder),inner_stage_folder=str(inner_folder),
            seeds=list(VIDEO_SEEDS),selected_epochs=selected,ensemble_components=names,
            weights=[w.tolist() for w in weights],reference_components=references,accuracy=acc,
            score=float(WEIGHTS@acc),selection_sources=sorted(set(train.source_id.iloc[hold])),
            fit_sources=sorted(set(train.source_id.iloc[tr])),outer_sources=sorted(set(train.source_id.iloc[va])),
            experiment_identity=stable_hash(CFG),private_inference=bool(private_inference),attempt=ATTEMPT_ID)
        assert not set(result['selection_sources']) & set(result['outer_sources'])
        arrays={f'{n}__task{j}':p[j][:len(va)] for n,p in op.items() for j in range(len(TARGETS))}
        if private_inference:
            arrays.update({f'private_ensemble__task{j}':op['ensemble'][j][len(va):] for j in range(len(TARGETS))})
        tmp=out/f'.component_predictions.{os.getpid()}.npz'
        np.savez_compressed(tmp,indices=va,clips=train['clip'].iloc[va].to_numpy(dtype=str),**arrays)
        tmp.replace(out/'component_predictions.npz')
        atomic_write_text(out/'result.json',json.dumps(result,indent=2))
        atomic_write_text(out/'COMPLETE',stable_hash(CFG))   # published last
        print(f'Fold {fold}: weighted accuracy={WEIGHTS@acc:.4f}; selected epochs={selected}')
        return result


def aggregate_nested(plan,fold_ids,private_inference=False):
    """Pool completed folds. Refuses while any fold is missing or was produced by another configuration."""
    missing=[f['fold'] for f in plan['folds'] if not (fold_output_dir(f['fold'])/'COMPLETE').exists()]
    if missing:
        print(f'Aggregation waits for folds {missing}; pooled reports are not written yet.'); return None
    reports=[json.loads((fold_output_dir(f['fold'])/'result.json').read_text()) for f in plan['folds']]
    for r in reports:
        if r['experiment_identity']!=stable_hash(CFG):
            raise ValueError(f'Fold {r["fold"]} has identity {r["experiment_identity"]}, expected {stable_hash(CFG)}.')
        if r['private_inference']!=bool(private_inference):
            raise ValueError(f'Fold {r["fold"]} private_inference={r["private_inference"]}; rerun it or change PREDICT_PRIVATE.')
    with RunWriterLock(RUN,'aggregate'):
        registry=ComponentOOF(train,Y,fold_ids,TARGETS,CLASSES,WEIGHTS)
        references={}; outer_mapping={}; quality=[]; frames=[]; click_rows=[]
        private=[np.zeros((len(test),len(c))) for c in CLASSES] if private_inference else None
        for item,r in zip(plan['folds'],reports):
            fold=int(item['fold']); va=np.asarray(item['validation'],dtype=int)
            references[fold]=r['reference_components']
            with np.load(fold_output_dir(fold)/'component_predictions.npz') as saved:
                if not np.array_equal(saved['indices'],va): raise ValueError(f'Fold {fold} predictions are misaligned')
                names=sorted({k.rsplit('__task',1)[0] for k in saved.files if '__task' in k and not k.startswith('private_')})
                for name in names:
                    registry.add(name,va,[saved[f'{name}__task{j}'] for j in range(len(TARGETS))],train['clip'].iloc[va].tolist(),CLASSES)
                if private_inference:
                    for j in range(len(TARGETS)): private[j]+=saved[f'private_ensemble__task{j}']/len(plan['folds'])
            folder=RUN/'stages'/Path(r['stage_folder']).name
            outer_mapping.update({int(i):folder/f'{int(i)}.pkl' for i in va})
            tdir=RUN/f'tracking_fold{fold}'
            quality.append(pd.read_csv(tdir/'tracking_quality.csv'))
            frames.append(pd.read_csv(tdir/'tracking_frames.csv'))
            try: click_rows.append(pd.read_csv(tdir/'role_click_audit.csv'))
            except pd.errors.EmptyDataError: pass
        atomic_write_text(RUN/'fold_results.json',json.dumps(reports,indent=2))
        q=pd.concat(quality,ignore_index=True)
        q.to_csv(RUN/'tracking_quality.csv',index=False)
        pd.concat(frames,ignore_index=True).to_csv(RUN/'tracking_frames.csv',index=False)
        combined_click_audits=pd.concat(click_rows,ignore_index=True) if click_rows else pd.DataFrame()
        combined_click_audits.to_csv(RUN/'role_click_audit.csv',index=False)
        summarize_role_click_audits(combined_click_audits).to_csv(RUN/'role_click_summary.csv',index=False)
        # Tracking exports carry source/fold keys too. Verify them against the saved
        # OOF assignment before removing duplicated columns for the clip-keyed join.
        expected=train[['clip','source_id']].assign(fold=fold_ids)
        if not q['clip'].is_unique or set(q['clip'])!=set(expected['clip']):
            raise ValueError('Tracking quality must cover every OOF clip exactly once.')
        aligned=expected.merge(q[['clip','source_id','fold']],on='clip',how='left',
                               suffixes=('_expected','_audit'),validate='one_to_one')
        if not ((aligned.source_id_expected==aligned.source_id_audit).all()
                and (aligned.fold_expected==aligned.fold_audit).all()):
            raise ValueError('Tracking quality source/fold provenance differs from the saved OOF plan.')
        exported=registry.export(RUN/'component_reports',
            quality=q.drop(columns=['source_id','fold']),references=references,ensemble='ensemble')
        # Standard artifact names retain compatibility with existing OOF diagnostics and heatmaps.
        oof=[np.full((len(train),len(c)),np.nan) for c in CLASSES]
        for item in plan['folds']:
            va=np.asarray(item['validation'])
            with np.load(fold_output_dir(item['fold'])/'component_predictions.npz') as saved:
                for j in range(len(TARGETS)): oof[j][va]=saved[f'ensemble__task{j}']
        tmp=RUN/f'.predictions.{os.getpid()}.npz'
        np.savez_compressed(tmp,fold_ids=fold_ids,
            **{f'oof_{j}':p for j,p in enumerate(oof)},
            **({f'private_{j}':p for j,p in enumerate(private)} if private is not None else {}))
        tmp.replace(RUN/'predictions.npz')
        export_run_inference_manifest(RUN)
        if GENERATE_TRACKING_AUDITS:
            # Export helper returns its errors table; use explicit ensemble rows only.
            errors=exported['errors']; errors=errors[errors.component=='ensemble']
            render_tracking_audits(errors,outer_mapping,RUN/'visual_audits',
                high_loss_count=AUDIT_HIGH_LOSS,correct_count=AUDIT_CORRECT,seed=SEED)
        return dict(oof=oof,private=private,fold_ids=fold_ids,reports=reports,exported=exported,run=str(RUN))


def run_nested_diagnostics(variant='production',private_inference=False,fold_selection=None):
    """fold_selection=None: every fold in this process, then aggregate. A list of fold numbers runs only
    those folds (one Slurm job per fold) and aggregates only once every fold is complete; [] aggregates only."""
    saved,plan,fold_ids=_nested_context(variant)
    try:
        selected=[f['fold'] for f in plan['folds']] if fold_selection is None else [int(f) for f in fold_selection]
        unknown=set(selected)-{f['fold'] for f in plan['folds']}
        if unknown: raise ValueError(f'FOLD_SELECTION has unknown folds {sorted(unknown)}')
        for item in plan['folds']:
            if item['fold'] in selected: run_nested_fold(item,private_inference)
        return aggregate_nested(plan,fold_ids,private_inference)
    finally:
        _restore_context(saved)


def run_optional_ablations():
    variants=list(ABLATION_VARIANTS)
    if len(set(variants))!=len(variants): raise ValueError('Duplicate ablation variants.')
    if any(name not in ABLATION_CONFIGS for name in variants): raise ValueError('Unknown ablation variant.')
    print(f'Ablation plan: {len(variants)} variants x {OUTER_FOLDS} folds x 2 stages x '
          f'{len(VIDEO_SEEDS)} seeds = {len(variants)*OUTER_FOLDS*2*len(VIDEO_SEEDS)} video fits (before cache reuse).')
    print('Numeric variants also fit two tabular specialists per stage; each stage fits cross-fitted role selectors.')
    if not RUN_ABLATIONS:
        print('Disabled. Set RUN_ABLATIONS=True only after inspecting diagnostics and the compute plan.'); return
    if QUICK_SMOKE: raise ValueError('Use the diagnostics smoke checks for cheap ablation checks; full ablations require QUICK_SMOKE=False.')
    if EVALUATION_MODE!='nested_cv': raise ValueError('Controlled ablations require the saved nested outer folds.')
    reference_path=RUN/'component_reports'/'component_metrics.csv'
    if not reference_path.exists(): raise ValueError('Finish the production nested diagnostics before ablations.')
    reference_manifest=json.loads((RUN/'experiment_manifest.json').read_text())
    split_id=saved_diagnostic_splits()['split_id']
    if reference_manifest['split_plan']['split_id']!=split_id:
        raise ValueError('Production report uses different saved folds; rerun production diagnostics first.')
    results={'production':dict(metrics=pd.read_csv(reference_path))}
    for variant in variants:
        if variant=='production': continue  # Main diagnostic run already supplies the reference.
        result=run_nested_diagnostics(variant,private_inference=False)
        candidate_manifest=json.loads((Path(result['run'])/'experiment_manifest.json').read_text())
        if candidate_manifest['split_plan']['split_id']!=split_id:
            raise ValueError('Ablation and production runs must use identical saved folds.')
        results[variant]=result['exported']
    # Comparison uses identical folds and reports both fold and pooled results.
    # This helper explicitly compares adjacent feature additions (and production
    # versus H), rather than treating every variant as a change from production.
    comparison=compare_ablation_metrics(results,RUN,component='ensemble')
    print('Saved matched-fold and pooled ablation deltas:',RUN/'ablation_comparisons.csv')
    if comparison.empty:
        print('No configured reference pairs were run; include both variants in a documented comparison pair.')
    return comparison


### Inference-only bindings
Used when `RUN_MODE='inference'`. They load saved selectors, scalers, trees, priors and weights from `TRAINED_RUN_DIR`; only label-free decoding/detection/cropping is new work.

In [ ]:
# CELL: inference_glue
# Notebook bindings for the inference-only workflow. These call only label-free decoding/detection/cropping
# (extract_raw, make_views with a SAVED selector) and load saved weights; no fitting function is referenced.

def resolve_inference_clips(spec):
    """Return (clip frame with clip/path/preferred_foot/gender/in_dataset, routing)."""
    columns=['clip','source_id','preferred_foot','gender']
    if spec=='private':
        frame=test[columns].copy(); routing='ensemble'
    elif spec=='public_oof':
        frame=train[columns].copy(); routing='oof_replay'   # metadata columns only; labels are not read
    else:
        frame=pd.read_csv(spec); routing='ensemble'
        missing={'clip','path','preferred_foot','gender'}-set(frame)
        if missing: raise ValueError(f'Inference CSV {spec} lacks columns {sorted(missing)}')
        bad=frame[~frame.preferred_foot.isin(['left','right'])|~frame.gender.isin(['F','M'])]
        if len(bad): raise ValueError(f'preferred_foot must be left/right and gender F/M (as in training): {bad.clip.tolist()}')
        frame['path']=[p if Path(p).is_absolute() else str(Path(spec).parent/p) for p in frame['path'].astype(str)]
        if 'source_id' not in frame: frame['source_id']='custom'
    if 'path' not in frame:
        frame['path']=[str(video_paths[c]) for c in frame['clip']]
    frame['path']=frame['path'].astype(str)
    for p in frame['path']:
        if not Path(p).is_file(): raise FileNotFoundError(f'Inference clip not found: {p}')
    frame['in_dataset']=[c in video_paths and Path(p).resolve()==Path(video_paths[c]).resolve()
                         for c,p in zip(frame['clip'],frame['path'])]
    return frame.reset_index(drop=True),routing

_INFERENCE_CONTENT={}
def inference_build_view(record,selector,member):
    if record['in_dataset']:
        raw=extract_raw(record['clip'])     # shared label-free detector cache for dataset clips
    else:
        key=content_id_for(record['path'],_INFERENCE_CONTENT)
        raw=extract_raw(record['clip'],path=record['path'],cache_path=INFERENCE_RAW/f'{key}.pkl')
    return make_views(raw,selector)

def inference_model_factory(manifest,member):
    set_active_ablation(manifest['feature']['variant'])
    if numeric_feature_manifest()!=manifest['feature']['numeric_features']:
        raise InferenceArtifactError('Saved numeric feature schema/order differs from this notebook version')
    if manifest['feature'].get('streams') is not None and list(ablation_stream_ids())!=list(manifest['feature']['streams']):
        raise InferenceArtifactError('Saved visual streams differ from this notebook version')
    # architecture= rebuilds the saved encoder configuration without downloading pretrained weights.
    return MultiViewVideo(manifest['feature']['numeric_dimension'],architecture=member['backbone']['architecture'])

def inference_numeric_features(view,metadata_row,feature):
    return ablation_numeric_features(view,metadata_row,ABLATION_CONFIGS[feature['variant']])

INFERENCE_HOOKS=dict(build_view=inference_build_view,save_view=atomic_pickle,
    load_view=lambda path,decode_video=True: load_view({0:path},0,decode_video=decode_video),
    model_factory=inference_model_factory,numeric_features=inference_numeric_features,load_joblib=joblib.load)

def migrate_legacy_run(run_dir):
    """One-time export for a run trained before inference manifests. Old folders are preserved: this only
    adds stages/*/class_prior.json (when missing) and inference_manifest.json."""
    run=Path(run_dir); experiment=json.loads((run/'experiment_manifest.json').read_text())
    config=experiment['config']
    mean,std=IMAGE_MEAN,IMAGE_STD  # cell above derived these from the SAVED backbone (processor or ImageNet)
    if 'backbone' in config:
        backbone=None  # new-style runs saved backbone_serialization.json in every stage
    else:
        from transformers import VideoMAEConfig
        spec=BACKBONE_SPEC  # derived from the saved config by the configuration cell (VideoMAE, saved checkpoint id)
        assert spec['family']=='videomae' and spec['checkpoint']==config['video'], 'Legacy runs are VideoMAE runs'
        architecture=dict(kind='videomae',config=VideoMAEConfig.from_pretrained(config['video']).to_dict())  # config JSON only
        backbone=dict(identity=backbone_identity(spec,'finetune'),spec=spec,architecture=architecture,
                      provenance=dict(note='migrated legacy VideoMAE run; architecture config fetched, weights from run'),
                      training_policy='finetune',policy_description=TRAINING_POLICIES['finetune'],
                      streams=list(ablation_stream_ids(ABLATION_CONFIGS[config.get('ablation','production')])))
    print('Migration may read public TRAINING labels to recover a missing class prior; each prior uses only '
          'its own stage training clips and is saved. Inference itself never reads labels.')
    return export_inference_manifest(run,preprocessing=preprocessing_record(config,mean,std,CAMERA_CACHE_VERSION),
        backbone=backbone,training_labels=train[['clip']+TARGETS],dataset_id=DATA_ID,
        extra_clip_names=test['clip'].tolist())

def export_run_inference_manifest(run_dir):
    """Called after training: same exporter, current-session preprocessing values."""
    try:
        return export_inference_manifest(run_dir,preprocessing=preprocessing_record(CFG,IMAGE_MEAN,IMAGE_STD,CAMERA_CACHE_VERSION),
            dataset_id=DATA_ID,extra_clip_names=test['clip'].tolist())
    except InferenceArtifactError as error:
        print('WARNING: inference manifest not exported; training outputs are intact.\n',error)

def inference_output_root(run_dir):
    return OUTPUT/'inference'/PLATFORM/Path(run_dir).name


def run_inference_mode():
    global INFERENCE_RAW
    if not (TRAINED_RUN_DIR/'inference_manifest.json').exists():
        if is_bundle(TRAINED_RUN_DIR):
            raise InferenceArtifactError('This bundle has no inference_manifest.json; export bundles from finished runs.')
        if not MIGRATE_LEGACY_RUN:
            raise InferenceArtifactError(f'{TRAINED_RUN_DIR/"inference_manifest.json"} not found. If this run finished '
                'before this notebook version, set MIGRATE_LEGACY_RUN=True once (exports from the saved artifacts).')
        migrate_legacy_run(TRAINED_RUN_DIR)
    manifest=load_inference_manifest(TRAINED_RUN_DIR,VERIFY_CHECKSUMS)
    pre=manifest['preprocessing']
    assert (N_FRAMES,SIZE,float(WINDOW))==(pre['frames'],pre['size'],pre['window']), 'Saved preprocessing not applied'
    assert np.allclose(IMAGE_MEAN,pre['image_mean']) and np.allclose(IMAGE_STD,pre['image_std'])
    raw_settings={k:v for k,v in RAW_CFG.items() if k not in ('dataset','click_timestamps')}
    INFERENCE_RAW=OUTPUT/('inference_detector_cache_'+stable_hash(raw_settings)); INFERENCE_RAW.mkdir(parents=True,exist_ok=True)
    frame,routing=resolve_inference_clips(INFERENCE_INPUT)
    print(f'Inference: {len(frame)} clip(s), routing={routing}, {len(manifest["members"])} fitted member(s), '
          f'backbone={manifest["backbone"]["identity"]["name"]} ({manifest["backbone"]["training_policy"]})')
    # Outputs/caches go to this platform's OUTPUT_ROOT, never into the trained run or imported bundle.
    out_root=inference_output_root(TRAINED_RUN_DIR)
    result=run_saved_inference(TRAINED_RUN_DIR,frame,hooks=INFERENCE_HOOKS,routing=routing,
        verify_checksums=VERIFY_CHECKSUMS,device=DEVICE,dataset_id=DATA_ID,output_root=out_root)
    release_pose_model()
    if INFERENCE_INPUT=='private':
        path=result['output_dir']/'submission.csv'
        submission=write_submission(result,template,path); display(submission.head())
        print('Saved submission:',path)
    print(json.dumps(result['record'],indent=2,default=str))
    if result['record']['max_abs_difference_vs_saved_oof'] is not None:
        print('OOF replay reproduces saved outer predictions within:',result['record']['max_abs_difference_vs_saved_oof'])
    return result


In [ ]:
# CELL: smoke_holdout_dispatch

def audit_localization(indices,mapping):
    report=[]
    for i in indices:
        clip=rows['clip'].iloc[i]
        if clip not in clicks.index: continue
        d=load_view(mapping,i); raw=extract_raw(clip)
        for r,name in enumerate(['kicker','keeper']):
            row=clicks.loc[clip]; t=float(row[f'{name}_click_time_seconds'])
            k=np.abs(raw['times']-t).argmin(); predicted=d['traces'][k,r]
            truth=row[[f'{name}_x_norm',f'{name}_y_norm']].to_numpy(dtype=float)
            report.append(dict(clip=clip,role=name,detected=bool(predicted[2]>0),
                distance=float(np.linalg.norm(predicted[:2]-truth)) if predicted[2]>0 else np.nan,
                sample_offset_seconds=float(abs(raw['times'][k]-t))))
    return report

def run_smoke():
    import time
    started=time.monotonic()
    folder,mapping,selector=build_stage(smoke_train,smoke_hold,'end_to_end_smoke')
    # Verify saved cross-fitting provenance against the actual prediction-source IDs.
    provenance=json.loads((folder/'provenance.json').read_text())
    for entry in provenance:
        prediction_sources=set(rows.source_id.iloc[entry['prediction_indices']])
        assert not prediction_sources & set(entry['fit_sources'])
    model,scaler=fit_video(smoke_train,mapping,folder,VIDEO_SEEDS[0])
    probabilities=predict_video(model,scaler,smoke_hold,mapping)
    for j,prob in enumerate(probabilities):
        assert prob.shape==(len(smoke_hold),len(CLASSES[j]))
        assert np.isfinite(prob).all() and np.allclose(prob.sum(1),1,atol=1e-5)
    # Reload the serialized model and scaler, and check inference reproducibility.
    checkpoint=folder/f'video_seed{VIDEO_SEEDS[0]}.pt'
    model.load_state_dict(torch.load(checkpoint,map_location=DEVICE,weights_only=True)['model'])
    reloaded_scaler=joblib.load(folder/'numeric_scaler.joblib')
    repeated=predict_video(model,reloaded_scaler,smoke_hold[:1],mapping)
    for j in range(len(TARGETS)): np.testing.assert_allclose(probabilities[j][:1],repeated[j],rtol=1e-3,atol=1e-4)
    timing_estimate=benchmark_video(model,scaler,smoke_train,mapping) if RUN_TIMING_BENCHMARK else None
    del model; gc.collect(); torch.cuda.empty_cache()
    # Exercise the numeric specialist and probability-ensemble interface too.
    x=np.stack([np.r_[numeric_summary(load_view(mapping,i)),metadata[i]] for i in smoke_train])
    z=np.stack([np.r_[numeric_summary(load_view(mapping,i)),metadata[i]] for i in smoke_hold])
    ensemble=[]
    for j,classes in enumerate(CLASSES):
        if len(np.unique(Y[smoke_train,j]))>1:
            tree=CatBoostClassifier(iterations=20,depth=2,verbose=False,random_seed=42,thread_count=4,loss_function='MultiClass')
        else:
            tree=DummyClassifier(strategy='prior')
        tree.fit(x,Y[smoke_train,j])
        p=np.zeros((len(smoke_hold),len(classes)))
        p[:,np.asarray(tree.classes_,dtype=int)]=tree.predict_proba(z)
        mixture=.5*probabilities[j]+.5*p
        assert np.isfinite(mixture).all() and np.allclose(mixture.sum(1),1,atol=1e-5)
        ensemble.append(mixture)
    preview=pd.DataFrame({'clip':rows['clip'].iloc[smoke_hold].to_numpy()})
    for j,(target,classes) in enumerate(zip(TARGETS,CLASSES)):
        preview[target]=np.array(classes)[ensemble[j].argmax(1)]
    assert not preview.isna().any().any()
    preview.to_csv(folder/'smoke_public_predictions.csv',index=False)
    training_checks=json.loads((folder/'video_training_checks.json').read_text())
    coverage=np.stack([load_view(mapping,i)['traces'][:,:,2]>0 for i in smoke_hold]).mean(axis=(0,1))
    report=dict(status='PASS',train_clips=len(smoke_train),heldout_clips=len(smoke_hold),
                train_sources=4,heldout_sources=2,role_counts=selector['counts'],
                heldout_role_detection_fraction=dict(zip(['kicker','keeper'],coverage.tolist())),
                elapsed_seconds=time.monotonic()-started,training=training_checks,
                timing=timing_estimate,
                checked=['role fit','source-cross-fitted crops','backbone gradients and updates',
                         'two-target inference','checkpoint reload','numeric specialist','ensemble'],
                note='Execution check only. Tiny-subset accuracy is not a performance estimate.')
    (RUN/'smoke_report.json').write_text(json.dumps(report,indent=2))
    print('END-TO-END SMOKE TEST: PASS'); print(json.dumps(report,indent=2)); display(preview)
    print('No competition CSV generated. Set QUICK_SMOKE=False, then restart and run all for full training.')

def probability_metrics(probabilities,indices):
    result={}; weighted=0.; score=0.
    for j,target in enumerate(TARGETS):
        p=probabilities[j]; truth=Y[indices,j]
        loss=float(-np.log(np.clip(p[np.arange(len(indices)),truth],1e-12,1)).mean())
        accuracy=float((p.argmax(1)==truth).mean())
        result[target+'_loss']=loss; result[target+'_accuracy']=accuracy
        weighted+=float(WEIGHTS[j])*loss; score+=float(WEIGHTS[j])*accuracy
    return dict(loss=weighted,weighted_accuracy=score,**result)

def run_holdout():
    import matplotlib.pyplot as plt
    training=split_indices['training']; validation=split_indices['validation']; testing=split_indices['test']
    (RUN/'holdout_manifest.json').write_text(json.dumps(dict(config=dict(CFG,ablation=ACTIVE_ABLATION_NAME,
        feature_manifest=numeric_feature_manifest()),seeds=VIDEO_SEEDS,
        interpretation='Holdout validation selects checkpoints: a tuning metric, not an untouched estimate.'),indent=2))
    folder,mapping,_=build_stage(training,np.r_[validation,testing],'train_validation_test')
    predictions={name:[np.zeros((len(ids),len(c))) for c in CLASSES]
                 for name,ids in [('validation',validation),('test',testing)]}
    histories=[]
    for seed in VIDEO_SEEDS:
        model,scaler=fit_video(training,mapping,folder,seed,validation_indices=validation)
        history=pd.read_csv(folder/f'loss_history_seed{seed}.csv'); history['seed']=seed; histories.append(history)
        # Test labels never select epochs, weights, scalers, crops, or hyperparameters.
        for name,ids in [('validation',validation),('test',testing)]:
            for j,p in enumerate(predict_video(model,scaler,ids,mapping)):
                predictions[name][j]+=p/len(VIDEO_SEEDS)
        del model; gc.collect(); torch.cuda.empty_cache()
    history=pd.concat(histories,ignore_index=True); history.to_csv(RUN/'loss_history.csv',index=False)
    fig,ax=plt.subplots(figsize=(9,5))
    for seed,h in history.groupby('seed'):
        ax.plot(h.epoch,h.train_loss,marker='o',label=f'Train / seed {seed}')
        ax.plot(h.epoch,h.validation_loss,marker='o',linestyle='--',label=f'Validation / seed {seed}')
    ax.set(xlabel='Epoch',ylabel='Weighted cross-entropy (70% kick / 30% keeper)',title='Training and validation loss')
    ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); fig.savefig(RUN/'loss_curves.png',dpi=150)
    display(fig); plt.close(fig)
    metrics=pd.DataFrame([dict(split=name,clips=len(ids),**probability_metrics(predictions[name],ids))
        for name,ids in [('validation',validation),('test',testing)]])
    metrics.to_csv(RUN/'holdout_metrics.csv',index=False); display(metrics)
    np.savez_compressed(RUN/'holdout_predictions.npz',validation_indices=validation,test_indices=testing,
        **{f'{name}_{j}':p for name,ps in predictions.items() for j,p in enumerate(ps)})
    export_run_inference_manifest(RUN)
    print('Best validation checkpoint per seed restored. Test loss is a final assessment, not an epoch tuning curve.')
    print('Saved split manifest, loss history, curves, and final metrics to:',RUN)

def development_split():
    """Fixed, label-free, source-disjoint development subset (seeded source permutation). Saved once and reused;
    never re-drawn after looking at scores."""
    sources=np.array(sorted(train.source_id.unique()))
    order=np.random.default_rng(SEED).permutation(sources)
    n_train=max(ROLE_FOLDS,int(round(DEV_TRAIN_SOURCE_FRACTION*len(sources))))
    n_val=max(1,int(round(DEV_VALIDATION_SOURCE_FRACTION*len(sources))))
    if n_train+n_val>len(sources): raise ValueError('Development fractions leave no disjoint validation sources.')
    training=np.flatnonzero(train.source_id.isin(order[:n_train]).to_numpy())
    validation=np.flatnonzero(train.source_id.isin(order[n_train:n_train+n_val]).to_numpy())
    for j,c in enumerate(CLASSES):
        if len(np.unique(Y[training,j]))!=len(c):
            raise ValueError(f'Development training subset lacks a {TARGETS[j]} class; change DEV_TRAIN_SOURCE_FRACTION '
                             '(a new, recorded split), not the seed after seeing results.')
    plan=dict(dataset=DATA_ID,seed=SEED,train_source_fraction=DEV_TRAIN_SOURCE_FRACTION,
              validation_source_fraction=DEV_VALIDATION_SOURCE_FRACTION,training=training.tolist(),validation=validation.tolist(),
              training_sources=sorted(order[:n_train].tolist()),validation_sources=sorted(order[n_train:n_train+n_val].tolist()))
    split_id='dev_'+stable_hash(plan)
    path=OUTPUT/'splits'/f'{split_id}.json'
    if path.exists():
        assert json.loads(path.read_text())['plan']==plan, 'Saved development split differs'
    else:
        atomic_write_text(path,json.dumps(dict(split_id=split_id,plan=plan),indent=2))
    return split_id,training,validation


def run_development():
    """Bounded preset: one seed, few epochs, fixed subset. DEVELOPMENT ONLY: the validation subset also selects
    the checkpoint, so its score is a selection-biased screening metric, never complete OOF performance."""
    with RunWriterLock(RUN,'run'):
        split_id,training,validation=development_split()
        split=train[['clip','source_id']].copy(); split['split']='unused'
        split.loc[training,'split']='development_training'; split.loc[validation,'split']='development_validation'
        split.to_csv(RUN/'development_split.csv',index=False)
        atomic_write_text(RUN/'development_manifest.json',json.dumps(dict(config=CFG,split_id=split_id,seeds=VIDEO_SEEDS,
            epochs=VIDEO_EPOCHS,training_clips=len(training),validation_clips=len(validation),
            training_sources=int(train.source_id.iloc[training].nunique()),validation_sources=int(train.source_id.iloc[validation].nunique()),
            evaluation_scope='DEVELOPMENT ONLY: bounded subset, inner-validation screening; not complete OOF performance',
            experiment_identity=EXPERIMENT_ID,attempt=ATTEMPT_ID),indent=2))
        folder,mapping,_=build_stage(training,validation,'development',selection=validation)
        averaged=[np.zeros((len(validation),len(c))) for c in CLASSES]
        for seed in VIDEO_SEEDS:
            model,scaler=fit_video(training,mapping,folder,seed,validation_indices=validation)
            for j,p in enumerate(predict_video(model,scaler,validation,mapping)): averaged[j]+=p/len(VIDEO_SEEDS)
            del model; gc.collect(); torch.cuda.empty_cache()
        metrics=pd.DataFrame([dict(split='development_validation',clips=len(validation),
            evaluation_scope='development_only_selection_biased',**probability_metrics(averaged,validation))])
        metrics.to_csv(RUN/'development_metrics.csv',index=False)
        np.savez_compressed(RUN/'development_predictions.npz',validation_indices=validation,
                            **{f'validation_{j}':p for j,p in enumerate(averaged)})
        display(metrics)
        print('DEVELOPMENT RESULT ONLY (subset, one seed, checkpoint chosen on this validation subset). '
              'Compare only with other development runs on split', split_id)
        return metrics


# ------------------------------ Execution routing (RUN_MODE from Settings) ------------------------------
inference_result=None; diagnostic_result=None
if RUN_MODE == 'reports':
    pass                                           # reports mode stops before this cell; definitions only
elif RUN_MODE == 'inference':
    inference_result=run_inference_mode()          # loads saved artifacts only; no training or selection
elif QUICK_SMOKE:
    with RunWriterLock(RUN,'run'): run_smoke()
elif EVALUATION_MODE=='holdout':
    with RunWriterLock(RUN,'run'): run_holdout()
elif EVALUATION_MODE=='development':
    development_result=run_development()
else:
    diagnostic_result=run_nested_diagnostics('production',private_inference=PREDICT_PRIVATE,fold_selection=FOLD_SELECTION)
    if diagnostic_result is None:
        print('Selected fold job(s) finished; the pooled reports are written by the job that completes the last fold.')
    else:
        oof=diagnostic_result['oof']; private=diagnostic_result['private']
        fold_ids=diagnostic_result['fold_ids']; reports=diagnostic_result['reports']
        display(diagnostic_result['exported']['metrics'])
        if FOLD_SELECTION is None: run_optional_ablations()


Validation loss diagnostics
Cross-entropy highlights confident mistakes; weighted error matches the 70/30 accuracy metric.
Analyze outer-held-out predictions only. Repeatedly tuning against them makes them a tuning set.


In [ ]:
# CELL: validation_report
# Run after all outer validation folds finish.
def validation_loss_report(train, Y, oof, targets, classes, weights, fold_ids):
    assert len(targets) == len(classes) == len(weights) == len(oof) == 2
    assert Y.shape == (len(train), 2)
    assert np.asarray(fold_ids).shape == (len(train),) and (np.asarray(fold_ids) >= 0).all()
    assert np.isclose(np.sum(weights), 1)
    columns = [c for c in ['clip', 'source_id', 'preferred_foot', 'gender'] if c in train]
    errors = train[columns].copy()
    errors['weighted_log_loss'] = 0.0
    errors['weighted_error'] = 0.0
    summary = []
    class_summary = []
    matrices = {}
    for j, target in enumerate(targets):
        p = np.asarray(oof[j])
        assert p.shape == (len(train), len(classes[j]))
        assert np.isfinite(p).all() and (p >= 0).all()
        assert np.allclose(p.sum(axis=1), 1, atol=1e-5)
        truth = Y[:, j]
        pred = p.argmax(axis=1)
        loss = -np.log(np.clip(p[np.arange(len(train)), truth], 1e-12, 1.0))
        wrong = (pred != truth).astype(float)
        errors[target + '_true'] = np.asarray(classes[j])[truth]
        errors[target + '_pred'] = np.asarray(classes[j])[pred]
        errors[target + '_true_probability'] = p[np.arange(len(train)), truth]
        errors[target + '_log_loss'] = loss
        errors[target + '_wrong'] = wrong
        errors['weighted_log_loss'] += weights[j] * loss
        errors['weighted_error'] += weights[j] * wrong
        summary.append(dict(target=target, accuracy=1-wrong.mean(), mean_log_loss=loss.mean(),
                            weighted_score_lost=weights[j]*wrong.mean()))
        for k, label in enumerate(classes[j]):
            mask = truth == k
            class_summary.append(dict(target=target, true_class=label, clips=int(mask.sum()),
                accuracy=float(1-wrong[mask].mean()) if mask.any() else np.nan,
                mean_log_loss=float(loss[mask].mean()) if mask.any() else np.nan,
                weighted_score_lost=float(weights[j]*wrong[mask].sum()/len(train))))
        matrices[target] = pd.crosstab(errors[target+'_true'], errors[target+'_pred']).reindex(
            index=classes[j], columns=classes[j], fill_value=0)
    return errors, pd.DataFrame(summary), pd.DataFrame(class_summary), matrices

if RUN_MODE in {'train', 'resume'} and not QUICK_SMOKE and EVALUATION_MODE=='nested_cv' and diagnostic_result is not None:
    errors, target_report, class_report, confusion_tables = validation_loss_report(
        train, Y, oof, TARGETS, CLASSES, WEIGHTS, fold_ids
    )
    # Same-fold majority baseline distinguishes learned signal from class balance.
    majority_error=np.zeros(len(train))
    for fold in np.unique(fold_ids):
        fit=np.flatnonzero(fold_ids!=fold); held=np.flatnonzero(fold_ids==fold)
        for j,w in enumerate(WEIGHTS):
            majority=np.bincount(Y[fit,j],minlength=len(CLASSES[j])).argmax()
            majority_error[held]+=float(w)*(Y[held,j]!=majority)
    print('Fold-majority baseline:',1-majority_error.mean(),
          '| Model lift:',majority_error.mean()-errors['weighted_error'].mean())
    # Correlate prediction errors with role/camera coverage, using held-out caches only.
    for role in ['kicker','keeper']: errors[role+'_tracked_fraction']=np.nan
    errors['camera_valid_fraction']=np.nan
    for record in reports:
        folder=Path(record['stage_folder'])
        for i in np.flatnonzero(fold_ids==record['fold']):
            cached=load_view({int(i):folder/f'{i}.pkl'},int(i))
            for r,role in enumerate(['kicker','keeper']):
                errors.loc[i,role+'_tracked_fraction']=float((cached['traces'][:,r,2]>0).mean())
            errors.loc[i,'camera_valid_fraction']=float((cached['numeric'][:,-3]>0).mean())
    for role in ['kicker','keeper']:
        errors[role+'_tracking']=pd.cut(errors[role+'_tracked_fraction'],[-.01,0,.5,1.01],
                                       labels=['missing','partial','mostly tracked'])
    print('New OOF score:', 1-errors['weighted_error'].mean())
    print('Per-target performance:'); display(target_report)
    print('True classes costing the most leaderboard points:')
    display(class_report.sort_values('weighted_score_lost', ascending=False))
    print('Most confidently wrong / highest-loss validation clips:')
    display(errors.sort_values('weighted_log_loss', ascending=False).head(20))
    for field in ['source_id', 'preferred_foot', 'gender', 'kicker_tracking', 'keeper_tracking']:
        if field not in errors: continue
        grouped = errors.groupby(field).agg(
            clips=('clip', 'size'), mean_log_loss=('weighted_log_loss', 'mean'),
            error_rate=('weighted_error', 'mean'), error_sum=('weighted_error', 'sum'))
        grouped['weighted_score_lost'] = grouped.pop('error_sum') / len(errors)
        grouped.to_csv(RUN / f'validation_by_{field}.csv')
        print(f'Error breakdown by {field} (inspect sample counts):')
        display(grouped.sort_values('weighted_score_lost', ascending=False).head(20))
    for target, matrix in confusion_tables.items():
        print(target, '(rows: true, columns: predicted)'); display(matrix)
        matrix.to_csv(RUN / f'confusion_{target}.csv')
    errors.to_csv(RUN / 'validation_errors.csv', index=False)
    target_report.to_csv(RUN / 'validation_targets.csv', index=False)
    class_report.to_csv(RUN / 'validation_classes.csv', index=False)
else:
    print('OOF diagnostics apply to nested_cv training runs. Holdout mode saves loss_history.csv and holdout_metrics.csv.')



Held-out player-region heatmaps
Highest-loss public clips are explained with their own held-out fold ensemble.
Training clicks inspire matching/tracking; audit clicks never change validation inputs.
Regions are obscured across all tracked frames, not interpreted as frame-specific saliency.


In [ ]:
# CELL: heatmaps
# Explain highest-loss OUTER validation clips, using their held-out-fold models.
# Tube occlusion obscures the same crop-relative region across tracked frames.
# Red loss regions: masking hurts the true class. Blue: masking helps it.
# These are sensitivity maps, not segmentation masks or proof of causation.

def occlusion_regions(size,grid):
    assert 1<=grid<=size
    edges=np.linspace(0,size,grid+1,dtype=int)
    return [(r,c,int(edges[r]),int(edges[r+1]),int(edges[c]),int(edges[c+1]))
            for r in range(grid) for c in range(grid)]


def occlude_player_video(video,region,valid,fill='blur'):
    # video: T,C,H,W uint8. Missing-role full-frame fallbacks remain untouched.
    _,_,y0,y1,x0,x1=region; result=video.copy()
    for t in np.flatnonzero(valid):
        if fill=='blur':
            rgb=video[t].transpose(1,2,0)
            replacement=cv2.GaussianBlur(rgb,(51,51),0).transpose(2,0,1)
            result[t,:,y0:y1,x0:x1]=replacement[:,y0:y1,x0:x1]
        elif fill=='mean':
            patch=video[t,:,y0:y1,x0:x1]
            result[t,:,y0:y1,x0:x1]=patch.mean(axis=(1,2),keepdims=True).astype(video.dtype)
        else: raise ValueError('HEATMAP_FILL must be blur or mean')
    return result


def occlusion_effects(base,perturbed,true_label,predicted_label):
    base=np.asarray(base,dtype=float); perturbed=np.asarray(perturbed,dtype=float)
    loss_delta=-np.log(np.clip(perturbed[...,true_label],1e-12,1))+np.log(np.clip(base[true_label],1e-12,1))
    predicted_drop=base[predicted_label]-perturbed[...,predicted_label]
    return loss_delta,predicted_drop


def combine_occluded_ensemble(oof_probability,video_baseline,video_occluded,video_weight):
    # Trees/priors are unchanged; only the visual branch is intervened on.
    changed=np.asarray(oof_probability)+float(video_weight)*(np.asarray(video_occluded)-np.asarray(video_baseline))
    if not np.isfinite(changed).all() or changed.min() < -2e-3 or changed.max()>1.002:
        raise ValueError('Reconstructed ensemble is inconsistent. Check fold checkpoints, seeds and crop cache.')
    changed=np.clip(changed,0,1)
    return changed/changed.sum(axis=-1,keepdims=True)


def assert_heldout_explanation(folder,index):
    provenance=json.loads((folder/'provenance.json').read_text())
    entries=[x for x in provenance if x['kind']=='evaluation' and int(index) in x['prediction_indices']]
    assert len(entries)==1, 'Heatmap clip must be an evaluation clip, not a training cross-fit clip.'
    assert train.source_id.iloc[index] not in entries[0]['fit_sources'], 'Training-source leakage in explanation.'


@torch.inference_mode()
def seed_occlusion_probabilities(model,scaler,index,mapping):
    model.eval()
    d=load_view(mapping,index,decode_video=True)
    video,numeric,_=ClipDataset([index],mapping,scaler)[0]
    video=video[None].to(DEVICE); numeric=numeric[None].to(DEVICE)
    if not hasattr(model,'encode_stream'):
        raise NotImplementedError(f'Occlusion heatmaps are not supported for {type(model).__name__}.')
    def encode(v):
        # Backbone adapter: VideoMAE, V-JEPA 2.1 and V-JEPA 2 share [B,T,C,H,W] -> [B,feature_dim].
        with torch.autocast('cuda',dtype=torch.float16):
            return model.encode_stream(v)
    def probabilities(features):
        with torch.autocast('cuda',dtype=torch.float16):
            fused=torch.cat(features+[model.numeric(numeric)],dim=1)
            logits=[head(model.dropout(fused)) for head in model.heads]
        return [x.float().softmax(1)[0].cpu().numpy() for x in logits]
    features=[encode(video[:,view]) for view in range(3)]
    baseline=probabilities(features)
    # Cached unchanged-stream embeddings must reproduce the normal forward method.
    with torch.autocast('cuda',dtype=torch.float16): direct=model(video,numeric)
    for a,b in zip(baseline,direct):
        np.testing.assert_allclose(a,b.float().softmax(1)[0].cpu().numpy(),atol=2e-4,rtol=2e-3)
    maps=[np.empty((2,HEATMAP_GRID,HEATMAP_GRID,len(c)),dtype=np.float32) for c in CLASSES]
    regions=occlusion_regions(SIZE,HEATMAP_GRID)
    for role in range(2):
        valid=d['traces'][:,role,2]>0
        for region in tqdm(regions,desc=f'Occlude {rows["clip"].iloc[index]} role {role}',leave=False):
            r,c,*_=region
            if not valid.any(): altered=baseline
            else:
                pixels=occlude_player_video(d['video'][role+1],region,valid,HEATMAP_FILL).astype(np.float32)/255
                pixels=(pixels-IMAGE_MEAN[None,:,None,None])/IMAGE_STD[None,:,None,None]
                changed=list(features)
                changed[role+1]=encode(torch.from_numpy(pixels)[None].to(DEVICE))
                altered=probabilities(changed)
            for j in range(len(TARGETS)): maps[j][role,r,c]=altered[j]
    return baseline,maps


def save_tracking_preview(index,folder,output):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Rectangle
    d=load_view({index:folder/f'{index}.pkl'},index)
    raw=extract_raw(rows['clip'].iloc[index])
    anchor=int(d.get('anchor_frame',np.abs(raw['times']-(raw['times'][-1]-.23)).argmin()))
    selected=sorted(set([0,anchor,len(raw['frames'])-1]))
    fig,axes=plt.subplots(1,len(selected),figsize=(6*len(selected),4),squeeze=False)
    for ax,t in zip(axes[0],selected):
        rgb=raw['frames'][t]; h,w=rgb.shape[:2]; ax.imshow(rgb)
        for role,color in enumerate(['orange','cyan']):
            if 'track_ids' in d and d['track_ids'][t,role]>=0:
                box=raw['detections'][t]['boxes'][d['track_ids'][t,role]]*[w,h,w,h]
                ax.add_patch(Rectangle(box[:2],box[2]-box[0],box[3]-box[1],fill=False,edgecolor=color,linewidth=2))
            elif d['traces'][t,role,2]>0:
                ax.scatter(*list(d['traces'][t,role,:2]*[w,h]),c=color,s=70)
        clip=rows['clip'].iloc[index]
        if clip in clicks.index:
            for name,color in [('kicker','orange'),('keeper','cyan')]:
                row=clicks.loc[clip]
                if abs(raw['times'][t]-row[f'{name}_click_time_seconds'])<=.06:
                    ax.scatter(row[f'{name}_x_norm']*w,row[f'{name}_y_norm']*h,marker='x',c=color,s=100)
        ax.set_title(f'{raw["times"][t]-raw["times"][-1]:+.2f}s from final frame'); ax.axis('off')
    fig.suptitle(f'{rows["clip"].iloc[index]} | orange kicker, cyan keeper; crosses: public audit clicks only')
    fig.tight_layout(); fig.savefig(output,dpi=150,bbox_inches='tight'); display(fig); plt.close(fig)


def run_validation_heatmaps():
    import matplotlib.pyplot as plt
    from matplotlib.colors import TwoSlopeNorm
    assert not QUICK_SMOKE and (fold_ids>=0).all()
    output=RUN/'heatmaps'; output.mkdir(exist_ok=True)
    ranking=np.zeros(len(train))
    for j,w in enumerate(WEIGHTS):
        ranking+=float(w)*(-np.log(np.clip(oof[j][np.arange(len(train)),Y[:,j]],1e-12,1)))
    chosen=np.argsort(-ranking)[:min(HEATMAP_TOP_K,len(train))]
    fold_reports={int(r['fold']):r for r in json.loads((RUN/'fold_results.json').read_text())}
    rows_report=[]
    for fold in sorted(set(fold_ids[chosen])):
        report=fold_reports[int(fold)]
        folder=Path(report['stage_folder'])
        indices=[int(i) for i in chosen if fold_ids[i]==fold]
        mapping={i:folder/f'{i}.pkl' for i in indices}
        for i in indices: assert_heldout_explanation(folder,i)
        seeds=report.get('seeds',VIDEO_SEEDS)
        accumulated={i:dict(base=[np.zeros(len(c)) for c in CLASSES],
            maps=[np.zeros((2,HEATMAP_GRID,HEATMAP_GRID,len(c))) for c in CLASSES]) for i in indices}
        for seed in seeds:
            scaler=joblib.load(folder/'numeric_scaler.joblib')
            model=MultiViewVideo(scaler.n_features_in_).to(DEVICE)
            state=torch.load(folder/f'video_seed{seed}.pt',map_location='cpu',weights_only=True)
            model.load_state_dict(state['model'])
            for i in indices:
                base,maps=seed_occlusion_probabilities(model,scaler,i,mapping)
                for j in range(len(TARGETS)):
                    accumulated[i]['base'][j]+=base[j]/len(seeds)
                    accumulated[i]['maps'][j]+=maps[j]/len(seeds)
            del model,state; gc.collect(); torch.cuda.empty_cache()
        for i in indices:
            clip=rows['clip'].iloc[i]; stem=Path(clip).stem; d=load_view(mapping,i,decode_video=True)
            save_tracking_preview(i,folder,output/f'{stem}_tracking.png')
            # Verify the unmasked reconstructed ensemble against its stored OOF prediction.
            trees=joblib.load(folder/'pose_camera_trees.joblib')
            numeric=np.r_[numeric_summary(d),metadata[i]][None]
            eval_provenance=next(x for x in json.loads((folder/'provenance.json').read_text()) if x['kind']=='evaluation')
            fit_indices=np.flatnonzero(train.source_id.isin(eval_provenance['fit_sources']).to_numpy())
            for j,target in enumerate(TARGETS):
                video_base=accumulated[i]['base'][j]; video_maps=accumulated[i]['maps'][j]
                mixture=np.asarray(report['weights'][j]); tree_p=np.zeros(len(CLASSES[j]))
                tree_p[np.asarray(trees[j].classes_,dtype=int)]=trees[j].predict_proba(numeric)[0]
                prior=np.bincount(Y[fit_indices,j],minlength=len(CLASSES[j]))/len(fit_indices)
                reconstructed=mixture[0]*video_base+mixture[1]*tree_p+mixture[2]*prior
                np.testing.assert_allclose(reconstructed,oof[j][i],atol=2e-3,rtol=2e-3,
                    err_msg='Heatmap model/crop/ensemble does not reproduce OOF probabilities')
                perturbed=combine_occluded_ensemble(oof[j][i],video_base,video_maps,mixture[0])
                true_label=int(Y[i,j]); predicted_label=int(oof[j][i].argmax())
                loss_map,prediction_map=occlusion_effects(oof[j][i],perturbed,true_label,predicted_label)
                fig,axes=plt.subplots(2,3,figsize=(13,8))
                scales=[max(float(np.abs(loss_map).max()),1e-6),max(float(np.abs(prediction_map).max()),1e-6)]
                for role,name in enumerate(['kicker','keeper']):
                    valid=np.flatnonzero(d['traces'][:,role,2]>0)
                    t=int(valid[-1]) if len(valid) else N_FRAMES-1
                    image=d['video'][role+1,t].transpose(1,2,0)
                    axes[role,0].imshow(image); axes[role,0].set_title(f'{name}: tracked {len(valid)}/{N_FRAMES} frames')
                    for col,(heat,label) in enumerate([(loss_map[role],'Change in true-class loss'),
                                                       (prediction_map[role],'Drop in predicted-class probability')],start=1):
                        ax=axes[role,col]; ax.imshow(image)
                        resized=cv2.resize(heat.astype(np.float32),(SIZE,SIZE),interpolation=cv2.INTER_NEAREST)
                        overlay=ax.imshow(resized,cmap='coolwarm',alpha=.6,
                            norm=TwoSlopeNorm(vmin=-scales[col-1],vcenter=0,vmax=scales[col-1]))
                        ax.set_title(label); fig.colorbar(overlay,ax=ax,fraction=.046)
                    for ax in axes[role]: ax.axis('off')
                    rows_report.append(dict(clip=clip,fold=int(fold),target=target,role=name,
                        weighted_oof_loss=float(ranking[i]),true_class=CLASSES[j][true_label],
                        predicted_class=CLASSES[j][predicted_label],video_ensemble_weight=float(mixture[0]),
                        tracked_fraction=len(valid)/N_FRAMES,max_loss_increase=float(loss_map[role].max()),
                        largest_loss_decrease=float(loss_map[role].min())))
                fig.suptitle(f'{clip} | {target}: true={CLASSES[j][true_label]}, predicted={CLASSES[j][predicted_label]}\n'
                    f'OOF weighted loss={ranking[i]:.3f}; video weight={mixture[0]:.2f}; {HEATMAP_FILL} tube occlusion across tracked frames')
                fig.tight_layout(); fig.savefig(output/f'{stem}_{target}_heatmap.png',dpi=150,bbox_inches='tight')
                display(fig); plt.close(fig)
                np.savez_compressed(output/f'{stem}_{target}_maps.npz',loss_delta=loss_map,
                    predicted_probability_drop=prediction_map,base_probability=oof[j][i],perturbed_probability=perturbed)
    pd.DataFrame(rows_report).to_csv(output/'heatmap_summary.csv',index=False)
    pd.DataFrame({'clip':train['clip'].to_numpy(),'weighted_oof_loss':ranking}).sort_values(
        'weighted_oof_loss',ascending=False).to_csv(output/'validation_loss_ranking.csv',index=False)
    print('Saved held-out heatmaps and tracking previews to:',output)
    print('Context and numeric pose/motion inputs remain fixed. Duplicate evidence can make a region appear unimportant.')

if GENERATE_HEATMAPS and RUN_MODE in {'train', 'resume'} and not QUICK_SMOKE and EVALUATION_MODE=='nested_cv' and diagnostic_result is not None:
    run_validation_heatmaps()
else:
    print('Heatmaps run after full OOF validation; smoke mode does not estimate trustworthy high-loss examples.')



Submission and interpretation
This compensates motion features, not video pixels. Original frames preserve fine visual detail;
the learned video model still sees camera movement. Compensation is approximate and invalid
estimates are masked, never presented as measured zero motion. It does not reconstruct 3D camera
rotation, correct strong perspective/parallax, or guarantee robustness across cuts. Role selection
still depends on person detection; missing or ambiguous training matches are skipped.
All backbone parameters are fine-tuned; this is not a frozen feature-only implementation.
Run-all is expensive (inner and outer training, multiple seeds, three video streams). Keep
checkpoints/caches on Drive. The default smoke mode does not enter full CV or write a submission.
Verify available Drive space; cross-fitted RGB caches can use
tens of GB. Cache IDs include data and pipeline config to avoid reuse across different runs.
Outer scores are assessment, not another tuning set. No winning score is guaranteed.


In [ ]:
# CELL: submission
if RUN_MODE == 'inference':
    if inference_result is not None and (inference_result['output_dir']/'submission.csv').exists():
        print('Inference submission:', inference_result['output_dir']/'submission.csv')
        offer_download(inference_result['output_dir']/'submission.csv')
    else:
        print('Inference output (no submission for this input):', inference_result['output_dir'])
elif QUICK_SMOKE:
    print("Smoke mode: no competition submission is generated.")
elif EVALUATION_MODE in {'holdout','development'}:
    print(f'{EVALUATION_MODE} run complete. Submissions come only from finished nested_cv runs (or inference mode).')
elif diagnostic_result is None:
    print('Nested folds are still incomplete; no submission until every fold is aggregated.')
elif not PREDICT_PRIVATE:
    print('Private inference was disabled. Set PREDICT_PRIVATE=True and rerun nested diagnostics to export a submission.')
else:
    submission_targets = ["kick_direction", "goalkeeper_direction"]
    submission_weights = np.array([0.70, 0.30])
    indices = [TARGETS.index(target) for target in submission_targets]

    pred = pd.DataFrame({"clip": test["clip"].to_numpy()})
    for target, j in zip(submission_targets, indices):
        probabilities = np.asarray(private[j])
        classes = np.asarray(CLASSES[j])
        assert probabilities.shape == (len(test), len(classes))
        assert np.isfinite(probabilities).all()
        assert (probabilities >= 0).all()
        assert np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
        pred[target] = classes[probabilities.argmax(axis=1)]

    submission = template[["clip"]].merge(
        pred, on="clip", how="left", validate="one_to_one"
    )
    assert len(submission) == len(test) == 900
    assert submission["clip"].is_unique
    assert not submission.isna().any().any()
    assert set(submission["clip"]) == set(test["clip"])
    assert list(submission.columns) == ["clip"] + submission_targets

    # Recalculate validation accuracy using the new competition metric.
    assert (fold_ids >= 0).all(), "Finish all folds before exporting."
    new_accuracies = np.array([
        accuracy_score(Y[:, j], oof[j].argmax(axis=1)) for j in indices
    ])
    print("OOF accuracies:", dict(zip(submission_targets, new_accuracies)))
    print(f"New weighted OOF score: {submission_weights @ new_accuracies:.6f}")

    output_path = RUN / "submission.csv"
    submission.to_csv(output_path, index=False)
    export_config = dict(CFG, submission_targets=submission_targets,
                         submission_weights=submission_weights.tolist())
    (RUN / "config.json").write_text(json.dumps(export_config, indent=2))
    import sys
    (RUN / "environment.txt").write_text(
        subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
    )
    display(submission.head())
    print("Saved:", output_path)
    offer_download(output_path)


## Pipeline diagnosis and next improvements
Summarizes a saved run from its report files only (it never loads a model): validation of run identity, class order, folds and coverage; an overview; and at most five evidence-backed **investigation priorities** with `priority, suspected_component, finding, evidence_files, component_or_comparison, sample_count, fold_support, effect_value, effect_units, evidence_type, suggested_change, next_controlled_test, limitations`.

Evidence types: `retrained_comparison` (matched-fold ablations), `predictive_comparison` (ensemble vs inner-selected reference, specialist vs prior, loss curves, target contributions), `quality_association` (tracking/camera quality vs error; not causal), `visual_audit_needed` (click-audit proxies), `insufficient_evidence`. Priority is an investigation order, not a causal ranking; the ranking rule and thresholds are saved in `pipeline_summary.json`. Weighted error contributions use the full evaluated population (`0.70*wrong_kicks/N`, `0.30*wrong_keepers/N`); accuracy deltas are percentage points; losses are reported separately. Slices under 30 clips are flagged small; fold support is a count, not a significance test. Missing ablations are reported as *not evaluated*; missing folds mark the run incomplete and suppress complete-OOF conclusions. Mixed or incompatible report files are rejected. Writes `pipeline_summary.md`, `pipeline_summary.json`, `improvement_priorities.csv` in the run folder. A recommendation is a proposed future experiment: do not tune it against the same outer labels and call the result untouched.


In [ ]:
# CELL: pipeline_summary_run
# Pipeline diagnosis for the run this session trained, or for the fitted run used by inference
# (inference outputs carry no labels). Summaries of foreign runs/bundles are written outside them.
SUMMARY_RUN_DIR = TRAINED_RUN_DIR if RUN_MODE == 'inference' else RUN
SUMMARY_OUTPUT = report_output_dir(SUMMARY_RUN_DIR) if RUN_MODE == 'inference' else RUN
if QUICK_SMOKE:
    print('Smoke runs produce no outer-fold reports; the summary below will state that evidence is insufficient.')
pipeline_summary, pipeline_summary_markdown = write_pipeline_summary(SUMMARY_RUN_DIR, output_dir=SUMMARY_OUTPUT)
print('Saved:', Path(SUMMARY_OUTPUT)/'pipeline_summary.md', '|', Path(SUMMARY_OUTPUT)/'pipeline_summary.json',
      '|', Path(SUMMARY_OUTPUT)/'improvement_priorities.csv')
show_markdown(pipeline_summary_markdown)


In [ ]:
# CELL: bundle_export
# Optional: copy a finished run into a relocatable, checksummed bundle (e.g. HPC -> Drive). The copy is staged,
# verified and only then marked complete; an interrupted transfer has no marker and is rejected on import.
if EXPORT_BUNDLE_TO:
    _source = RUN if RUN_MODE in TRAINING_MODES else TRAINED_RUN_DIR
    exported_bundle = export_bundle(_source, Path(EXPORT_BUNDLE_TO))
    verify_bundle(exported_bundle)
    print('Bundle ready. Load it elsewhere with RUN_MODE="inference" or "reports" and TRAINED_RUN_DIR =', exported_bundle)


References and disclosure
Pretrained VideoMAE: https://huggingface.co/MCG-NJU/videomae-base-finetuned-kinetics
VideoMAE API: https://huggingface.co/docs/transformers/model_doc/videomae
V-JEPA 2 / 2.1 official code and checkpoints: https://github.com/facebookresearch/vjepa2 (code pinned at 204698b45b3712590f06245fbfba32d3be539812)
V-JEPA 2 in Transformers: https://huggingface.co/docs/transformers/model_doc/vjepa2 and https://huggingface.co/facebook/vjepa2-vitl-fpc64-256
Known 2.1 checkpoint/config reports for continued pretraining: https://github.com/facebookresearch/vjepa2/issues/149 (the adapter therefore loads the encoder with strict key/shape checking)
Ultralytics pose: https://docs.ultralytics.com/tasks/pose/
OpenCV similarity estimation: https://docs.opencv.org/4.11.0/d9/d0c/group__calib3d.html
Name these pretrained resources and disclose LLM assistance in the assignment report.
Authoring validation is separate from Colab execution; no competition result is prefilled. No backbone is claimed to improve the competition score.


## Outputs and how to read them

Run folders live at `OUTPUT_ROOT/runs/<platform>/<EXPERIMENT_NAME>__learned_<experiment id>/`. Each execution writes `attempts/<attempt id>/attempt.json`; per-fold jobs write `folds/fold<k>/` and pooled files appear only after every fold is complete.

| Output | Meaning |
|---|---|
| `experiment_manifest.json`, `folds.csv` | Configuration, class mappings, seeds, fixed outer/inner assignments and interpretation |
| `component_reports/component_metrics.csv` | Per-fold and pooled kick/keeper accuracy and cross-entropy, plus weighted 70/30 versions |
| `component_reports/component_oof_probabilities.npz`, `component_manifest.json` | Aligned complete OOF probabilities, labels, clip order, component and class mapping |
| `component_reports/component_clip_errors.csv` | One row per component/clip with both targets, all probabilities, correctness, loss, metadata and quality |
| `component_reports/highest_loss.csv`, `greatest_accuracy_error.csv` | Separate rankings for confident mistakes versus weighted accuracy errors |
| `component_reports/ensemble_comparison.csv` | Corrections, newly introduced mistakes, disagreement and score change versus an inner-selected reference |
| `component_reports/component_slices.csv`, `component_confusions.csv` | Counts, accuracy/loss, full-population error contributions and confusion matrices |
| `tracking_quality.csv`, `tracking_frames.csv` | Coverage, gaps, jumps, confidence, crop clipping, explicit full-frame fallback, pose and camera quality |
| `role_click_audit.csv`, `role_click_summary.csv` | Timestamp offsets, normalized distance, containment and detector/selector failure proxies |
| `visual_audits/` | Several timestamps per selected clip, context/boxes/pose/crops and public post-prediction click overlays |
| `stages/*/component_provenance.json`, `provenance.json`, `video_seed*_fit_manifest.json` | Sources permitted for each component, crop cross-fitting, scaler and checkpoint selection |
| `ablations/<variant>/`, `ablation_comparisons.csv` | Separately retrained experiments and matched-fold/pooled differences for available reference pairs |
| `stages/*/class_prior.json`, `backbone_serialization.json`, `stage_identity.json` | Training-only priors, backbone/architecture/pooling/trainable parameters, fitted-stage identity (excludes prediction clips) |
| `inference_manifest.json` | Everything inference needs: relative paths + sha256 of selectors, scalers, trees, seed weights; priors; frozen ensemble weights; preprocessing; feature schema; class order |
| `OUTPUT_ROOT/inference/<platform>/<run>/` | Inference-only crop caches, per-clip prediction caches keyed by fitted-artifact ID + clip content + preprocessing + metadata, and `runs/<input>/` outputs (`final_probabilities.csv`, `probabilities.npz`, `inference_run.json`, `submission.csv`) |
| `pipeline_summary.md/.json`, `improvement_priorities.csv` | Final *Pipeline diagnosis and next improvements* (report-only) |
| `folds/fold<k>/result.json`, `component_predictions.npz`, `COMPLETE` | One outer fold's selection, weights and predictions (written only by that fold's job) |
| `development_manifest.json`, `development_metrics.csv`, `development_split.csv` | Development preset: subset, one seed, few epochs. DEVELOPMENT ONLY, never a full OOF result |
| `locks/`, `attempts/` | Writer locks (heartbeat) and per-attempt records (host, Slurm IDs, settings, versions, code fingerprint) |

Weighted accuracy is `0.70*kick_accuracy + 0.30*keeper_accuracy`; higher is better. Weighted cross-entropy is a separate loss; lower is better. High cross-entropy can identify confident wrong predictions even when two clips have the same weighted accuracy error. Slice error contributions use the entire evaluated population as denominator. Slices overlap and must not be added together; inspect counts before interpreting small slices. Quality/performance associations do not prove causation.

Role audits compare predictions at the nearest sampled classification timestamp, record mismatch, and exclude large mismatches from summary rates. Separate annotation-nearest detections help distinguish detector failures from role-selection failures. Click-inside-box is a localization proxy, not identity proof. These data cannot establish an automatic identity-switch error rate. Visual audits help inspect possible switches without changing predictions.


## Recipes (Colab)

Every recipe: run the install cell (skip for reports), **Runtime > Restart session**, edit **Settings**, then **Run after** from Settings.

**1. Real-backbone smoke test (default):** `RUN_MODE='smoke'`, choose `BACKBONE`/`TRAINING_POLICY`. Uses real clips, the real checkpoint and the GPU; writes `smoke_report.json` and `timing_estimate.json` in the run folder.

**2. Development experiment (bounded):** `RUN_MODE='train'`, `EVALUATION_MODE='development'`, a new `EXPERIMENT_NAME` per idea, one change at a time (e.g. `TRAINING_POLICY='frozen_probe'` vs `'finetune'`). Same saved development split for every idea (`OUTPUT_ROOT/splits/dev_*.json`). Results are labeled DEVELOPMENT ONLY and are never a submission or a full OOF table. After a disconnect: same settings with `RUN_MODE='resume'`.

**3. Full nested evaluation (deliberate):** `RUN_MODE='train'`, `EVALUATION_MODE='nested_cv'` (optionally `PREDICT_PRIVATE=True`). Long runs belong on HPC; here they resume epoch by epoch from Drive.

**4. Inference from a saved run or an HPC bundle (no training):** copy the bundle folder to Drive or `/content`, then `RUN_MODE='inference'`, `TRAINED_RUN_DIR=<bundle>`, `INFERENCE_INPUT='private'` (needs the ZIP; writes `submission.csv`), `'public_oof'` (held-out fold per clip), or a CSV `clip,path,preferred_foot,gender` (no ZIP, no labels). Bundles are checksum-verified first; outputs go to `OUTPUT_ROOT/inference/colab/<run>/`, never into the bundle.

**5. Reports only:** `RUN_MODE='reports'`, `TRAINED_RUN_DIR=<run or bundle>`. Skip the install cell. Only pandas/numpy are used; Drive is mounted only if the path is on Drive; the notebook stops after the summary.

**Keep on Drive:** `OUTPUT_ROOT/runs/colab/...` (checkpoints, reports), `OUTPUT_ROOT/splits/`, `OUTPUT_ROOT/diagnostic_splits/`, detector caches. `/content` (SCRATCH_ROOT) is disposable. Never point Colab at an HPC run folder that is still training; use an exported bundle.

**Validation status:** see `docs/VALIDATION.md`. CPU synthetic tests pass; real-checkpoint/GPU smoke, pilot and full evaluation have not been run by the author of these notebooks.
